In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# This cell verifies the project paths and confirms the AMI resources available for the base-paper reproduction.

import os
import json
import glob

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

AMI_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "ami"
)

print("=" * 70)
print("BASE-PAPER REPRODUCTION — AMI RESOURCE CHECK")
print("=" * 70)

print("Project directory:", PROJECT_DIR)
print("AMI directory:", AMI_DIR)
print("AMI exists:", os.path.exists(AMI_DIR))

meeting_id = "ES2004a"
meeting_dir = os.path.join(AMI_DIR, meeting_id)

print("\nMeeting:", meeting_id)
print("Meeting directory:", meeting_dir)
print("Exists:", os.path.exists(meeting_dir))

if os.path.exists(meeting_dir):

    files = glob.glob(
        os.path.join(meeting_dir, "**", "*"),
        recursive=True
    )

    files = [
        f for f in files
        if os.path.isfile(f)
    ]

    print("\nFiles available:")

    for f in files:
        print(" -", f)

BASE-PAPER REPRODUCTION — AMI RESOURCE CHECK
Project directory: /content/drive/MyDrive/MTechIndProj/MoM_Project
AMI directory: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami
AMI exists: True

Meeting: ES2004a
Meeting directory: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a
Exists: True

Files available:
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/ES2004a_reference_summary.txt
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/ES2004a_verification.json
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/video/ES2004a.PreferredOverview.avi
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/video/ES2004a.Closeup1.avi
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/audio/ES2004a.Mix-Headset.wav
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/ES2004a.586.44__693.07.jpg
 - /content/drive/MyDrive/MTechIndProj/MoM_

In [ ]:
# This cell installs the Whisper implementation and audio dependencies required for reproducing the ASR stage.

!pip install -q openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# This cell verifies that Whisper can access the Tesla T4 GPU before processing the meeting audio.

import torch
import whisper

print("=" * 70)
print("WHISPER ENVIRONMENT CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Whisper imported successfully.")

WHISPER ENVIRONMENT CHECK
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Whisper imported successfully.


In [ ]:
# This cell defines the ES2004a AMI audio file that will be transcribed using Whisper.

AUDIO_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "ami",
    "ES2004a",
    "audio",
    "ES2004a.Mix-Headset.wav"
)

print("=" * 70)
print("WHISPER INPUT")
print("=" * 70)

print("Audio path:", AUDIO_PATH)
print("Exists:", os.path.exists(AUDIO_PATH))

if os.path.exists(AUDIO_PATH):
    size_mb = os.path.getsize(AUDIO_PATH) / (1024 * 1024)
    print(f"Audio size: {size_mb:.2f} MB")

WHISPER INPUT
Audio path: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/audio/ES2004a.Mix-Headset.wav
Exists: True
Audio size: 32.02 MB


In [ ]:
# This cell loads the Whisper-small model on the available GPU for the initial AMI ASR reproduction test.

WHISPER_MODEL_NAME = "small"

whisper_model = whisper.load_model(
    WHISPER_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("WHISPER MODEL LOADED")
print("=" * 70)

print("Model:", WHISPER_MODEL_NAME)
print("Device:", whisper_model.device)

100%|████████████████████████████████████████| 461M/461M [00:03<00:00, 149MiB/s]


WHISPER MODEL LOADED
Model: small
Device: cuda:0


In [ ]:
# This cell runs Whisper on the ES2004a audio and produces a timestamped ASR transcription.

print("=" * 70)
print("WHISPER ASR — ES2004a")
print("=" * 70)

whisper_result = whisper_model.transcribe(
    AUDIO_PATH,
    language="en",
    fp16=torch.cuda.is_available(),
    verbose=False
)

print("Transcription completed.")

print(
    "Number of segments:",
    len(whisper_result["segments"])
)

print("\nFirst 10 Whisper segments:\n")

for segment in whisper_result["segments"][:10]:

    print(
        f"[{segment['start']:.2f} - "
        f"{segment['end']:.2f}] "
        f"{segment['text'].strip()}"
    )

WHISPER ASR — ES2004a


100%|██████████| 104935/104935 [01:06<00:00, 1571.42frames/s]

Transcription completed.
Number of segments: 327

First 10 Whisper segments:

[0.00 - 16.00] We're not allowed to dim lights so we can see that a little better.
[16.00 - 17.00] Yeah.
[17.00 - 18.00] Okay.
[18.00 - 19.00] That's fine.
[19.00 - 24.00] Am I supposed to be standing up there?
[24.00 - 27.00] So we've got both of these clipped on.
[28.00 - 30.00] She can own some people.
[30.00 - 31.00] Yeah, it's good.
[31.00 - 32.00] Both of them.
[32.00 - 33.00] Okay.


In [ ]:
# This cell saves the Whisper ASR output for ES2004a without modifying the original AMI transcript.

import json

ASR_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "transcripts"
)

os.makedirs(ASR_OUTPUT_DIR, exist_ok=True)

ASR_OUTPUT_PATH = os.path.join(
    ASR_OUTPUT_DIR,
    "ES2004a_whisper_small.json"
)

with open(
    ASR_OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "meeting_id": "ES2004a",
            "model": "openai-whisper-small",
            "language": "en",
            "segments": whisper_result["segments"],
            "text": whisper_result["text"]
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 70)
print("WHISPER ASR SAVED")
print("=" * 70)

print("Path:", ASR_OUTPUT_PATH)
print("Exists:", os.path.exists(ASR_OUTPUT_PATH))

WHISPER ASR SAVED
Path: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_whisper_small.json
Exists: True


In [ ]:
# This cell prepares the Word Error Rate metric for evaluating our Whisper transcription against the AMI transcript.

import importlib.util

if importlib.util.find_spec("jiwer") is None:
    !pip install -q jiwer

from jiwer import wer

print("=" * 70)
print("WER EVALUATION READY")
print("=" * 70)

print("jiwer imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.9 MB/s eta 0:00:00
WER EVALUATION READY
jiwer imported successfully.


In [ ]:
# This cell calculates the Word Error Rate of Whisper-small against the original AMI ES2004a transcript.

AMI_TRANSCRIPT_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "ami",
    "ES2004a",
    "transcript",
    "ES2004a_transcript.txt"
)

with open(
    AMI_TRANSCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    ami_transcript = f.read()

whisper_transcript = whisper_result["text"]

es2004a_wer = wer(
    ami_transcript,
    whisper_transcript
)

print("=" * 70)
print("WHISPER ASR — WER RESULT")
print("=" * 70)

print("Meeting: ES2004a")
print("Model: openai-whisper-small")
print(f"WER: {es2004a_wer:.4f}")
print(f"WER (%): {es2004a_wer * 100:.2f}%")

WHISPER ASR — WER RESULT
Meeting: ES2004a
Model: openai-whisper-small
WER: 1.0000
WER (%): 100.00%


In [ ]:
# This cell removes AMI timestamps, speaker labels, punctuation, and formatting differences before calculating WER.

import re
from jiwer import wer

def normalize_ami_transcript(text):
    """
    Convert AMI timestamped/annotated transcript into plain reference text.
    """

    # Remove timestamp prefixes such as [10.99 - 11.02]
    text = re.sub(
        r"\[\s*\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?\s*\]",
        " ",
        text
    )

    # Remove speaker labels such as A:, B:, C:, D:
    text = re.sub(
        r"\b[A-D]\s*:",
        " ",
        text
    )

    # Convert to lowercase
    text = text.lower()

    # Keep words/numbers and remove punctuation
    text = re.sub(
        r"[^a-z0-9\s']",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# Normalize AMI reference
ami_reference_clean = normalize_ami_transcript(
    ami_transcript
)

# Normalize Whisper output using the same procedure
whisper_reference_clean = normalize_ami_transcript(
    whisper_transcript
)

# Calculate WER
es2004a_wer_clean = wer(
    ami_reference_clean,
    whisper_reference_clean
)

print("=" * 70)
print("CORRECTED WHISPER ASR — WER")
print("=" * 70)

print("Meeting:", "ES2004a")
print("Model:", "openai-whisper-small")

print(
    f"WER: {es2004a_wer_clean:.4f}"
)

print(
    f"WER (%): {es2004a_wer_clean * 100:.2f}%"
)

print("\nReference words:",
      len(ami_reference_clean.split()))

print("Whisper words:",
      len(whisper_reference_clean.split()))

CORRECTED WHISPER ASR — WER
Meeting: ES2004a
Model: openai-whisper-small
WER: 0.2914
WER (%): 29.14%

Reference words: 2653
Whisper words: 2122


In [ ]:
# This cell displays the beginning of the normalized AMI reference and Whisper transcription for a sanity check.

print("=" * 70)
print("NORMALIZED TRANSCRIPT COMPARISON")
print("=" * 70)

print("\nAMI REFERENCE — first 500 characters:")
print(ami_reference_clean[:500])

print("\nWHISPER — first 500 characters:")
print(whisper_reference_clean[:500])

NORMALIZED TRANSCRIPT COMPARISON

AMI REFERENCE — first 500 characters:
hmm hmm hmm are we we're not allowed to dim the lights so people can see that a bit better yeah okay that's fine am i supposed to be standing up there so okay we've got both of these clipped on she gonna answer me yeah or not i've got right both of them okay yes god jesus it's gonna fall off okay yep yep okay okay tu tu tu tu hello everybody hi good morning um i'm sarah the project manager and this is our first meeting surprisingly enough okay this is our agenda um we will do some stuff get to k

WHISPER — first 500 characters:
we're not allowed to dim lights so we can see that a little better yeah okay that's fine am i supposed to be standing up there so we've got both of these clipped on she can own some people yeah it's good both of them okay yeah good i'm just going to fall off it doesn't it okay hello everybody i'm sarah project manager and this is our first meeting surprisingly nice okay this is our agenda we

In [ ]:
# This cell runs Whisper-small on all 10 AMI meetings and saves each timestamped transcription separately.

MEETING_IDS = [
    "ES2004a",
    "ES2004b",
    "ES2004c",
    "ES2004d",
    "ES2005a",
    "ES2005b",
    "ES2005c",
    "ES2006a",
    "ES2006b",
    "ES2008a"
]

ASR_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "transcripts"
)

os.makedirs(ASR_OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("WHISPER ASR — ALL AMI MEETINGS")
print("=" * 70)

asr_results = {}

for index, meeting_id in enumerate(MEETING_IDS, start=1):

    print(
        f"\n[{index}/{len(MEETING_IDS)}] "
        f"Processing {meeting_id}..."
    )

    audio_path = os.path.join(
        PROJECT_DIR,
        "data",
        "raw",
        "ami",
        meeting_id,
        "audio",
        f"{meeting_id}.Mix-Headset.wav"
    )

    output_path = os.path.join(
        ASR_OUTPUT_DIR,
        f"{meeting_id}_whisper_small.json"
    )

    if not os.path.exists(audio_path):
        print("⚠ Audio missing:", audio_path)
        continue

    result = whisper_model.transcribe(
        audio_path,
        language="en",
        fp16=torch.cuda.is_available(),
        verbose=False
    )

    asr_results[meeting_id] = result

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            {
                "meeting_id": meeting_id,
                "model": "openai-whisper-small",
                "language": "en",
                "segments": result["segments"],
                "text": result["text"]
            },
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        "  Segments:",
        len(result["segments"])
    )

    print(
        "  Saved:",
        output_path
    )

print("\n" + "=" * 70)
print("WHISPER ASR RUN COMPLETE")
print("=" * 70)

print(
    "Meetings processed:",
    len(asr_results)
)

WHISPER ASR — ALL AMI MEETINGS

[1/10] Processing ES2004a...


100%|██████████| 104935/104935 [01:00<00:00, 1723.92frames/s]


  Segments: 350
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_whisper_small.json

[2/10] Processing ES2004b...


100%|██████████| 234549/234549 [02:28<00:00, 1584.56frames/s]


  Segments: 1017
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004b_whisper_small.json

[3/10] Processing ES2004c...


...

  Segments: 165
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2006a_whisper_small.json

[9/10] Processing ES2006b...


100%|██████████| 218312/218312 [02:06<00:00, 1726.02frames/s]


  Segments: 535
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2006b_whisper_small.json

[10/10] Processing ES2008a...


 97%|█████████▋| 101336/104336 [01:00<00:01, 1686.68frames/s]

  Segments: 236
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2008a_whisper_small.json

WHISPER ASR RUN COMPLETE
Meetings processed: 10


In [ ]:
# This cell calculates normalized Word Error Rate for Whisper-small across all 10 AMI meetings.

import pandas as pd
import json
import os

wer_results = []

for meeting_id in MEETING_IDS:

    print(f"Evaluating {meeting_id}...")

    # AMI reference transcript
    ami_path = os.path.join(
        PROJECT_DIR,
        "data",
        "raw",
        "ami",
        meeting_id,
        "transcript",
        f"{meeting_id}_transcript.txt"
    )

    # Our Whisper output
    whisper_path = os.path.join(
        PROJECT_DIR,
        "data",
        "transcripts",
        f"{meeting_id}_whisper_small.json"
    )

    if not os.path.exists(ami_path):
        print("  ⚠ AMI transcript missing")
        continue

    if not os.path.exists(whisper_path):
        print("  ⚠ Whisper output missing")
        continue

    with open(ami_path, "r", encoding="utf-8") as f:
        ami_text = f.read()

    with open(whisper_path, "r", encoding="utf-8") as f:
        whisper_data = json.load(f)

    whisper_text = whisper_data["text"]

    # Normalize both transcripts
    reference = normalize_ami_transcript(ami_text)
    hypothesis = normalize_ami_transcript(whisper_text)

    meeting_wer = wer(
        reference,
        hypothesis
    )

    wer_results.append({
        "meeting_id": meeting_id,
        "reference_words": len(reference.split()),
        "whisper_words": len(hypothesis.split()),
        "wer": meeting_wer,
        "wer_percent": meeting_wer * 100
    })

    print(
        f"  WER: {meeting_wer:.4f} "
        f"({meeting_wer * 100:.2f}%)"
    )


wer_df = pd.DataFrame(wer_results)

print("\n" + "=" * 70)
print("WHISPER ASR — ALL MEETINGS WER")
print("=" * 70)

display(wer_df)

print("\nAverage WER:",
      f"{wer_df['wer'].mean():.4f}")

print("Average WER (%):",
      f"{wer_df['wer_percent'].mean():.2f}%")

Evaluating ES2004a...
  WER: 0.3072 (30.72%)
Evaluating ES2004b...
  WER: 0.2834 (28.34%)
Evaluating ES2004c...
  WER: 0.2482 (24.82%)
Evaluating ES2004d...
  WER: 0.3290 (32.90%)
Evaluating ES2005a...
  WER: 0.3835 (38.35%)
Evaluating ES2005b...
  WER: 0.3194 (31.94%)
Evaluating ES2005c...
  WER: 0.3234 (32.34%)
Evaluating ES2006a...
  WER: 0.2099 (20.99%)
Evaluating ES2006b...
  WER: 0.2624 (26.24%)
Evaluating ES2008a...
  WER: 0.2778 (27.78%)

WHISPER ASR — ALL MEETINGS WER


,meeting_id,reference_words,whisper_words,wer,wer_percent
0,ES2004a,2653,2131,0.307199,30.719940
1,ES2004b,6874,5587,0.283387,28.338667
2,ES2004c,7084,5835,0.248165,24.816488
3,ES2004d,6204,4982,0.328981,32.898130
4,ES2005a,764,694,0.383508,38.350785
5,ES2005b,6297,4934,0.319358,31.935842
6,ES2005c,6802,5340,0.323434,32.343428
7,ES2006a,2806,2326,0.209907,20.990734
8,ES2006b,6277,5057,0.262386,26.238649
9,ES2008a,2541,2161,0.277843,27.784337



Average WER: 0.2944
Average WER (%): 29.44%


In [ ]:
# This cell saves the Whisper WER evaluation so it can be used later in the thesis and final comparison.

WER_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "evaluation_results"
)

os.makedirs(
    WER_OUTPUT_DIR,
    exist_ok=True
)

WER_OUTPUT_PATH = os.path.join(
    WER_OUTPUT_DIR,
    "whisper_small_wer.csv"
)

wer_df.to_csv(
    WER_OUTPUT_PATH,
    index=False
)

print("=" * 70)
print("WHISPER WER RESULTS SAVED")
print("=" * 70)

print(WER_OUTPUT_PATH)

WHISPER WER RESULTS SAVED
/content/drive/MyDrive/MTechIndProj/MoM_Project/evaluation_results/whisper_small_wer.csv


In [ ]:
# This cell checks the installed pyannote.audio version and confirms whether the diarization package is available.

import importlib.util

print("=" * 70)
print("PYANNOTE DIARIZATION ENVIRONMENT CHECK")
print("=" * 70)

pyannote_available = (
    importlib.util.find_spec("pyannote.audio") is not None
)

print("pyannote.audio available:", pyannote_available)

if pyannote_available:
    import pyannote.audio
    print("pyannote.audio version:", pyannote.audio.__version__)
else:
    print("pyannote.audio is not installed.")

PYANNOTE DIARIZATION ENVIRONMENT CHECK


ModuleNotFoundError: No module named 'pyannote'

In [ ]:
# This cell installs pyannote.audio 3.1 for the speaker diarization stage of the project.

!pip install -q "pyannote.audio==3.1.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.7/208.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/

In [ ]:
# This cell checks the current NumPy, Numba, PyTorch, and pyannote versions before fixing the dependency conflict.

import numpy
import torch

print("=" * 70)
print("ENVIRONMENT VERSION CHECK")
print("=" * 70)

print("NumPy:", numpy.__version__)
print("PyTorch:", torch.__version__)

try:
    import numba
    print("Numba:", numba.__version__)
except Exception as e:
    print("Numba import error:", e)

try:
    import pyannote.audio
    print("pyannote.audio:", pyannote.audio.__version__)
except Exception as e:
    print("pyannote import error:", e)

ENVIRONMENT VERSION CHECK
NumPy: 2.1.3
PyTorch: 2.11.0+cu128
Numba: 0.61.2
pyannote import error: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'


In [ ]:
# This cell checks the scientific Python packages involved in the pyannote import error before making any environment changes.

import numpy
import scipy
import sklearn
import torch

print("=" * 70)
print("PYANNOTE DEPENDENCY DIAGNOSTICS")
print("=" * 70)

print("Python:", __import__("sys").version)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)

try:
    import torchaudio
    print("Torchaudio:", torchaudio.__version__)
except Exception as e:
    print("Torchaudio ERROR:", repr(e))

try:
    import librosa
    print("Librosa:", librosa.__version__)
except Exception as e:
    print("Librosa ERROR:", repr(e))

try:
    import pyannote.core
    print("pyannote.core:", pyannote.core.__version__)
except Exception as e:
    print("pyannote.core ERROR:", repr(e))

AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'

In [ ]:
# This cell repairs the NumPy, SciPy, and scikit-learn compatibility stack required by pyannote.

!pip install -q --force-reinstall --no-cache-dir \
    "numpy==2.2.6" \
    "scipy==1.15.3" \
    "scikit-learn==1.7.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 329.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 297.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 321.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 351.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 428.5 MB/s eta 0:00:00


In [ ]:
# This cell verifies that NumPy, SciPy, and scikit-learn can import correctly after the repair.

import numpy
import scipy
import sklearn

print("=" * 70)
print("REPAIRED SCIENTIFIC STACK")
print("=" * 70)

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

AttributeError: 'numpy.ufunc' object has no attribute '__module__' and no __dict__ for setting new attributes

In [ ]:
# This cell checks the fresh Colab runtime before installing any additional diarization dependencies.

import sys
import numpy
import torch

print("=" * 70)
print("FRESH COLAB ENVIRONMENT")
print("=" * 70)

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

FRESH COLAB ENVIRONMENT
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
NumPy: 2.2.6
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# This cell restores the project directory and AMI paths after restarting the Colab runtime.

import os
import torch

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

AMI_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "ami"
)

TRANSCRIPT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "transcripts"
)

EVALUATION_DIR = os.path.join(
    PROJECT_DIR,
    "evaluation_results"
)

print("=" * 70)
print("PROJECT ENVIRONMENT RESTORED")
print("=" * 70)

print("Project:", PROJECT_DIR)
print("AMI:", AMI_DIR)
print("Transcripts:", TRANSCRIPT_DIR)
print("Evaluation:", EVALUATION_DIR)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT ENVIRONMENT RESTORED
Project: /content/drive/MyDrive/MTechIndProj/MoM_Project
AMI: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami
Transcripts: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts
Evaluation: /content/drive/MyDrive/MTechIndProj/MoM_Project/evaluation_results

CUDA available: True
GPU: Tesla T4


In [ ]:
# This cell checks whether pyannote.audio is available in the fresh runtime without modifying the environment.

import importlib.util

pyannote_spec = importlib.util.find_spec("pyannote")

print("=" * 70)
print("PYANNOTE AVAILABILITY")
print("=" * 70)

print("pyannote package available:", pyannote_spec is not None)

if pyannote_spec is not None:
    try:
        import pyannote.audio
        print("pyannote.audio version:", pyannote.audio.__version__)
    except Exception as e:
        print("pyannote.audio import error:", repr(e))
else:
    print("pyannote is not installed.")

PYANNOTE AVAILABILITY
pyannote package available: True
pyannote.audio import error: AttributeError("module 'torchaudio' has no attribute 'set_audio_backend'")


In [ ]:
# This cell installs the current pyannote.audio package for speaker diarization.

!pip install -q pyannote.audio

In [ ]:
# This cell verifies that pyannote.audio imports correctly in the clean runtime.

import pyannote.audio

print("=" * 70)
print("PYANNOTE INSTALLATION CHECK")
print("=" * 70)

print("pyannote.audio version:", pyannote.audio.__version__)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

AttributeError: module 'torchaudio' has no attribute 'set_audio_backend'

In [ ]:
import importlib.metadata as metadata

print("=" * 70)
print("PYANNOTE / TORCHAUDIO VERSIONS")
print("=" * 70)

print("pyannote.audio:",
      metadata.version("pyannote.audio"))

print("torchaudio:",
      metadata.version("torchaudio"))

print("torch:",
      metadata.version("torch"))

PYANNOTE / TORCHAUDIO VERSIONS
pyannote.audio: 3.1.1
torchaudio: 2.11.0+cu128
torch: 2.11.0+cu128


In [ ]:
# Remove the incompatible pyannote.audio 3.1.1
!pip uninstall -y pyannote.audio

# Install the current pyannote.audio
!pip install -q -U pyannote.audio

Found existing installation: pyannote.audio 3.1.1
Uninstalling pyannote.audio-3.1.1:
  Successfully uninstalled pyannote.audio-3.1.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 20.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [ ]:
import importlib.metadata as metadata

print("=" * 70)
print("PYANNOTE VERSION AFTER UPDATE")
print("=" * 70)

print("pyannote.audio:",
      metadata.version("pyannote.audio"))

print("torchaudio:",
      metadata.version("torchaudio"))

print("torch:",
      metadata.version("torch"))

PYANNOTE VERSION AFTER UPDATE
pyannote.audio: 4.0.7
torchaudio: 2.11.0+cu128
torch: 2.11.0+cu128


In [ ]:
import torch
import pyannote.audio

print("=" * 70)
print("PYANNOTE IMPORT TEST")
print("=" * 70)

print("pyannote.audio:",
      pyannote.audio.__version__)

print("PyTorch:",
      torch.__version__)

print("CUDA:",
      torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:",
          torch.cuda.get_device_name(0))

PYANNOTE IMPORT TEST
pyannote.audio: 4.0.7
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


If you already have a Hugging Face token, you can log in through Colab's secret mechanism rather than putting the token directly in the notebook.

In Colab:

Secrets → add HF_TOKEN

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("=" * 70)
print("HUGGING FACE TOKEN CHECK")
print("=" * 70)

if HF_TOKEN:
    print("HF_TOKEN found: True")
    print("Token length:", len(HF_TOKEN))
else:
    print("HF_TOKEN found: False")

HUGGING FACE TOKEN CHECK
HF_TOKEN found: True
Token length: 37


In [ ]:
from pyannote.audio import Pipeline
import torch

print("=" * 70)
print("LOADING PYANNOTE DIARIZATION PIPELINE")
print("=" * 70)

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

pipeline.to(torch.device("cuda"))

print("Diarization pipeline loaded successfully.")
print("Device: CUDA")

ModuleNotFoundError: No module named 'pyannote'

In [ ]:
!pip install -q pyannote.audio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/

In [ ]:
import importlib.metadata as metadata

print("=" * 70)
print("PYANNOTE INSTALLATION")
print("=" * 70)

print("pyannote.audio:",
      metadata.version("pyannote.audio"))

print("torchaudio:",
      metadata.version("torchaudio"))

print("torch:",
      metadata.version("torch"))

PYANNOTE INSTALLATION
pyannote.audio: 4.0.7
torchaudio: 2.11.0+cpu
torch: 2.11.0+cpu


In [ ]:
import pyannote.audio

print("=" * 70)
print("PYANNOTE IMPORT TEST")
print("=" * 70)

print("pyannote.audio:",
      pyannote.audio.__version__)

ImportError: cannot import name '_slice' from 'numpy._core.umath' (/usr/local/lib/python3.13/dist-packages/numpy/_core/umath.py)

In [ ]:
!pip install -q --force-reinstall \
    torch==2.11.0 \
    torchaudio==2.11.0 \
    --index-url https://download.pytorch.org/whl/cu128

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 22.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 66.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 74.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 35.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 156.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 89.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 171.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 156.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 93.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 55.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 55.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import torchaudio
import numpy

print("=" * 70)
print("POST-REPAIR ENVIRONMENT CHECK")
print("=" * 70)

print("NumPy:", numpy.__version__)
print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

POST-REPAIR ENVIRONMENT CHECK
NumPy: 2.5.2
PyTorch: 2.11.0+cu128
TorchAudio: 2.11.0+cu128
CUDA available: False


In [ ]:
import torch

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU CHECK
PyTorch: 2.11.0+cpu
CUDA: False
CUDA version: None
GPU count: 0


In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import os
import torch
import subprocess

print("=" * 70)
print("COLAB GPU DIAGNOSTIC")
print("=" * 70)

print("CUDA_VISIBLE_DEVICES:",
      os.environ.get("CUDA_VISIBLE_DEVICES"))

print("NVIDIA_VISIBLE_DEVICES:",
      os.environ.get("NVIDIA_VISIBLE_DEVICES"))

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

print("\nChecking NVIDIA driver...")

result = subprocess.run(
    ["bash", "-c", "command -v nvidia-smi || true"],
    capture_output=True,
    text=True
)

print("nvidia-smi location:", result.stdout.strip() or "NOT FOUND")

COLAB GPU DIAGNOSTIC
CUDA_VISIBLE_DEVICES: None
NVIDIA_VISIBLE_DEVICES: all
PyTorch: 2.11.0+cu128
PyTorch CUDA: 12.8
CUDA available: True
GPU count: 1

Checking NVIDIA driver...
nvidia-smi location: /opt/bin/nvidia-smi


In [ ]:
import torch

print("=" * 70)
print("GPU CONFIRMATION")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU CONFIRMATION
CUDA available: True
GPU count: 1
GPU: Tesla T4


In [ ]:
import pyannote.audio

print("=" * 70)
print("PYANNOTE CHECK")
print("=" * 70)

print("pyannote.audio:",
      pyannote.audio.__version__)

ModuleNotFoundError: No module named 'pyannote'

In [ ]:
!pip install -q "pyannote.audio==4.0.7"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983

In [ ]:
import importlib.metadata as metadata

print("=" * 70)
print("PYANNOTE INSTALLATION CHECK")
print("=" * 70)

print("pyannote.audio:", metadata.version("pyannote.audio"))
print("torch:", metadata.version("torch"))
print("torchaudio:", metadata.version("torchaudio"))

PYANNOTE INSTALLATION CHECK
pyannote.audio: 4.0.7
torch: 2.11.0+cu128
torchaudio: 2.11.0+cu128


In [ ]:
from pyannote.audio import Pipeline
import torch

print("=" * 70)
print("LOADING PYANNOTE DIARIZATION PIPELINE")
print("=" * 70)

# Retrieve the Hugging Face token from Colab Secrets
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN available:", bool(HF_TOKEN))

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

# Use GPU
pipeline.to(torch.device("cuda"))

print("Diarization pipeline loaded successfully.")
print("Device: CUDA")

ImportError: cannot import name '_slice' from 'numpy._core.umath' (/usr/local/lib/python3.13/dist-packages/numpy/_core/umath.py)

In [ ]:
!pip install -q --force-reinstall "numpy==2.2.6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 106.4 MB/s eta 0:00:00


In [ ]:
import numpy
import scipy

print("=" * 70)
print("NUMPY / SCIPY CHECK")
print("=" * 70)

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)

NUMPY / SCIPY CHECK
NumPy: 2.1.3
SciPy: 1.16.3


In [ ]:
import torch

print("=" * 70)
print("CUDA CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA CHECK
PyTorch: 2.11.0+cu128
CUDA: True
GPU count: 1
GPU: Tesla T4


In [ ]:
import pyannote.audio

print("=" * 70)
print("PYANNOTE IMPORT CHECK")
print("=" * 70)

print("pyannote.audio:", pyannote.audio.__version__)

AttributeError: 'numpy.ufunc' object has no attribute '__module__' and no __dict__ for setting new attributes

In [ ]:
import sys
import numpy

print("Python:", sys.version)
print("NumPy:", numpy.__version__)

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
NumPy: 2.1.3


In [ ]:
!pip install -q --force-reinstall \
    "numpy==2.2.6" \
    "scipy==1.15.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 25.6 MB/s eta 0:00:00


In [ ]:
import numpy
import scipy

print("=" * 70)
print("SCIENTIFIC STACK CHECK")
print("=" * 70)

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)

SCIENTIFIC STACK CHECK
NumPy: 2.2.6
SciPy: 1.15.3


In [ ]:
import torch

print("=" * 70)
print("PYTORCH / GPU CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PYTORCH / GPU CHECK
PyTorch: 2.11.0+cu128
CUDA: True
GPU count: 1
GPU: Tesla T4


In [ ]:
import scipy.signal
import torchmetrics
import lightning

print("=" * 70)
print("DEPENDENCY CHAIN CHECK")
print("=" * 70)

print("SciPy: OK")
print("TorchMetrics: OK")
print("Lightning: OK")

DEPENDENCY CHAIN CHECK
SciPy: OK
TorchMetrics: OK
Lightning: OK


In [ ]:
import pyannote.audio

print("=" * 70)
print("PYANNOTE IMPORT CHECK")
print("=" * 70)

print("pyannote.audio:", pyannote.audio.__version__)

PYANNOTE IMPORT CHECK
pyannote.audio: 4.0.7


In [ ]:
import torch
from pyannote.audio import Pipeline
from google.colab import userdata

print("=" * 70)
print("LOADING PYANNOTE DIARIZATION PIPELINE")
print("=" * 70)

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN available:", bool(HF_TOKEN))
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

pipeline.to(torch.device("cuda"))

print("\nDiarization pipeline loaded successfully.")
print("Device: CUDA")

LOADING PYANNOTE DIARIZATION PIPELINE
HF_TOKEN available: True
CUDA available: True
GPU: Tesla T4


config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

segmentation/pytorch_model.bin: downloading bytes:           |  0.00B            

plda/xvec_transform.npz: reconstructing file:   0%|          |  0.00B /  134kB            

...

plda/plda.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/plda.npz: downloading bytes:           |  0.00B            

embedding/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

embedding/pytorch_model.bin: downloading bytes:           |  0.00B            


Diarization pipeline loaded successfully.
Device: CUDA


In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


# WhisperX Automatic Speech Recognition and Word Alignment

### Purpose
This notebook converts the meeting recording into a timestamped
transcript using WhisperX.

### Pipeline Position
Recorded Audio
→ Silero VAD
→ **WhisperX ASR**
→ Word-Level Alignment
→ Pyannote Speaker Diarization

### Expected Output
A timestamped transcript containing:
- Segment-level text
- Word-level timestamps
- Start and end times

These timestamps will later be combined with Pyannote speaker
labels to create speaker-attributed transcript evidence.

In [2]:
# ============================================================
# CELL 16: INSTALL WHISPERX
# Purpose:
# Install WhisperX for automatic speech recognition and
# word-level timestamp alignment.
#
# Important:
# We intentionally do NOT reinstall PyTorch, torchaudio,
# or Pyannote because the current environment is already
# working correctly.
# ============================================================

!pip install -q whisperx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 799.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [3]:
# ============================================================
# CELL 17: VERIFY WHISPERX INSTALLATION
# Purpose:
# Confirm that WhisperX can be imported successfully after
# installation and check the installed version.
# ============================================================

import whisperx

print("WhisperX imported successfully.")
print("WhisperX version:", getattr(whisperx, "__version__", "Version not exposed"))

WhisperX imported successfully.
WhisperX version: Version not exposed


In [4]:
# ============================================================
# CELL 18: VERIFY WHISPERX GPU ENVIRONMENT
# Purpose:
# Confirm that the current runtime provides CUDA/GPU support
# for WhisperX ASR processing.
# ============================================================

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("WhisperX device :", DEVICE)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU device      :", torch.cuda.get_device_name(0))
    print("CUDA version    :", torch.version.cuda)
else:
    print("WARNING: CUDA is not available. WhisperX will run on CPU.")

WhisperX device : cuda
CUDA available  : True
GPU device      : Tesla T4
CUDA version    : 12.8


In [5]:
# ============================================================
# CELL 19: LOAD WHISPERX ASR MODEL
# Purpose:
# Load the Whisper Small speech recognition model using
# WhisperX for meeting transcription.
#
# The model will run on the available Tesla T4 GPU.
# ============================================================

import whisperx

# Select device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Whisper model configuration
WHISPER_MODEL = "small"

print("Loading WhisperX ASR model...")
print("Model :", WHISPER_MODEL)
print("Device:", DEVICE)

whisper_model = whisperx.load_model(
    WHISPER_MODEL,
    DEVICE,
    compute_type="float16"
)

print("\nWhisperX ASR model loaded successfully.")

Loading WhisperX ASR model...
Model : small
Device: cuda


ImportError: cannot import name '_slice' from 'numpy._core.umath' (/usr/local/lib/python3.13/dist-packages/numpy/_core/umath.py)

In [6]:
# ============================================================
# CELL 19A: FIX NUMPY / SCIPY / SCIKIT-LEARN COMPATIBILITY
# Purpose:
# Repair the NumPy/SciPy/scikit-learn dependency mismatch
# introduced while installing WhisperX.
#
# We keep the existing PyTorch, CUDA, and Pyannote setup intact.
# ============================================================

!pip install -q --force-reinstall \
    "numpy==2.2.6" \
    "scipy==1.15.3" \
    "scikit-learn==1.6.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 21.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
# ============================================================
# CELL 20: VERIFY REPAIRED PYTHON ENVIRONMENT
# Purpose:
# Verify that NumPy, SciPy, scikit-learn, PyTorch, CUDA,
# and WhisperX are working correctly after the restart.
# ============================================================

import numpy as np
import scipy
import sklearn
import torch
import whisperx

print("NumPy version        :", np.__version__)
print("SciPy version        :", scipy.__version__)
print("scikit-learn version :", sklearn.__version__)
print("PyTorch version      :", torch.__version__)
print("CUDA available       :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU device           :", torch.cuda.get_device_name(0))

print("WhisperX import      : SUCCESS")

NumPy version        : 2.2.6
SciPy version        : 1.15.3
scikit-learn version : 1.6.1
PyTorch version      : 2.8.0+cu128
CUDA available       : True
GPU device           : Tesla T4
WhisperX import      : SUCCESS


In [2]:
# ============================================================
# CELL 21: LOAD WHISPERX SMALL MODEL
# Purpose:
# Load the Whisper Small ASR model through WhisperX.
# The model will use the Tesla T4 GPU with FP16 precision.
# ============================================================

import whisperx
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

WHISPER_MODEL = "small"

print("Loading WhisperX ASR model...")
print("Model :", WHISPER_MODEL)
print("Device:", DEVICE)

whisper_model = whisperx.load_model(
    WHISPER_MODEL,
    DEVICE,
    compute_type="float16"
)

print("\nWhisperX ASR model loaded successfully.")

Loading WhisperX ASR model...
Model : small
Device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/484M [00:00<?, ?B/s]

2026-09-03 05:42:15 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-09-03 05:42:15 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`



WhisperX ASR model loaded successfully.


In [3]:
# ============================================================
# CELL 22: RUN WHISPERX ASR
# Purpose:
# Transcribe the AMI meeting recording using WhisperX.
# The output will contain speech segments with timestamps.
#
# The detected language will be English for this AMI meeting.
# ============================================================

print("Starting WhisperX transcription...")
print("Input audio:", INPUT_AUDIO)

# Load the audio
audio = whisperx.load_audio(str(INPUT_AUDIO))

# Transcribe the meeting
result = whisper_model.transcribe(
    audio,
    batch_size=8,
    language="en"
)

print("\nWhisperX transcription completed.")
print("Detected/selected language:", result.get("language"))

print("Number of transcript segments:", len(result["segments"]))

Starting WhisperX transcription...


NameError: name 'INPUT_AUDIO' is not defined

In [5]:
# ============================================================
# CELL 22A: RESTORE PROJECT PATHS
# Purpose:
# Recreate the project and input-audio paths after the Colab
# runtime restart.
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

DATA_DIR = PROJECT_DIR / "data"

INPUT_AUDIO = (
    DATA_DIR
    / "raw"
    / "ami"
    / "ES2004a"
    / "audio"
    / "ES2004a.Mix-Headset.wav"
)

print("Project directory:", PROJECT_DIR)
print("Input audio:", INPUT_AUDIO)
print("File exists:", INPUT_AUDIO.exists())

Project directory: /content/drive/MyDrive/MTechIndProj/MoM_Project
Input audio: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/audio/ES2004a.Mix-Headset.wav
File exists: True


In [6]:
# ============================================================
# CELL 22: RUN WHISPERX ASR
# Purpose:
# Transcribe the AMI meeting recording using WhisperX.
# The output will contain segment-level text and timestamps.
# ============================================================

import whisperx

print("Starting WhisperX transcription...")
print("Input audio:", INPUT_AUDIO)

# Load audio
audio = whisperx.load_audio(str(INPUT_AUDIO))

# Transcribe using Whisper Small
result = whisper_model.transcribe(
    audio,
    batch_size=8,
    language="en"
)

print("\nWhisperX transcription completed.")
print("Language:", result.get("language"))
print("Number of transcript segments:", len(result["segments"]))

Starting WhisperX transcription...
Input audio: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/audio/ES2004a.Mix-Headset.wav


/usr/local/lib/python3.13/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(



WhisperX transcription completed.
Language: en
Number of transcript segments: 43


In [8]:
# ============================================================
# CELL 23: INSPECT WHISPERX TRANSCRIPT
# Purpose:
# Display the first 10 transcript segments produced by
# WhisperX along with their start/end timestamps.
# This verifies the transcription before word-level alignment.
# ============================================================

def format_timestamp(seconds):
    """Convert seconds into HH:MM:SS.ss format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60

    return f"{hours:02d}:{minutes:02d}:{secs:05.2f}"


print("First 10 WhisperX transcript segments:\n")

for i, segment in enumerate(result["segments"][:10], start=1):

    start = segment["start"]
    end = segment["end"]
    text = segment["text"].strip()

    print(
        f"Segment {i:02d}: "
        f"{format_timestamp(start)} → "
        f"{format_timestamp(end)}"
    )

    print(f"Text: {text}")
    print("-" * 80)

First 10 WhisperX transcript segments:

Segment 01: 00:00:10.90 → 00:00:32.31
Text: Are we, we're not like the dim lights, so we can see that a bit better. Yeah. Okay, that's fine. Am I supposed to be standing up there? So we've got both of these clipped on. Is she gonna answer me? Yeah, I've got both of them.
--------------------------------------------------------------------------------
Segment 02: 00:00:50.60 → 00:01:12.10
Text: I'm just gonna follow if it doesn't let you put it on. Yep, yep, yep. Okay.
--------------------------------------------------------------------------------
Segment 03: 00:01:20.47 → 00:01:35.51
Text: Okay, hello everybody. I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.
--------------------------------------------------------------------------------
Segment 04: 00:01:35.51 → 00:01:55.93
Text: We will do some stuff, get to know each other a bit better, feel more comfortable with each other. Then 

In [9]:
# ============================================================
# CELL 24: LOAD WHISPERX ALIGNMENT MODEL
# Purpose:
# Load the language-specific alignment model required to
# obtain accurate word-level timestamps from the WhisperX
# transcript.
# ============================================================

print("Loading WhisperX alignment model...")

align_model, align_metadata = whisperx.load_align_model(
    language_code="en",
    device=DEVICE
)

print("WhisperX alignment model loaded successfully.")
print("Alignment language:", align_metadata["language"])

Loading WhisperX alignment model...
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:04<00:00, 86.2MB/s]


WhisperX alignment model loaded successfully.
Alignment language: en


In [10]:
# ============================================================
# CELL 24: LOAD WHISPERX ALIGNMENT MODEL
# Purpose:
# Load the language-specific alignment model required to
# obtain accurate word-level timestamps from the WhisperX
# transcript.
# ============================================================

print("Loading WhisperX alignment model...")

align_model, align_metadata = whisperx.load_align_model(
    language_code="en",
    device=DEVICE
)

print("WhisperX alignment model loaded successfully.")
print("Alignment language:", align_metadata["language"])

Loading WhisperX alignment model...
WhisperX alignment model loaded successfully.
Alignment language: en


In [11]:
# ============================================================
# CELL 25: PERFORM WORD-LEVEL ALIGNMENT
# Purpose:
# Align the WhisperX transcript with the original audio to
# obtain accurate word-level start and end timestamps.
#
# These timestamps will later support speaker-word alignment
# and timestamp-linked evidence retrieval.
# ============================================================

print("Starting word-level alignment...")

aligned_result = whisperx.align(
    result["segments"],
    align_model,
    align_metadata,
    audio,
    DEVICE,
    return_char_alignments=False
)

print("\nWord-level alignment completed successfully.")
print("Number of aligned segments:", len(aligned_result["segments"]))

Starting word-level alignment...

Word-level alignment completed successfully.
Number of aligned segments: 236


In [12]:
# ============================================================
# CELL 26: INSPECT WORD-LEVEL TIMESTAMPS
# Purpose:
# Display the first few aligned transcript segments and their
# word-level timestamps to verify the WhisperX alignment output.
# ============================================================

print("First 3 aligned segments with word timestamps:\n")

shown_segments = 0

for segment in aligned_result["segments"]:

    if "words" not in segment:
        continue

    words = segment["words"]

    if not words:
        continue

    shown_segments += 1

    print(
        f"Segment {shown_segments}: "
        f"{format_timestamp(segment['start'])} → "
        f"{format_timestamp(segment['end'])}"
    )

    for word in words[:15]:
        word_text = word.get("word", "").strip()
        word_start = word.get("start")
        word_end = word.get("end")

        if word_start is not None and word_end is not None:
            print(
                f"  {format_timestamp(word_start)} → "
                f"{format_timestamp(word_end)} : "
                f"{word_text}"
            )

    print("-" * 80)

    if shown_segments >= 3:
        break

First 3 aligned segments with word timestamps:

Segment 1: 00:00:11.00 → 00:00:14.52
  00:00:11.00 → 00:00:11.16 : Are
  00:00:11.18 → 00:00:11.38 : we,
  00:00:12.18 → 00:00:12.32 : we're
  00:00:12.36 → 00:00:12.46 : not
  00:00:12.48 → 00:00:12.62 : like
  00:00:12.64 → 00:00:12.70 : the
  00:00:12.72 → 00:00:12.92 : dim
  00:00:12.94 → 00:00:13.18 : lights,
  00:00:13.20 → 00:00:13.34 : so
  00:00:13.36 → 00:00:13.52 : we
  00:00:13.58 → 00:00:13.72 : can
  00:00:13.74 → 00:00:13.84 : see
  00:00:13.86 → 00:00:14.02 : that
  00:00:14.08 → 00:00:14.12 : a
  00:00:14.14 → 00:00:14.26 : bit
--------------------------------------------------------------------------------
Segment 2: 00:00:17.94 → 00:00:18.16
  00:00:17.94 → 00:00:18.16 : Yeah.
--------------------------------------------------------------------------------
Segment 3: 00:00:18.94 → 00:00:20.95
  00:00:18.94 → 00:00:19.22 : Okay,
  00:00:19.72 → 00:00:19.94 : that's
  00:00:19.96 → 00:00:20.95 : fine.
--------------------

In [13]:
# ============================================================
# CELL 27: SAVE WHISPERX ALIGNED TRANSCRIPT
# Purpose:
# Save the WhisperX word-aligned transcript as a JSON file.
# This preserves segment-level and word-level timestamps for
# later speaker attribution and evidence retrieval.
# ============================================================

import json

# Create transcript output directory
TRANSCRIPT_DIR = DATA_DIR / "transcripts"
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

# Output file
WHISPERX_OUTPUT_FILE = (
    TRANSCRIPT_DIR / "ES2004a_whisperx_aligned.json"
)

# Save aligned result
with open(WHISPERX_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(aligned_result, f, indent=2, ensure_ascii=False)

print("WhisperX aligned transcript saved successfully.")
print("Output file:", WHISPERX_OUTPUT_FILE)
print("Aligned segments:", len(aligned_result["segments"]))

WhisperX aligned transcript saved successfully.
Output file: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_whisperx_aligned.json
Aligned segments: 236


In [14]:
# ============================================================
# CELL 28: CHECK PYANNOTE AUDIO
# Purpose:
# Verify that Pyannote Audio is still installed and importable
# after the WhisperX dependency changes.
#
# We do NOT reinstall Pyannote because it was already working
# with the Tesla T4 environment.
# ============================================================

import pyannote.audio

print("Pyannote Audio imported successfully.")
print(
    "Pyannote Audio version:",
    getattr(pyannote.audio, "__version__", "Version not exposed")
)

Pyannote Audio imported successfully.
Pyannote Audio version: 4.0.7


In [15]:
# ============================================================
# CELL 29: LOAD PYANNOTE SPEAKER DIARIZATION PIPELINE
# Purpose:
# Load the pretrained Pyannote speaker diarization pipeline.
#
# Pipeline Position:
# Audio → Silero VAD → WhisperX ASR → Alignment
#                              ↓
#                    Pyannote Diarization
#
# Expected Output:
# A diarization pipeline ready to identify
# "who spoke when" in the meeting recording.
# ============================================================

from pyannote.audio import Pipeline

# Load pretrained speaker diarization pipeline
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1"
)

# Move pipeline to GPU if CUDA is available
if torch.cuda.is_available():
    diarization_pipeline.to(torch.device("cuda"))

print("Pyannote speaker diarization pipeline loaded successfully.")
print("Device:", DEVICE)


Could not download Pipeline from pyannote/speaker-diarization-3.1.
It might be because the repository is private or gated:

* visit https://hf.co/pyannote/speaker-diarization-3.1 to accept user conditions
* visit https://hf.co/settings/tokens to create an authentication token
* load the Pipeline with the `token` argument:
    >>> Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', token='hf_....')



GatedRepoError: 401 Client Error. (Request ID: Root=1-6a990ed4-0cad120678544a6361c3c69d;13374d73-888a-4536-aa3b-84a71a48ea7e)

Cannot access gated repo for url https://huggingface.co/pyannote/speaker-diarization-3.1/resolve/main/config.yaml.
Access to model pyannote/speaker-diarization-3.1 is restricted. You must have access to it and be authenticated to access it. Please log in.

In [16]:
# ============================================================
# CELL 29A: CHECK HUGGING FACE AUTHENTICATION
# Purpose:
# Verify that the Hugging Face token is available in Colab.
#
# IMPORTANT:
# The token value itself is never printed.
# ============================================================

import os

HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    print("Hugging Face token found.")
    print("Token length:", len(HF_TOKEN))
else:
    print("Hugging Face token NOT found.")

Hugging Face token NOT found.


In [17]:
# ============================================================
# CELL 29B: LOAD HUGGING FACE TOKEN FROM COLAB SECRET
# Purpose:
# Load the Hugging Face authentication token securely from
# Google Colab Secrets.
#
# IMPORTANT:
# The token is NOT printed or displayed.
# ============================================================

from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Hugging Face token loaded successfully.")
except Exception as e:
    HF_TOKEN = None
    print("Could not load HF_TOKEN from Colab Secrets.")
    print("Error:", type(e).__name__)

Could not load HF_TOKEN from Colab Secrets.
Error: SecretNotFoundError


In [18]:
# ============================================================
# CELL 29C: VERIFY HUGGING FACE TOKEN
# Purpose:
# Confirm that the HF_TOKEN secret is now available.
#
# IMPORTANT:
# The actual token value is never displayed.
# ============================================================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("Hugging Face token loaded successfully.")
    print("Token is available for authenticated model downloads.")
else:
    print("Hugging Face token is empty.")

Hugging Face token loaded successfully.
Token is available for authenticated model downloads.


In [20]:
# ============================================================
# CELL 29D: LOAD PYANNOTE SPEAKER DIARIZATION PIPELINE
# Purpose:
# Load the pretrained Pyannote speaker diarization model
# using the authenticated Hugging Face token.
#
# Expected Output:
# A ready-to-use pipeline that identifies:
#     Speaker 1 → when they spoke
#     Speaker 2 → when they spoke
#     ...
# ============================================================

from pyannote.audio import Pipeline
import torch

# Load the pretrained diarization pipeline
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
)

# Use GPU when available
if torch.cuda.is_available():
    diarization_pipeline.to(torch.device("cuda"))

print("Pyannote speaker diarization pipeline loaded successfully.")
print("Device:", DEVICE)

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

Pyannote speaker diarization pipeline loaded successfully.
Device: cuda


In [21]:
# ============================================================
# CELL 30: RUN SPEAKER DIARIZATION
# Purpose:
# Identify speaker segments in the meeting recording.
#
# Output:
# Speaker labels with start and end timestamps.
#
# Example:
# SPEAKER_00 → 00:01:20 → 00:01:35
# SPEAKER_01 → 00:01:35 → 00:01:55
# ============================================================

# Run speaker diarization
diarization = diarization_pipeline(str(INPUT_AUDIO))

print("Speaker diarization completed successfully.")

/usr/local/lib/python3.13/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


Speaker diarization completed successfully.


In [22]:
# ============================================================
# CELL 31: INSPECT SPEAKER DIARIZATION RESULT
# Purpose:
# Examine the speakers and their speaking intervals detected
# by Pyannote.
#
# Expected Output:
#   - Number of detected speakers
#   - Number of speaker segments
#   - First few speaker segments with timestamps
# ============================================================

# Collect diarization segments
diarization_segments = []

for turn, _, speaker in diarization.itertracks(yield_label=True):
    diarization_segments.append({
        "speaker": speaker,
        "start": turn.start,
        "end": turn.end,
        "duration": turn.end - turn.start
    })

# Unique speakers
speakers = sorted(
    set(segment["speaker"] for segment in diarization_segments)
)

print("Number of speaker segments:", len(diarization_segments))
print("Number of detected speakers:", len(speakers))
print("Speakers:", speakers)

print("\nFirst 10 speaker segments:")
for i, segment in enumerate(diarization_segments[:10], start=1):
    print(
        f"{i:02d}. "
        f"{segment['speaker']} | "
        f"{format_timestamp(segment['start'])} → "
        f"{format_timestamp(segment['end'])} | "
        f"{segment['duration']:.2f} sec"
    )

AttributeError: 'DiarizeOutput' object has no attribute 'itertracks'

In [23]:
# ============================================================
# CELL 31A: INSPECT PYANNOTE 4.x DIARIZATION OUTPUT
# Purpose:
# Check the structure of the DiarizeOutput returned by
# Pyannote Audio 4.x.
#
# We will use this information to correctly extract the
# speaker segments without changing the environment.
# ============================================================

print("Output type:")
print(type(diarization))

print("\nAvailable attributes:")
print([attr for attr in dir(diarization) if not attr.startswith("_")])

Output type:
<class 'pyannote.audio.pipelines.speaker_diarization.DiarizeOutput'>

Available attributes:
['exclusive_speaker_diarization', 'serialize', 'speaker_diarization', 'speaker_embeddings']


In [24]:
# ============================================================
# CELL 31B: EXTRACT SPEAKER SEGMENTS
# Purpose:
# Extract speaker labels and their start/end timestamps from
# the Pyannote 4.x DiarizeOutput object.
#
# Expected Output:
# Number of speaker segments
# Number of detected speakers
# First 10 speaker segments
# ============================================================

# Get the actual speaker annotation from Pyannote 4.x
speaker_annotation = diarization.speaker_diarization

# Extract speaker segments
diarization_segments = []

for turn, _, speaker in speaker_annotation.itertracks(yield_label=True):
    diarization_segments.append({
        "speaker": speaker,
        "start": turn.start,
        "end": turn.end,
        "duration": turn.end - turn.start
    })

# Identify unique speakers
speakers = sorted(
    set(segment["speaker"] for segment in diarization_segments)
)

print("Number of speaker segments:", len(diarization_segments))
print("Number of detected speakers:", len(speakers))
print("Speakers:", speakers)

print("\nFirst 10 speaker segments:")

for i, segment in enumerate(diarization_segments[:10], start=1):
    print(
        f"{i:02d}. "
        f"{segment['speaker']} | "
        f"{format_timestamp(segment['start'])} → "
        f"{format_timestamp(segment['end'])} | "
        f"{segment['duration']:.2f} sec"
    )

Number of speaker segments: 264
Number of detected speakers: 4
Speakers: ['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02', 'SPEAKER_03']

First 10 speaker segments:
01. SPEAKER_02 | 00:00:10.90 → 00:00:14.81 | 3.91 sec
02. SPEAKER_02 | 00:00:14.88 → 00:00:14.97 | 0.08 sec
03. SPEAKER_01 | 00:00:14.97 → 00:00:15.13 | 0.17 sec
04. SPEAKER_01 | 00:00:17.87 → 00:00:18.37 | 0.51 sec
05. SPEAKER_02 | 00:00:18.81 → 00:00:20.35 | 1.54 sec
06. SPEAKER_02 | 00:00:22.32 → 00:00:23.74 | 1.42 sec
07. SPEAKER_03 | 00:00:25.12 → 00:00:26.54 | 1.42 sec
08. SPEAKER_03 | 00:00:29.07 → 00:00:32.33 | 3.26 sec
09. SPEAKER_02 | 00:00:29.80 → 00:00:31.49 | 1.69 sec
10. SPEAKER_02 | 00:00:34.22 → 00:00:34.61 | 0.39 sec


In [25]:
# ============================================================
# CELL 32: SAVE SPEAKER DIARIZATION RESULTS
# Purpose:
# Save the Pyannote speaker diarization output as JSON so
# that it can be reused by the downstream alignment stage
# without rerunning the diarization model.
#
# Output:
# data/transcripts/ES2004a_pyannote_diarization.json
# ============================================================

import json

DIARIZATION_FILE = (
    DATA_DIR
    / "transcripts"
    / "ES2004a_pyannote_diarization.json"
)

# Make sure the transcript directory exists
DIARIZATION_FILE.parent.mkdir(parents=True, exist_ok=True)

# Prepare output
diarization_output = {
    "meeting_id": "ES2004a",
    "model": "pyannote/speaker-diarization-3.1",
    "pyannote_version": "4.0.7",
    "num_speakers": len(speakers),
    "speakers": speakers,
    "num_segments": len(diarization_segments),
    "segments": diarization_segments
}

# Save as JSON
with open(DIARIZATION_FILE, "w", encoding="utf-8") as f:
    json.dump(diarization_output, f, indent=2)

print("Diarization results saved successfully.")
print("File:", DIARIZATION_FILE)
print("Segments saved:", len(diarization_segments))
print("Speakers:", speakers)

Diarization results saved successfully.
File: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_pyannote_diarization.json
Segments saved: 264
Speakers: ['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02', 'SPEAKER_03']


In [26]:
# ============================================================
# CELL 33: INSPECT WHISPERX WORD-LEVEL TRANSCRIPT
# Purpose:
# Verify the structure of the WhisperX aligned transcript
# before performing speaker-word alignment.
#
# We will inspect:
#   - Number of aligned segments
#   - Number of words
#   - First few words with timestamps
# ============================================================

# Check that the aligned result is available
print("Number of aligned segments:", len(aligned_result["segments"]))

# Collect word-level information
word_items = []

for segment in aligned_result["segments"]:
    for word in segment.get("words", []):
        if "start" in word and "end" in word:
            word_items.append(word)

print("Number of words with timestamps:", len(word_items))

print("\nFirst 15 words:")

for i, word in enumerate(word_items[:15], start=1):
    print(
        f"{i:02d}. "
        f"{word.get('word', '')} | "
        f"{format_timestamp(word['start'])} → "
        f"{format_timestamp(word['end'])}"
    )

Number of aligned segments: 236
Number of words with timestamps: 2118

First 15 words:
01. Are | 00:00:11.00 → 00:00:11.16
02. we, | 00:00:11.18 → 00:00:11.38
03. we're | 00:00:12.18 → 00:00:12.32
04. not | 00:00:12.36 → 00:00:12.46
05. like | 00:00:12.48 → 00:00:12.62
06. the | 00:00:12.64 → 00:00:12.70
07. dim | 00:00:12.72 → 00:00:12.92
08. lights, | 00:00:12.94 → 00:00:13.18
09. so | 00:00:13.20 → 00:00:13.34
10. we | 00:00:13.36 → 00:00:13.52
11. can | 00:00:13.58 → 00:00:13.72
12. see | 00:00:13.74 → 00:00:13.84
13. that | 00:00:13.86 → 00:00:14.02
14. a | 00:00:14.08 → 00:00:14.12
15. bit | 00:00:14.14 → 00:00:14.26


In [27]:
# ============================================================
# CELL 34: SPEAKER–WORD ALIGNMENT
# Purpose:
# Assign a Pyannote speaker label to each WhisperX word
# using maximum temporal overlap.
#
# Input:
#   1. WhisperX word-level timestamps
#   2. Pyannote speaker-level timestamps
#
# Output:
#   Each word will contain:
#       - word
#       - start
#       - end
#       - speaker
#
# This creates the foundation for the
# speaker-attributed meeting transcript.
# ============================================================

def calculate_overlap(word_start, word_end, speaker_start, speaker_end):
    """
    Calculate temporal overlap between a word and a
    speaker segment.
    """
    overlap_start = max(word_start, speaker_start)
    overlap_end = min(word_end, speaker_end)

    return max(0.0, overlap_end - overlap_start)


# Create a flat list of speaker segments
speaker_segments = [
    {
        "speaker": segment["speaker"],
        "start": segment["start"],
        "end": segment["end"]
    }
    for segment in diarization_segments
]


# Perform speaker-word alignment
speaker_word_items = []

for word in word_items:

    word_start = word["start"]
    word_end = word["end"]

    best_speaker = None
    best_overlap = 0.0

    for speaker_segment in speaker_segments:

        overlap = calculate_overlap(
            word_start,
            word_end,
            speaker_segment["start"],
            speaker_segment["end"]
        )

        if overlap > best_overlap:
            best_overlap = overlap
            best_speaker = speaker_segment["speaker"]

    speaker_word_items.append({
        "word": word.get("word", "").strip(),
        "start": word_start,
        "end": word_end,
        "speaker": best_speaker,
        "overlap": best_overlap
    })


print("Speaker-word alignment completed.")
print("Total words:", len(speaker_word_items))

print("\nFirst 20 aligned words:")

for i, item in enumerate(speaker_word_items[:20], start=1):
    print(
        f"{i:02d}. "
        f"{item['speaker']} | "
        f"{item['word']} | "
        f"{format_timestamp(item['start'])} → "
        f"{format_timestamp(item['end'])} | "
        f"overlap={item['overlap']:.2f}s"
    )

Speaker-word alignment completed.
Total words: 2118

First 20 aligned words:
01. SPEAKER_02 | Are | 00:00:11.00 → 00:00:11.16 | overlap=0.16s
02. SPEAKER_02 | we, | 00:00:11.18 → 00:00:11.38 | overlap=0.20s
03. SPEAKER_02 | we're | 00:00:12.18 → 00:00:12.32 | overlap=0.14s
04. SPEAKER_02 | not | 00:00:12.36 → 00:00:12.46 | overlap=0.10s
05. SPEAKER_02 | like | 00:00:12.48 → 00:00:12.62 | overlap=0.14s
06. SPEAKER_02 | the | 00:00:12.64 → 00:00:12.70 | overlap=0.06s
07. SPEAKER_02 | dim | 00:00:12.72 → 00:00:12.92 | overlap=0.20s
08. SPEAKER_02 | lights, | 00:00:12.94 → 00:00:13.18 | overlap=0.24s
09. SPEAKER_02 | so | 00:00:13.20 → 00:00:13.34 | overlap=0.14s
10. SPEAKER_02 | we | 00:00:13.36 → 00:00:13.52 | overlap=0.16s
11. SPEAKER_02 | can | 00:00:13.58 → 00:00:13.72 | overlap=0.14s
12. SPEAKER_02 | see | 00:00:13.74 → 00:00:13.84 | overlap=0.10s
13. SPEAKER_02 | that | 00:00:13.86 → 00:00:14.02 | overlap=0.16s
14. SPEAKER_02 | a | 00:00:14.08 → 00:00:14.12 | overlap=0.04s
15. SPEAK

In [29]:
# ============================================================
# CELL 35A: IMPROVED SPEAKER UTTERANCE GROUPING
# Purpose:
# Group words into natural speaker utterances using:
#
#   1. Speaker change
#   2. Silence/gap between consecutive words
#
# This prevents very long utterances when the same speaker
# speaks again after a noticeable pause.
#
# Gap threshold:
#   1.0 second
#
# Output:
#   Speaker-attributed utterances with timestamps.
# ============================================================

GAP_THRESHOLD = 1.0  # seconds

speaker_utterances = []

current_speaker = None
current_words = []
current_start = None
current_end = None


def add_current_utterance():
    """Save the current utterance if it contains words."""

    if current_words and current_speaker is not None:
        speaker_utterances.append({
            "speaker": current_speaker,
            "start": current_start,
            "end": current_end,
            "duration": current_end - current_start,
            "text": " ".join(current_words)
        })


for item in speaker_word_items:

    speaker = item["speaker"]

    # Ignore words for which no speaker was assigned
    if speaker is None:
        continue

    word_start = item["start"]
    word_end = item["end"]

    # First word
    if current_speaker is None:

        current_speaker = speaker
        current_words = [item["word"]]
        current_start = word_start
        current_end = word_end

        continue

    # Time gap from previous word
    gap = word_start - current_end

    # Start a new utterance when:
    #   1. Speaker changes
    #   OR
    #   2. There is a significant silence
    if speaker != current_speaker or gap > GAP_THRESHOLD:

        add_current_utterance()

        current_speaker = speaker
        current_words = [item["word"]]
        current_start = word_start
        current_end = word_end

    else:

        # Continue current utterance
        current_words.append(item["word"])
        current_end = word_end


# Save final utterance
add_current_utterance()


print("Improved speaker utterance grouping completed.")
print("Gap threshold:", GAP_THRESHOLD, "seconds")
print("Total speaker utterances:", len(speaker_utterances))

print("\nFirst 15 speaker utterances:")

for i, utterance in enumerate(speaker_utterances[:15], start=1):

    print(
        f"\n{i:02d}. {utterance['speaker']}"
        f" | {format_timestamp(utterance['start'])}"
        f" → {format_timestamp(utterance['end'])}"
        f" | {utterance['duration']:.2f}s"
    )

    print("    ", utterance["text"])

Improved speaker utterance grouping completed.
Gap threshold: 1.0 seconds
Total speaker utterances: 148

First 15 speaker utterances:

01. SPEAKER_02 | 00:00:11.00 → 00:00:14.52 | 3.52s
     Are we, we're not like the dim lights, so we can see that a bit better.

02. SPEAKER_01 | 00:00:17.94 → 00:00:18.16 | 0.22s
     Yeah.

03. SPEAKER_02 | 00:00:18.94 → 00:00:20.95 | 2.00s
     Okay, that's fine.

04. SPEAKER_02 | 00:00:22.41 → 00:00:23.71 | 1.30s
     Am I supposed to be standing up there?

05. SPEAKER_03 | 00:00:25.13 → 00:00:26.51 | 1.38s
     So we've got both of these clipped on.

06. SPEAKER_03 | 00:00:29.09 → 00:00:32.07 | 2.98s
     Is she gonna answer me? Yeah, I've got both of them.

07. SPEAKER_03 | 00:00:50.60 → 00:00:51.35 | 0.74s
     I'm just gonna

08. SPEAKER_02 | 00:01:20.57 → 00:01:23.02 | 2.44s
     Okay, hello everybody.

09. SPEAKER_02 | 00:01:24.66 → 00:01:33.13 | 8.47s
     I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Oka

In [30]:
# ============================================================
# CELL 36: SAVE SPEAKER-ATTRIBUTED TRANSCRIPT
# Purpose:
# Save the aligned and grouped transcript for downstream
# MoM generation, evidence retrieval, and verification.
#
# Each segment contains:
#   - Speaker
#   - Start time
#   - End time
#   - Duration
#   - Transcript text
#
# Output:
# data/transcripts/ES2004a_speaker_transcript.json
# ============================================================

SPEAKER_TRANSCRIPT_FILE = (
    DATA_DIR
    / "transcripts"
    / "ES2004a_speaker_transcript.json"
)

speaker_transcript_output = {
    "meeting_id": "ES2004a",
    "source": "WhisperX + Pyannote speaker-word alignment",
    "gap_threshold_seconds": GAP_THRESHOLD,
    "num_speakers": len(speakers),
    "speakers": speakers,
    "num_utterances": len(speaker_utterances),
    "utterances": speaker_utterances
}

with open(SPEAKER_TRANSCRIPT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        speaker_transcript_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Speaker-attributed transcript saved successfully.")
print("File:", SPEAKER_TRANSCRIPT_FILE)
print("Speakers:", speakers)
print("Utterances saved:", len(speaker_utterances))

Speaker-attributed transcript saved successfully.
File: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_speaker_transcript.json
Speakers: ['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02', 'SPEAKER_03']
Utterances saved: 148


In [31]:
# ============================================================
# CELL 37: PREPARE TRANSCRIPT FOR MOM GENERATION
# Purpose:
# Convert the speaker-attributed transcript into a clean,
# speaker-aware text format for the MoM generation stage.
#
# The original JSON remains unchanged.
# This formatted text will be used as input to the
# summarization model.
# ============================================================

mom_input_lines = []

for utterance in speaker_utterances:

    start_time = format_timestamp(utterance["start"])
    end_time = format_timestamp(utterance["end"])
    speaker = utterance["speaker"]
    text = utterance["text"]

    mom_input_lines.append(
        f"[{speaker} | {start_time} - {end_time}]\n{text}"
    )

mom_input_text = "\n\n".join(mom_input_lines)

print("MoM input preparation completed.")
print("Number of utterances:", len(speaker_utterances))
print("Characters in MoM input:", len(mom_input_text))

print("\nPreview of MoM input:\n")
print(mom_input_text[:5000])

MoM input preparation completed.
Number of utterances: 148
Characters in MoM input: 17367

Preview of MoM input:

[SPEAKER_02 | 00:00:11.00 - 00:00:14.52]
Are we, we're not like the dim lights, so we can see that a bit better.

[SPEAKER_01 | 00:00:17.94 - 00:00:18.16]
Yeah.

[SPEAKER_02 | 00:00:18.94 - 00:00:20.95]
Okay, that's fine.

[SPEAKER_02 | 00:00:22.41 - 00:00:23.71]
Am I supposed to be standing up there?

[SPEAKER_03 | 00:00:25.13 - 00:00:26.51]
So we've got both of these clipped on.

[SPEAKER_03 | 00:00:29.09 - 00:00:32.07]
Is she gonna answer me? Yeah, I've got both of them.

[SPEAKER_03 | 00:00:50.60 - 00:00:51.35]
I'm just gonna

[SPEAKER_02 | 00:01:20.57 - 00:01:23.02]
Okay, hello everybody.

[SPEAKER_02 | 00:01:24.66 - 00:01:33.13]
I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.

[SPEAKER_02 | 00:01:35.79 - 00:01:41.31]
We will do some stuff, get to know each other a bit better, feel more comfortable with each

In [32]:
# ============================================================
# CELL 38: CHECK BART / TRANSFORMERS ENVIRONMENT
# Purpose:
# Verify that the Transformers library is available and check
# its version before loading the BART summarization model.
#
# We will NOT install or change anything in this cell.
# ============================================================

import transformers
import torch

print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Transformers version: 4.57.6
PyTorch version: 2.8.0+cu128
CUDA available: True
GPU: Tesla T4


In [33]:
# ============================================================
# CELL 39: LOAD BART SUMMARIZATION MODEL
# Purpose:
# Load the pretrained BART model for the initial MoM
# generation baseline.
#
# Model:
#   facebook/bart-large-cnn
#
# Device:
#   GPU (Tesla T4) when CUDA is available.
# ============================================================

from transformers import BartForConditionalGeneration, BartTokenizer

BART_MODEL_NAME = "facebook/bart-large-cnn"

# Load tokenizer
bart_tokenizer = BartTokenizer.from_pretrained(
    BART_MODEL_NAME
)

# Load BART model
bart_model = BartForConditionalGeneration.from_pretrained(
    BART_MODEL_NAME
)

# Move model to GPU when available
BART_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

bart_model = bart_model.to(BART_DEVICE)

print("BART model loaded successfully.")
print("Model:", BART_MODEL_NAME)
print("Device:", BART_DEVICE)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART model loaded successfully.
Model: facebook/bart-large-cnn
Device: cuda


In [34]:
# ============================================================
# Cell 40: Check BART Input and Generation Limits
# ============================================================
# Purpose:
# Check the tokenizer/model limits before preparing the
# speaker-attributed meeting transcript for MoM generation.
# ============================================================

print("BART maximum position embeddings:",
      bart_model.config.max_position_embeddings)

print("Tokenizer model maximum length:",
      bart_tokenizer.model_max_length)

print("Maximum generation length:",
      bart_model.generation_config.max_length)

BART maximum position embeddings: 1024
Tokenizer model maximum length: 1000000000000000019884624838656
Maximum generation length: 142


In [35]:
# ============================================================
# Cell 41: Load Speaker-Attributed Transcript
# ============================================================
# Purpose:
# Load the WhisperX + Pyannote speaker-attributed transcript
# that will be used as input for MoM generation.
# ============================================================

import json

TRANSCRIPT_FILE = (
    DATA_DIR / "transcripts" / "ES2004a_speaker_transcript.json"
)

with open(TRANSCRIPT_FILE, "r", encoding="utf-8") as f:
    speaker_transcript = json.load(f)

print("Speaker-attributed transcript loaded successfully.")
print("File:", TRANSCRIPT_FILE)
print("Top-level type:", type(speaker_transcript))

Speaker-attributed transcript loaded successfully.
File: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_speaker_transcript.json
Top-level type: <class 'dict'>


In [36]:
# ============================================================
# Cell 42: Inspect Speaker Transcript Structure
# ============================================================
# Purpose:
# Inspect the JSON structure and identify the field containing
# the speaker-attributed utterances.
# ============================================================

print("Top-level keys:")
print(list(speaker_transcript.keys()))

print("\n--------------------------------------------------")

for key, value in speaker_transcript.items():
    print(f"Key: {key}")
    print(f"Type: {type(value)}")

    if isinstance(value, list):
        print(f"Number of items: {len(value)}")
        if len(value) > 0:
            print("First item:")
            print(value[0])

    elif isinstance(value, dict):
        print("Dictionary keys:")
        print(list(value.keys())[:20])

    print("--------------------------------------------------")

Top-level keys:
['meeting_id', 'source', 'gap_threshold_seconds', 'num_speakers', 'speakers', 'num_utterances', 'utterances']

--------------------------------------------------
Key: meeting_id
Type: <class 'str'>
--------------------------------------------------
Key: source
Type: <class 'str'>
--------------------------------------------------
Key: gap_threshold_seconds
Type: <class 'float'>
--------------------------------------------------
Key: num_speakers
Type: <class 'int'>
--------------------------------------------------
Key: speakers
Type: <class 'list'>
Number of items: 4
First item:
SPEAKER_00
--------------------------------------------------
Key: num_utterances
Type: <class 'int'>
--------------------------------------------------
Key: utterances
Type: <class 'list'>
Number of items: 148
First item:
{'speaker': 'SPEAKER_02', 'start': 10.998, 'end': 14.521, 'duration': 3.5230000000000015, 'text': "Are we, we're not like the dim lights, so we can see that a bit better."}
-

In [37]:
# ============================================================
# Cell 43: Prepare Timestamped Transcript for MoM Generation
# ============================================================
# Purpose:
# Convert the speaker-attributed JSON transcript into a
# timestamped text representation suitable for BART processing.
#
# The original transcript remains unchanged. Speaker labels and
# timestamps are preserved because they will be required later
# for evidence retrieval and verification.
# ============================================================

def format_timestamp(seconds):
    """Convert seconds into HH:MM:SS format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


timestamped_utterances = []

for utt in speaker_transcript["utterances"]:
    start_time = format_timestamp(utt["start"])
    end_time = format_timestamp(utt["end"])

    line = (
        f"[{utt['speaker']} | {start_time} - {end_time}] "
        f"{utt['text']}"
    )

    timestamped_utterances.append(line)


bart_input_text = "\n".join(timestamped_utterances)

print("Timestamped transcript prepared successfully.")
print("Number of utterances:", len(timestamped_utterances))
print("Characters:", len(bart_input_text))

print("\nFirst 10 utterances:\n")
print("\n".join(timestamped_utterances[:10]))

Timestamped transcript prepared successfully.
Number of utterances: 148
Characters: 16332

First 10 utterances:

[SPEAKER_02 | 00:00:10 - 00:00:14] Are we, we're not like the dim lights, so we can see that a bit better.
[SPEAKER_01 | 00:00:17 - 00:00:18] Yeah.
[SPEAKER_02 | 00:00:18 - 00:00:20] Okay, that's fine.
[SPEAKER_02 | 00:00:22 - 00:00:23] Am I supposed to be standing up there?
[SPEAKER_03 | 00:00:25 - 00:00:26] So we've got both of these clipped on.
[SPEAKER_03 | 00:00:29 - 00:00:32] Is she gonna answer me? Yeah, I've got both of them.
[SPEAKER_03 | 00:00:50 - 00:00:51] I'm just gonna
[SPEAKER_02 | 00:01:20 - 00:01:23] Okay, hello everybody.
[SPEAKER_02 | 00:01:24 - 00:01:33] I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.
[SPEAKER_02 | 00:01:35 - 00:01:41] We will do some stuff, get to know each other a bit better, feel more comfortable with each other.


In [38]:
# ============================================================
# Cell 44: Create BART-Compatible Transcript Chunks
# ============================================================
# Purpose:
# Split the timestamped meeting transcript into chunks that
# fit safely within BART's 1024-token input limitation.
#
# The split is performed at utterance boundaries so that an
# individual speaker utterance is not unnecessarily divided.
# ============================================================

MAX_BART_INPUT_TOKENS = 900

bart_chunks = []
current_chunk = []
current_tokens = 0

for utterance in timestamped_utterances:

    # Count tokens for the individual utterance
    utterance_tokens = len(
        bart_tokenizer.encode(
            utterance,
            add_special_tokens=False
        )
    )

    # If adding this utterance exceeds the safe limit,
    # save the current chunk and start a new one.
    if current_chunk and current_tokens + utterance_tokens > MAX_BART_INPUT_TOKENS:

        bart_chunks.append("\n".join(current_chunk))

        current_chunk = []
        current_tokens = 0

    current_chunk.append(utterance)
    current_tokens += utterance_tokens

# Add the final chunk
if current_chunk:
    bart_chunks.append("\n".join(current_chunk))


print("BART-compatible chunks created successfully.")
print("Number of chunks:", len(bart_chunks))

print("\nToken count of each chunk:")

for i, chunk in enumerate(bart_chunks, start=1):
    token_count = len(
        bart_tokenizer.encode(
            chunk,
            add_special_tokens=True
        )
    )

    print(f"Chunk {i}: {token_count} tokens")

BART-compatible chunks created successfully.
Number of chunks: 7

Token count of each chunk:
Chunk 1: 907 tokens
Chunk 2: 921 tokens
Chunk 3: 917 tokens
Chunk 4: 924 tokens
Chunk 5: 916 tokens
Chunk 6: 898 tokens
Chunk 7: 281 tokens


In [39]:
# ============================================================
# Cell 45: Generate BART Summaries for Transcript Chunks
# ============================================================
# Purpose:
# Generate an intermediate summary for each transcript chunk
# using the BART model.
#
# These chunk summaries will later be combined to produce the
# final Meeting Minutes of Meeting (MoM).
# ============================================================

chunk_summaries = []

bart_model.eval()

for i, chunk in enumerate(bart_chunks, start=1):

    print(f"Processing chunk {i}/{len(bart_chunks)}...")

    inputs = bart_tokenizer(
        chunk,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    # Move tokenized input to GPU
    inputs = {
        key: value.to(BART_DEVICE)
        for key, value in inputs.items()
    }

    # Generate summary
    with torch.no_grad():
        summary_ids = bart_model.generate(
            **inputs,
            max_length=142,
            min_length=30,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )

    summary = bart_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    chunk_summaries.append(summary)

    print(f"Chunk {i} summary:")
    print(summary)
    print("-" * 70)


print("\nAll chunk summaries generated successfully.")
print("Number of summaries:", len(chunk_summaries))

Processing chunk 1/7...
Chunk 1 summary:
Project manager introduces herself and the team. They will work on a remote control that can be used by dogs and grannies. The team will also do tool training and discuss the project plan.
----------------------------------------------------------------------
Processing chunk 2/7...
Chunk 2 summary:
A selection of some of the most popular questions asked by the audience. The questions ranged from a cat to a crocodile to a T-Rex to a dog.
----------------------------------------------------------------------
Processing chunk 3/7...
Chunk 3 summary:
CNN.com's John Defterios takes on the challenge to create an animated GIF of a fictional bird of prey. He tries to think on the spot of what type of bird he wants to depict. The result is a picture of an eagle, a seagull, a big cat, a vampire bat and an eagle.
----------------------------------------------------------------------
Processing chunk 4/7...
Chunk 4 summary:
Market Range International is ai

In [40]:
# ============================================================
# Cell 46: Inspect Transcript Chunk and BART Summary
# ============================================================
# Purpose:
# Compare the original transcript content of one problematic
# chunk with the summary generated by BART.
#
# This helps determine whether the generated content is
# supported by the actual meeting transcript.
# ============================================================

CHUNK_TO_INSPECT = 2

print("=" * 80)
print(f"ORIGINAL TRANSCRIPT CHUNK {CHUNK_TO_INSPECT}")
print("=" * 80)

print(bart_chunks[CHUNK_TO_INSPECT - 1])

print("\n" + "=" * 80)
print(f"BART SUMMARY FOR CHUNK {CHUNK_TO_INSPECT}")
print("=" * 80)

print(chunk_summaries[CHUNK_TO_INSPECT - 1])

ORIGINAL TRANSCRIPT CHUNK 2
[SPEAKER_02 | 00:04:14 - 00:04:17] And I see them as majestic
[SPEAKER_02 | 00:04:20 - 00:04:22] and independent
[SPEAKER_02 | 00:04:26 - 00:04:26] and
[SPEAKER_02 | 00:04:28 - 00:04:29] proud.
[SPEAKER_02 | 00:04:32 - 00:04:35] No. Who would like to go next? Me. Okay.
[SPEAKER_01 | 00:04:37 - 00:04:39] Wish. Cat.
[SPEAKER_01 | 00:04:45 - 00:04:46] Where does this come from?
[SPEAKER_02 | 00:04:47 - 00:04:48] There you
[SPEAKER_01 | 00:04:49 - 00:04:51] go. Thank you.
[SPEAKER_01 | 00:04:54 - 00:04:55] Maybe you can guess what I'm trying to make.
[SPEAKER_03 | 00:05:06 - 00:05:07] A kind of talk.
[SPEAKER_01 | 00:05:09 - 00:05:09] Yeah.
[SPEAKER_01 | 00:05:13 - 00:05:21] It's actually sitting. Sorry. It's sitting. It's not standing. Okay. I see it as one thing. It's very supportive.
[SPEAKER_01 | 00:05:24 - 00:05:31] It's your best friend in you. You can talk to a dog. It can be your best friend. It doesn't discriminate between you based on what you are. Sec

In [41]:
# ============================================================
# Cell 47: Save BART Baseline Summaries
# ============================================================
# Purpose:
# Save the chunk-level BART summaries as the baseline result.
# These results will later be useful for comparison with the
# evidence-grounded MoM generation and verification stages.
# ============================================================

BART_BASELINE_FILE = (
    DATA_DIR / "transcripts" / "ES2004a_bart_baseline.json"
)

bart_baseline = {
    "meeting_id": speaker_transcript["meeting_id"],
    "model": BART_MODEL_NAME,
    "num_chunks": len(bart_chunks),
    "summaries": [
        {
            "chunk_id": i + 1,
            "summary": summary
        }
        for i, summary in enumerate(chunk_summaries)
    ]
}

with open(BART_BASELINE_FILE, "w", encoding="utf-8") as f:
    json.dump(bart_baseline, f, indent=2, ensure_ascii=False)

print("BART baseline saved successfully.")
print("File:", BART_BASELINE_FILE)

BART baseline saved successfully.
File: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_bart_baseline.json


In [42]:
# ============================================================
# Cell 48: Prepare Combined BART Chunk Summaries
# ============================================================
# Purpose:
# Combine the seven intermediate BART summaries and check
# whether they can safely fit within BART's input limit.
# ============================================================

combined_bart_summary = "\n\n".join(
    f"[Chunk {i}] {summary}"
    for i, summary in enumerate(chunk_summaries, start=1)
)

combined_summary_tokens = len(
    bart_tokenizer.encode(
        combined_bart_summary,
        add_special_tokens=True
    )
)

print("Combined BART summaries prepared successfully.")
print("Number of chunk summaries:", len(chunk_summaries))
print("Combined token count:", combined_summary_tokens)
print("\nCombined summaries:\n")
print(combined_bart_summary)

Combined BART summaries prepared successfully.
Number of chunk summaries: 7
Combined token count: 365

Combined summaries:

[Chunk 1] Project manager introduces herself and the team. They will work on a remote control that can be used by dogs and grannies. The team will also do tool training and discuss the project plan.

[Chunk 2] A selection of some of the most popular questions asked by the audience. The questions ranged from a cat to a crocodile to a T-Rex to a dog.

[Chunk 3] CNN.com's John Defterios takes on the challenge to create an animated GIF of a fictional bird of prey. He tries to think on the spot of what type of bird he wants to depict. The result is a picture of an eagle, a seagull, a big cat, a vampire bat and an eagle.

[Chunk 4] Market Range International is aiming to be accessible and usable by all age groups. Half of the selling price is taken out by building it. It's not focusing on business market, any particular thing.

[Chunk 5] Six of the world's most popular 

In [43]:
# ============================================================
# Cell 49: Generate Candidate Meeting Minutes
# ============================================================
# Purpose:
# Generate a structured candidate Meeting Minutes of Meeting
# (MoM) from the intermediate BART chunk summaries.
#
# IMPORTANT:
# This is a CANDIDATE MoM. Statements will be checked against
# the original timestamped transcript during the evidence
# verification stage.
# ============================================================

mom_prompt = f"""
Summarize the following meeting information as concise
meeting minutes.

Organize the output into these sections:

1. Meeting Overview
2. Discussion Points
3. Decisions
4. Action Items

Use only information present in the provided text.
Do not invent names, decisions, responsibilities, deadlines,
or other details.

Meeting information:

{combined_bart_summary}
"""

inputs = bart_tokenizer(
    mom_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

inputs = {
    key: value.to(BART_DEVICE)
    for key, value in inputs.items()
}

with torch.no_grad():
    mom_ids = bart_model.generate(
        **inputs,
        max_length=250,
        min_length=80,
        num_beams=4,
        length_penalty=1.5,
        early_stopping=True
    )

candidate_mom = bart_tokenizer.decode(
    mom_ids[0],
    skip_special_tokens=True
)

print("=" * 80)
print("CANDIDATE MEETING MINUTES")
print("=" * 80)
print(candidate_mom)

CANDIDATE MEETING MINUTES
Meeting minutes. Summarize the following meeting information as concisely as possible. Use only information present in the provided text. Do not invent names, decisions, responsibilities, deadlines, or other details. The team will work on a remote control that can be used by dogs and grannies. This week's show focuses on remote controls and how they can be made easier to use.


In [4]:
# ============================================================
# Cell 50A: Reload Speaker-Attributed Transcript
# ============================================================
# Purpose:
# Reload the previously saved speaker-attributed transcript
# into the current Colab runtime.
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/MTechIndProj/MoM_Project")
DATA_DIR = PROJECT_DIR / "data"

TRANSCRIPT_FILE = (
    DATA_DIR / "transcripts" / "ES2004a_speaker_transcript.json"
)

with open(TRANSCRIPT_FILE, "r", encoding="utf-8") as f:
    speaker_transcript = json.load(f)

print("Speaker-attributed transcript reloaded successfully.")
print("Meeting ID:", speaker_transcript["meeting_id"])
print("Number of utterances:", speaker_transcript["num_utterances"])
print("Number of speakers:", speaker_transcript["num_speakers"])

Speaker-attributed transcript reloaded successfully.
Meeting ID: ES2004a
Number of utterances: 148
Number of speakers: 4


In [5]:
# ============================================================
# Cell 50: Create Candidate MoM Statements
# ============================================================
# Purpose:
# Identify meaningful speaker utterances that can act as
# candidate Meeting Minutes statements.
#
# These candidates retain their original speaker and timestamps.
# They will later be processed by BGE + FAISS evidence retrieval
# and Multi-Attribute Evidence Consistency verification.
#
# NOTE:
# This is a conservative candidate-generation stage.
# We are not modifying or inventing transcript content.
# ============================================================

candidate_statements = []

for idx, utt in enumerate(speaker_transcript["utterances"], start=1):

    text = utt["text"].strip()

    # Ignore extremely short conversational acknowledgements
    # that are unlikely to represent useful MoM information.
    if len(text.split()) < 5:
        continue

    candidate = {
        "candidate_id": len(candidate_statements) + 1,
        "source_utterance_id": idx,
        "speaker": utt["speaker"],
        "start": utt["start"],
        "end": utt["end"],
        "text": text
    }

    candidate_statements.append(candidate)


print("Candidate MoM statements created successfully.")
print("Original utterances:", len(speaker_transcript["utterances"]))
print("Candidate statements:", len(candidate_statements))

print("\nFirst 10 candidate statements:\n")

for candidate in candidate_statements[:10]:
    print(
        f"[{candidate['candidate_id']}] "
        f"{candidate['speaker']} | "
        f"{format_timestamp(candidate['start'])} - "
        f"{format_timestamp(candidate['end'])}"
    )
    print(candidate["text"])
    print("-" * 70)

Candidate MoM statements created successfully.
Original utterances: 148
Candidate statements: 101

First 10 candidate statements:



NameError: name 'format_timestamp' is not defined

In [6]:
# ============================================================
# Cell 50B: Display Candidate MoM Statements
# ============================================================
# Purpose:
# Display the first candidate statements with their speaker
# and timestamp information.
# ============================================================

def format_timestamp(seconds):
    """Convert seconds into HH:MM:SS format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


print("First 10 candidate statements:\n")

for candidate in candidate_statements[:10]:
    print(
        f"[{candidate['candidate_id']}] "
        f"{candidate['speaker']} | "
        f"{format_timestamp(candidate['start'])} - "
        f"{format_timestamp(candidate['end'])}"
    )
    print(candidate["text"])
    print("-" * 70)

First 10 candidate statements:

[1] SPEAKER_02 | 00:00:10 - 00:00:14
Are we, we're not like the dim lights, so we can see that a bit better.
----------------------------------------------------------------------
[2] SPEAKER_02 | 00:00:22 - 00:00:23
Am I supposed to be standing up there?
----------------------------------------------------------------------
[3] SPEAKER_03 | 00:00:25 - 00:00:26
So we've got both of these clipped on.
----------------------------------------------------------------------
[4] SPEAKER_03 | 00:00:29 - 00:00:32
Is she gonna answer me? Yeah, I've got both of them.
----------------------------------------------------------------------
[5] SPEAKER_02 | 00:01:24 - 00:01:33
I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.
----------------------------------------------------------------------
[6] SPEAKER_02 | 00:01:35 - 00:01:41
We will do some stuff, get to know each other a bit better, feel more comforta

In [7]:
# ============================================================
# Cell 51: Filter Likely MoM-Relevant Statements
# ============================================================
# Purpose:
# Remove obvious conversational/setup utterances and retain
# statements that are more likely to contain:
#   - project information
#   - discussion points
#   - plans
#   - decisions
#   - actions
#   - responsibilities
#   - deadlines
#
# This is a preliminary candidate-filtering stage.
# The original transcript remains unchanged.
# ============================================================

MOM_KEYWORDS = [
    "project",
    "plan",
    "agenda",
    "develop",
    "design",
    "decision",
    "decide",
    "agree",
    "agreed",
    "need to",
    "should",
    "will",
    "going to",
    "have to",
    "must",
    "responsible",
    "task",
    "action",
    "deadline",
    "by",
    "meeting",
    "team",
    "user",
    "customer",
    "product",
    "requirement",
    "feature",
    "interface",
    "control",
    "design",
    "cost",
    "price",
    "market",
    "prototype",
    "training",
    "idea",
    "problem",
    "solution"
]

filtered_candidates = []

for candidate in candidate_statements:

    text_lower = candidate["text"].lower()

    # Check whether the statement contains at least one
    # meeting/project-related cue.
    is_relevant = any(
        keyword in text_lower
        for keyword in MOM_KEYWORDS
    )

    if is_relevant:
        filtered_candidates.append(candidate)


print("Candidate filtering completed.")
print("Original candidate statements:", len(candidate_statements))
print("Filtered MoM candidates:", len(filtered_candidates))

print("\nFirst 15 filtered candidates:\n")

for candidate in filtered_candidates[:15]:

    print(
        f"[{candidate['candidate_id']}] "
        f"{candidate['speaker']} | "
        f"{format_timestamp(candidate['start'])} - "
        f"{format_timestamp(candidate['end'])}"
    )

    print(candidate["text"])
    print("-" * 70)

Candidate filtering completed.
Original candidate statements: 101
Filtered MoM candidates: 50

First 15 filtered candidates:

[5] SPEAKER_02 | 00:01:24 - 00:01:33
I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.
----------------------------------------------------------------------
[6] SPEAKER_02 | 00:01:35 - 00:01:41
We will do some stuff, get to know each other a bit better, feel more comfortable with each other.
----------------------------------------------------------------------
[7] SPEAKER_02 | 00:01:42 - 00:01:45
Then we'll go do tool training,
----------------------------------------------------------------------
[8] SPEAKER_02 | 00:01:46 - 00:01:49
talk about the project plan, discuss our own ideas and everything.
----------------------------------------------------------------------
[10] SPEAKER_02 | 00:01:57 - 00:02:00
Now, we're developing a remote control, which you probably already know.
-----------------------

In [8]:
# ============================================================
# Cell 52: Inspect Filtered MoM Candidates
# ============================================================
# Purpose:
# Review all currently filtered candidates and identify
# useful meeting information versus conversational noise.
#
# This inspection helps determine the next candidate-selection
# strategy before evidence retrieval is implemented.
# ============================================================

print("=" * 90)
print("FILTERED MoM CANDIDATES")
print("=" * 90)

for candidate in filtered_candidates:

    print(
        f"\n[{candidate['candidate_id']}] "
        f"{candidate['speaker']} | "
        f"{format_timestamp(candidate['start'])} - "
        f"{format_timestamp(candidate['end'])}"
    )

    print(candidate["text"])
    print("-" * 90)

print("\nTotal filtered candidates:", len(filtered_candidates))

FILTERED MoM CANDIDATES

[5] SPEAKER_02 | 00:01:24 - 00:01:33
I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.
------------------------------------------------------------------------------------------

[6] SPEAKER_02 | 00:01:35 - 00:01:41
We will do some stuff, get to know each other a bit better, feel more comfortable with each other.
------------------------------------------------------------------------------------------

[7] SPEAKER_02 | 00:01:42 - 00:01:45
Then we'll go do tool training,
------------------------------------------------------------------------------------------

[8] SPEAKER_02 | 00:01:46 - 00:01:49
talk about the project plan, discuss our own ideas and everything.
------------------------------------------------------------------------------------------

[10] SPEAKER_02 | 00:01:57 - 00:02:00
Now, we're developing a remote control, which you probably already know.
----------------------------------------

In [9]:
# ============================================================
# Cell 53: Save Filtered MoM Candidates
# ============================================================
# Purpose:
# Save the current rule-based candidate set as an intermediate
# result. This allows us to compare it later with the
# dialogue-act-based candidate selection.
# ============================================================

CANDIDATE_FILE = (
    DATA_DIR / "transcripts" / "ES2004a_mom_candidates_rule_based.json"
)

candidate_output = {
    "meeting_id": speaker_transcript["meeting_id"],
    "method": "rule_based_keyword_filter",
    "original_utterances": len(speaker_transcript["utterances"]),
    "candidate_count": len(filtered_candidates),
    "candidates": filtered_candidates
}

with open(CANDIDATE_FILE, "w", encoding="utf-8") as f:
    json.dump(
        candidate_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Rule-based candidate set saved successfully.")
print("File:", CANDIDATE_FILE)
print("Candidates saved:", len(filtered_candidates))

Rule-based candidate set saved successfully.
File: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_mom_candidates_rule_based.json
Candidates saved: 50


In [10]:
# ============================================================
# Cell 54: Check Dialogue Act Classification Environment
# ============================================================
# Purpose:
# Check the current Transformers environment and GPU before
# loading a dialogue-act classification model.
#
# No model or package is installed in this cell.
# ============================================================

import torch
import transformers

print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Transformers version: 5.16.1
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [11]:
# ============================================================
# Cell 55: Define Dialogue Act Categories
# ============================================================
# Purpose:
# Define the dialogue-act categories that will be used to
# identify MoM-relevant utterances.
#
# These categories are intentionally simple and interpretable
# for the first implementation.
# ============================================================

DIALOGUE_ACT_LABELS = [
    "INFORMATION",
    "DISCUSSION",
    "DECISION",
    "ACTION",
    "QUESTION",
    "BACKCHANNEL",
    "OTHER"
]

print("Dialogue-act categories:")
for i, label in enumerate(DIALOGUE_ACT_LABELS, start=1):
    print(f"{i}. {label}")

Dialogue-act categories:
1. INFORMATION
2. DISCUSSION
3. DECISION
4. ACTION
5. QUESTION
6. BACKCHANNEL
7. OTHER


In [12]:
# ============================================================
# Cell 56: Prepare Dialogue Act Test Samples
# ============================================================
# Purpose:
# Create a small set of representative utterances from the
# meeting transcript to test the dialogue-act classification
# approach before processing the complete meeting.
#
# These examples cover different conversational functions.
# ============================================================

dialogue_act_test_samples = [
    {
        "text": "Now, we're developing a remote control, which you probably already know.",
        "expected_type": "INFORMATION"
    },
    {
        "text": "Should we be making notes of this?",
        "expected_type": "QUESTION"
    },
    {
        "text": "We want it to be original, something that people haven't thought of.",
        "expected_type": "DISCUSSION"
    },
    {
        "text": "Right. Yeah, we have to buy one.",
        "expected_type": "ACTION"
    },
    {
        "text": "We will go off and do our individual things.",
        "expected_type": "ACTION"
    },
    {
        "text": "Yeah.",
        "expected_type": "BACKCHANNEL"
    }
]

print("Dialogue-act test samples prepared.")
print("Number of samples:", len(dialogue_act_test_samples))

print("\nSamples:\n")

for i, sample in enumerate(dialogue_act_test_samples, start=1):
    print(f"{i}. {sample['text']}")
    print(f"   Expected category: {sample['expected_type']}")

Dialogue-act test samples prepared.
Number of samples: 6

Samples:

1. Now, we're developing a remote control, which you probably already know.
   Expected category: INFORMATION
2. Should we be making notes of this?
   Expected category: QUESTION
3. We want it to be original, something that people haven't thought of.
   Expected category: DISCUSSION
4. Right. Yeah, we have to buy one.
   Expected category: ACTION
5. We will go off and do our individual things.
   Expected category: ACTION
6. Yeah.
   Expected category: BACKCHANNEL


In [13]:
# ============================================================
# Cell 57: Load Dialogue Act Classification Model
# ============================================================
# Purpose:
# Load a pretrained text-classification model for dialogue-act
# classification.
#
# We first use a general zero-shot classifier so that our
# custom meeting-specific categories can be tested without
# additional model training.
# ============================================================

from transformers import pipeline

DIALOGUE_MODEL_NAME = "facebook/bart-large-mnli"

dialogue_classifier = pipeline(
    "zero-shot-classification",
    model=DIALOGUE_MODEL_NAME,
    device=0 if torch.cuda.is_available() else -1
)

print("Dialogue-act classifier loaded successfully.")
print("Model:", DIALOGUE_MODEL_NAME)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Dialogue-act classifier loaded successfully.
Model: facebook/bart-large-mnli
Device: cuda


In [14]:
# ============================================================
# Cell 58: Test Dialogue Act Classifier
# ============================================================
# Purpose:
# Test the zero-shot dialogue-act classifier on the small
# representative sample prepared in Cell 56.
#
# The classifier compares each utterance against our
# meeting-specific dialogue-act categories.
# ============================================================

for i, sample in enumerate(dialogue_act_test_samples, start=1):

    result = dialogue_classifier(
        sample["text"],
        candidate_labels=DIALOGUE_ACT_LABELS,
        multi_label=False
    )

    predicted_label = result["labels"][0]
    confidence = result["scores"][0]

    print(f"Sample {i}")
    print("Text:", sample["text"])
    print("Expected:", sample["expected_type"])
    print("Predicted:", predicted_label)
    print(f"Confidence: {confidence:.4f}")

    print("\nTop 3 predictions:")

    for label, score in zip(
        result["labels"][:3],
        result["scores"][:3]
    ):
        print(f"  {label}: {score:.4f}")

    print("-" * 70)

Sample 1
Text: Now, we're developing a remote control, which you probably already know.
Expected: INFORMATION
Predicted: ACTION
Confidence: 0.3034

Top 3 predictions:
  ACTION: 0.3034
  DISCUSSION: 0.1340
  INFORMATION: 0.1324
----------------------------------------------------------------------
Sample 2
Text: Should we be making notes of this?
Expected: QUESTION
Predicted: DISCUSSION
Confidence: 0.4998

Top 3 predictions:
  DISCUSSION: 0.4998
  QUESTION: 0.2698
  INFORMATION: 0.0931
----------------------------------------------------------------------
Sample 3
Text: We want it to be original, something that people haven't thought of.
Expected: DISCUSSION
Predicted: OTHER
Confidence: 0.4894

Top 3 predictions:
  OTHER: 0.4894
  DISCUSSION: 0.1433
  QUESTION: 0.1373
----------------------------------------------------------------------
Sample 4
Text: Right. Yeah, we have to buy one.
Expected: ACTION
Predicted: ACTION
Confidence: 0.3932

Top 3 predictions:
  ACTION: 0.3932
  DECISION: 

In [15]:
# ============================================================
# Cell 59: Check Existing Dialogue-Act Data
# ============================================================
# Purpose:
# Check whether AMI dialogue-act annotations are already
# available in the project data directory.
# ============================================================

DIALOGUE_ACT_DIR = DATA_DIR / "dialogue_acts"

print("Dialogue-act directory:")
print(DIALOGUE_ACT_DIR)

if DIALOGUE_ACT_DIR.exists():

    files = list(DIALOGUE_ACT_DIR.rglob("*"))

    print("\nFiles found:", len(files))

    for file in files[:30]:
        if file.is_file():
            print(file)

else:
    print("\nDialogue-act directory does not exist yet.")

Dialogue-act directory:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/dialogue_acts

Files found: 0


In [1]:
# ============================================================
# Cell 60: Prepare Evidence Documents
# ============================================================
# Purpose:
# Convert the speaker-attributed transcript into searchable
# evidence documents.
#
# Each document retains the original speaker and precise
# timestamps so that retrieved evidence can later be linked
# back to the meeting recording.
# ============================================================

evidence_documents = []

for idx, utt in enumerate(
    speaker_transcript["utterances"],
    start=1
):

    evidence_document = {
        "evidence_id": idx,
        "speaker": utt["speaker"],
        "start": utt["start"],
        "end": utt["end"],
        "duration": utt["duration"],
        "text": utt["text"].strip()
    }

    evidence_documents.append(evidence_document)


print("Evidence documents prepared successfully.")
print("Number of evidence documents:", len(evidence_documents))

print("\nFirst 5 evidence documents:\n")

for evidence in evidence_documents[:5]:

    print(
        f"[Evidence {evidence['evidence_id']}] "
        f"{evidence['speaker']} | "
        f"{format_timestamp(evidence['start'])} - "
        f"{format_timestamp(evidence['end'])}"
    )

    print(evidence["text"])
    print("-" * 70)

NameError: name 'speaker_transcript' is not defined

In [4]:
# ============================================================
# Cell 60A: Reload Speaker Transcript
# ============================================================
# Purpose:
# Reload the saved speaker-attributed transcript and define
# the timestamp formatting function required by the next cell.
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

DATA_DIR = PROJECT_DIR / "data"

TRANSCRIPT_FILE = (
    DATA_DIR / "transcripts" / "ES2004a_speaker_transcript.json"
)

with open(TRANSCRIPT_FILE, "r", encoding="utf-8") as f:
    speaker_transcript = json.load(f)


def format_timestamp(seconds):
    """Convert seconds into HH:MM:SS format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)

    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


print("Speaker-attributed transcript reloaded successfully.")
print("Meeting ID:", speaker_transcript["meeting_id"])
print("Utterances:", len(speaker_transcript["utterances"]))
print("Speakers:", speaker_transcript["speakers"])

Speaker-attributed transcript reloaded successfully.
Meeting ID: ES2004a
Utterances: 148
Speakers: ['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02', 'SPEAKER_03']


In [5]:
# ============================================================
# Cell 60: Prepare Evidence Documents
# ============================================================
# Purpose:
# Convert the speaker-attributed transcript into searchable
# evidence documents.
#
# Each document retains the original speaker and precise
# timestamps so that retrieved evidence can later be linked
# back to the meeting recording.
# ============================================================

evidence_documents = []

for idx, utt in enumerate(
    speaker_transcript["utterances"],
    start=1
):

    evidence_document = {
        "evidence_id": idx,
        "speaker": utt["speaker"],
        "start": utt["start"],
        "end": utt["end"],
        "duration": utt["duration"],
        "text": utt["text"].strip()
    }

    evidence_documents.append(evidence_document)


print("Evidence documents prepared successfully.")
print("Number of evidence documents:", len(evidence_documents))

print("\nFirst 5 evidence documents:\n")

for evidence in evidence_documents[:5]:

    print(
        f"[Evidence {evidence['evidence_id']}] "
        f"{evidence['speaker']} | "
        f"{format_timestamp(evidence['start'])} - "
        f"{format_timestamp(evidence['end'])}"
    )

    print(evidence["text"])
    print("-" * 70)

Evidence documents prepared successfully.
Number of evidence documents: 148

First 5 evidence documents:

[Evidence 1] SPEAKER_02 | 00:00:10 - 00:00:14
Are we, we're not like the dim lights, so we can see that a bit better.
----------------------------------------------------------------------
[Evidence 2] SPEAKER_01 | 00:00:17 - 00:00:18
Yeah.
----------------------------------------------------------------------
[Evidence 3] SPEAKER_02 | 00:00:18 - 00:00:20
Okay, that's fine.
----------------------------------------------------------------------
[Evidence 4] SPEAKER_02 | 00:00:22 - 00:00:23
Am I supposed to be standing up there?
----------------------------------------------------------------------
[Evidence 5] SPEAKER_03 | 00:00:25 - 00:00:26
So we've got both of these clipped on.
----------------------------------------------------------------------


In [6]:
# ============================================================
# Cell 61: Check Sentence-Transformers Environment
# ============================================================
# Purpose:
# Check whether the sentence-transformers library is already
# available before loading the BGE embedding model.
#
# No package installation is performed in this cell.
# ============================================================

try:
    import sentence_transformers

    print("sentence-transformers is installed.")
    print("Version:", sentence_transformers.__version__)

except ImportError:
    print("sentence-transformers is NOT installed.")

sentence-transformers is installed.
Version: 5.7.0


In [8]:
# ============================================================
# Cell 62A: Load BGE Embedding Model
# ============================================================
# Purpose:
# Load the BGE-small embedding model for semantic representation
# of the timestamped meeting evidence.
#
# PyTorch is imported explicitly because the Colab runtime may
# have cleared previous imports.
# ============================================================

import torch
from sentence_transformers import SentenceTransformer

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

BGE_DEVICE = (
    "cuda" if torch.cuda.is_available() else "cpu"
)

bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=BGE_DEVICE
)

print("BGE embedding model loaded successfully.")
print("Model:", BGE_MODEL_NAME)
print("Device:", BGE_DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BGE embedding model loaded successfully.
Model: BAAI/bge-small-en-v1.5
Device: cuda
GPU: Tesla T4


In [9]:
# ============================================================
# Cell 63: Generate BGE Evidence Embeddings
# ============================================================
# Purpose:
# Convert each timestamped evidence document into a semantic
# embedding using BGE.
#
# These embeddings will later be stored in a FAISS index for
# fast evidence retrieval.
# ============================================================

evidence_texts = [
    evidence["text"]
    for evidence in evidence_documents
]

print("Generating embeddings...")
print("Number of evidence documents:", len(evidence_texts))

evidence_embeddings = bge_model.encode(
    evidence_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("\nEvidence embeddings generated successfully.")
print("Embedding shape:", evidence_embeddings.shape)
print("Embedding data type:", evidence_embeddings.dtype)

Generating embeddings...
Number of evidence documents: 148


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Evidence embeddings generated successfully.
Embedding shape: (148, 384)
Embedding data type: float32


In [10]:
# ============================================================
# Cell 64: Build FAISS Evidence Index
# ============================================================
# Purpose:
# Build a FAISS vector index over the BGE embeddings.
#
# Because the embeddings are normalized, inner-product
# similarity is equivalent to cosine similarity.
# ============================================================

import faiss
import numpy as np

# Get embedding dimension
embedding_dimension = evidence_embeddings.shape[1]

# Create an Inner Product index
faiss_index = faiss.IndexFlatIP(embedding_dimension)

# Add evidence embeddings to the index
faiss_index.add(evidence_embeddings)

print("FAISS evidence index created successfully.")
print("Index type:", type(faiss_index).__name__)
print("Embedding dimension:", embedding_dimension)
print("Number of indexed documents:", faiss_index.ntotal)

ModuleNotFoundError: No module named 'faiss'

In [11]:
# ============================================================
# Cell 64A: Install FAISS
# ============================================================
# Purpose:
# Install FAISS for efficient semantic similarity search.
#
# We use the CPU version because the evidence collection is
# small and FAISS indexing/search will be extremely lightweight.
# ============================================================

!pip install -q faiss-cpu

print("FAISS installation completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 91.9 MB/s eta 0:00:00
FAISS installation completed.


In [12]:
# ============================================================
# Cell 64: Build FAISS Evidence Index
# ============================================================
# Purpose:
# Build a FAISS vector index over the BGE embeddings.
#
# Because the embeddings are normalized, inner-product
# similarity is equivalent to cosine similarity.
# ============================================================

import faiss
import numpy as np

embedding_dimension = evidence_embeddings.shape[1]

# Create an Inner Product index
faiss_index = faiss.IndexFlatIP(embedding_dimension)

# Add evidence embeddings
faiss_index.add(evidence_embeddings)

print("FAISS evidence index created successfully.")
print("Index type:", type(faiss_index).__name__)
print("Embedding dimension:", embedding_dimension)
print("Number of indexed documents:", faiss_index.ntotal)

FAISS evidence index created successfully.
Index type: IndexFlatIP
Embedding dimension: 384
Number of indexed documents: 148


In [13]:
# ============================================================
# Cell 65: Test Semantic Evidence Retrieval
# ============================================================
# Purpose:
# Test the BGE + FAISS retrieval pipeline using a meeting-
# related query.
#
# The retrieved evidence will retain its speaker and timestamp,
# which will later support evidence-grounded MoM verification.
# ============================================================

query = "What are the requirements for the remote control?"

# Generate query embedding
query_embedding = bge_model.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Retrieve top 5 evidence documents
TOP_K = 5

similarity_scores, retrieved_indices = faiss_index.search(
    query_embedding,
    TOP_K
)

print("=" * 80)
print("EVIDENCE RETRIEVAL QUERY")
print("=" * 80)
print(query)

print("\n" + "=" * 80)
print("TOP RETRIEVED EVIDENCE")
print("=" * 80)

for rank, (score, index) in enumerate(
    zip(similarity_scores[0], retrieved_indices[0]),
    start=1
):

    evidence = evidence_documents[index]

    print(
        f"\nRank {rank} | Similarity: {score:.4f}"
    )

    print(
        f"Evidence ID: {evidence['evidence_id']}"
    )

    print(
        f"Speaker: {evidence['speaker']}"
    )

    print(
        f"Timestamp: "
        f"{format_timestamp(evidence['start'])} - "
        f"{format_timestamp(evidence['end'])}"
    )

    print(
        f"Text: {evidence['text']}"
    )

    print("-" * 80)

EVIDENCE RETRIEVAL QUERY
What are the requirements for the remote control?

TOP RETRIEVED EVIDENCE

Rank 1 | Similarity: 0.8159
Evidence ID: 105
Speaker: SPEAKER_01
Timestamp: 00:11:47 - 00:11:50
Text: remote controls. You want to integrate everything into one.
--------------------------------------------------------------------------------

Rank 2 | Similarity: 0.7897
Evidence ID: 117
Speaker: SPEAKER_03
Timestamp: 00:13:43 - 00:13:46
Text: Can he program his remote controller? Is he basic with that too?
--------------------------------------------------------------------------------

Rank 3 | Similarity: 0.7625
Evidence ID: 107
Speaker: SPEAKER_03
Timestamp: 00:11:53 - 00:12:09
Text: experience has only been given the remote control with the object I buy, not doing any tampering with it and programming, using it to program TV videos and things, but basically on off volume, up and down, channel one to that basic function. I don't think I could go any further with it.
-----------------

In [14]:
# ============================================================
# Cell 66: Retrieve Evidence for a Specific MoM Claim
# ============================================================
# Purpose:
# Test semantic evidence retrieval using a specific candidate
# MoM claim.
#
# This represents the type of claim that will later be checked
# by the Multi-Attribute Evidence Consistency module.
# ============================================================

claim = (
    "The remote control should be user-friendly and usable "
    "by a wide range of users."
)

# Generate embedding for the claim
claim_embedding = bge_model.encode(
    [claim],
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Retrieve top evidence
TOP_K = 5

claim_scores, claim_indices = faiss_index.search(
    claim_embedding,
    TOP_K
)

print("=" * 80)
print("CANDIDATE MoM CLAIM")
print("=" * 80)
print(claim)

print("\n" + "=" * 80)
print("RETRIEVED SUPPORTING EVIDENCE")
print("=" * 80)

for rank, (score, index) in enumerate(
    zip(claim_scores[0], claim_indices[0]),
    start=1
):

    evidence = evidence_documents[index]

    print(
        f"\nRank {rank} | Similarity: {score:.4f}"
    )

    print(
        f"Evidence ID: {evidence['evidence_id']}"
    )

    print(
        f"Speaker: {evidence['speaker']}"
    )

    print(
        f"Timestamp: "
        f"{format_timestamp(evidence['start'])} - "
        f"{format_timestamp(evidence['end'])}"
    )

    print(
        f"Text: {evidence['text']}"
    )

    print("-" * 80)

CANDIDATE MoM CLAIM
The remote control should be user-friendly and usable by a wide range of users.

RETRIEVED SUPPORTING EVIDENCE

Rank 1 | Similarity: 0.7502
Evidence ID: 105
Speaker: SPEAKER_01
Timestamp: 00:11:47 - 00:11:50
Text: remote controls. You want to integrate everything into one.
--------------------------------------------------------------------------------

Rank 2 | Similarity: 0.7395
Evidence ID: 14
Speaker: SPEAKER_02
Timestamp: 00:01:57 - 00:02:00
Text: Now, we're developing a remote control, which you probably already know.
--------------------------------------------------------------------------------

Rank 3 | Similarity: 0.6591
Evidence ID: 113
Speaker: SPEAKER_01
Timestamp: 00:13:14 - 00:13:32
Text: Right. I was thinking on the same lines you, instead of having too many buttons and make it complicated for the user. Maybe have an LCD display or something like that, like a mobile. With menus. And if it's somewhat similar to what you have on mobile phone, people m

In [15]:
# ============================================================
# Cell 67: Create Verification Test Claim
# ============================================================
# Purpose:
# Create a structured candidate MoM claim for testing the
# Multi-Attribute Evidence Consistency framework.
#
# The claim intentionally includes multiple attributes:
# content, speaker, time, and event type.
# ============================================================

verification_claim = {
    "claim_id": 1,

    "text": (
        "Speaker 02 stated that the remote control should "
        "be user-friendly and usable by a wide range of users."
    ),

    "speaker": "SPEAKER_02",

    "event_type": "INFORMATION",

    "start": 122.0,
    "end": 141.0
}


print("Verification claim created successfully.")

print("\nClaim:")
print(verification_claim["text"])

print("\nExpected speaker:")
print(verification_claim["speaker"])

print("\nExpected event type:")
print(verification_claim["event_type"])

print(
    "\nExpected time:",
    format_timestamp(verification_claim["start"]),
    "-",
    format_timestamp(verification_claim["end"])
)

Verification claim created successfully.

Claim:
Speaker 02 stated that the remote control should be user-friendly and usable by a wide range of users.

Expected speaker:
SPEAKER_02

Expected event type:
INFORMATION

Expected time: 00:02:02 - 00:02:21


In [16]:
# ============================================================
# Cell 68: Retrieve Evidence for Verification
# ============================================================
# Purpose:
# Retrieve candidate evidence for the structured MoM claim.
#
# We retrieve the top 10 results so that the verification
# stage has multiple possible evidence sources to evaluate.
# ============================================================

claim_text = verification_claim["text"]

# Generate claim embedding
claim_embedding = bge_model.encode(
    [claim_text],
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Retrieve top 10 evidence documents
VERIFICATION_TOP_K = 10

verification_scores, verification_indices = faiss_index.search(
    claim_embedding,
    VERIFICATION_TOP_K
)

retrieved_evidence = []

for rank, (score, index) in enumerate(
    zip(
        verification_scores[0],
        verification_indices[0]
    ),
    start=1
):

    evidence = evidence_documents[index]

    result = {
        "rank": rank,
        "similarity": float(score),
        "evidence_id": evidence["evidence_id"],
        "speaker": evidence["speaker"],
        "start": evidence["start"],
        "end": evidence["end"],
        "text": evidence["text"]
    }

    retrieved_evidence.append(result)


print("=" * 80)
print("VERIFICATION EVIDENCE")
print("=" * 80)

for evidence in retrieved_evidence:

    print(
        f"\nRank {evidence['rank']} | "
        f"Similarity: {evidence['similarity']:.4f}"
    )

    print(
        f"Evidence ID: {evidence['evidence_id']}"
    )

    print(
        f"Speaker: {evidence['speaker']}"
    )

    print(
        f"Timestamp: "
        f"{format_timestamp(evidence['start'])} - "
        f"{format_timestamp(evidence['end'])}"
    )

    print(
        f"Text: {evidence['text']}"
    )

    print("-" * 80)

VERIFICATION EVIDENCE

Rank 1 | Similarity: 0.7019
Evidence ID: 14
Speaker: SPEAKER_02
Timestamp: 00:01:57 - 00:02:00
Text: Now, we're developing a remote control, which you probably already know.
--------------------------------------------------------------------------------

Rank 2 | Similarity: 0.6807
Evidence ID: 105
Speaker: SPEAKER_01
Timestamp: 00:11:47 - 00:11:50
Text: remote controls. You want to integrate everything into one.
--------------------------------------------------------------------------------

Rank 3 | Similarity: 0.6047
Evidence ID: 102
Speaker: SPEAKER_02
Timestamp: 00:11:22 - 00:11:44
Text: we had three videos, a TV and a sort of amp thing all set up. So we got one of the universal remote controls that you program each of your things into. But that kept losing the signals, so we'd have to reprogram it every now and again. I think it was quite cheapy as well. So that might have had something to do with it. But that was quite good, the fact that you could
-----

In [17]:
# =============================================================================
# Multi-Attribute Evidence Consistency
# =============================================================================
# Purpose:
# Check whether retrieved evidence supports a generated MoM claim across
# multiple attributes:
#   1. Content consistency
#   2. Speaker consistency
#   3. Time consistency
#   4. Event-type consistency
#
# This is the main verification component of the proposed methodology.
# =============================================================================

def check_speaker_consistency(claim, evidence):
    """
    Check whether the speaker in the evidence matches the claimed speaker.
    """
    return claim["speaker"] == evidence["speaker"]


def check_time_consistency(claim, evidence):
    """
    Check whether the evidence timestamp overlaps the claimed time interval.
    """

    evidence_start = evidence["start"]
    evidence_end = evidence["end"]

    claim_start = claim["start"]
    claim_end = claim["end"]

    # Check whether the two time intervals overlap
    overlap = max(
        0.0,
        min(claim_end, evidence_end) -
        max(claim_start, evidence_start)
    )

    return overlap > 0


def check_content_consistency(claim_text, evidence_text):
    """
    Basic content consistency check.

    For this prototype, we use semantic similarity from BGE.
    A later version can use a stronger NLI/entailment model.
    """

    embedding = bge_model.encode(
        [claim_text, evidence_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    similarity = float(embedding[0] @ embedding[1])

    # Initial threshold for the prototype
    CONTENT_THRESHOLD = 0.60

    return similarity >= CONTENT_THRESHOLD, similarity


def check_event_type_consistency(claim, evidence):
    """
    Initial event-type consistency check.

    The current transcript does not yet contain reliable dialogue-act
    annotations, so this prototype uses simple linguistic indicators.

    This is intentionally kept separate from the rejected zero-shot
    BART-MNLI dialogue-act experiment.
    """

    evidence_text = evidence["text"].lower()

    event_type = claim["event_type"]

    if event_type == "ACTION":
        action_words = [
            "will", "need to", "have to", "should",
            "i'll", "we'll", "going to", "do"
        ]
        return any(word in evidence_text for word in action_words)

    elif event_type == "DECISION":
        decision_words = [
            "decided", "agreed", "agree", "final",
            "we will", "let's go with"
        ]
        return any(word in evidence_text for word in decision_words)

    elif event_type == "QUESTION":
        return "?" in evidence_text

    elif event_type == "INFORMATION":
        # For information, absence of explicit question/action indicators
        # is used as a simple prototype heuristic.
        return True

    elif event_type == "DISCUSSION":
        return True

    return False


# ---------------------------------------------------------------------------
# Run verification against all retrieved evidence
# ---------------------------------------------------------------------------

verification_results = []

for evidence in retrieved_evidence:

    # 1. Speaker consistency
    speaker_pass = check_speaker_consistency(
        verification_claim,
        evidence
    )

    # 2. Time consistency
    time_pass = check_time_consistency(
        verification_claim,
        evidence
    )

    # 3. Content consistency
    content_pass, content_similarity = check_content_consistency(
        verification_claim["text"],
        evidence["text"]
    )

    # 4. Event-type consistency
    event_pass = check_event_type_consistency(
        verification_claim,
        evidence
    )

    # Overall consistency
    overall_pass = (
        content_pass
        and speaker_pass
        and time_pass
        and event_pass
    )

    verification_results.append({
        "rank": evidence["rank"],
        "evidence_id": evidence["evidence_id"],
        "similarity": evidence["similarity"],
        "content_similarity": content_similarity,
        "speaker_consistent": speaker_pass,
        "time_consistent": time_pass,
        "event_type_consistent": event_pass,
        "overall_consistent": overall_pass,
        "speaker": evidence["speaker"],
        "start": evidence["start"],
        "end": evidence["end"],
        "text": evidence["text"]
    })


# ---------------------------------------------------------------------------
# Display verification results
# ---------------------------------------------------------------------------

print("=" * 100)
print("MULTI-ATTRIBUTE EVIDENCE CONSISTENCY RESULTS")
print("=" * 100)

for result in verification_results:

    print(
        f"\nRank {result['rank']} | "
        f"Evidence ID: {result['evidence_id']}"
    )

    print(
        f"Content similarity : "
        f"{result['content_similarity']:.4f}"
    )

    print(
        f"Speaker consistency: "
        f"{'PASS' if result['speaker_consistent'] else 'FAIL'}"
    )

    print(
        f"Time consistency   : "
        f"{'PASS' if result['time_consistent'] else 'FAIL'}"
    )

    print(
        f"Event consistency  : "
        f"{'PASS' if result['event_type_consistent'] else 'FAIL'}"
    )

    print(
        f"Overall consistency: "
        f"{'PASS' if result['overall_consistent'] else 'FAIL'}"
    )

    print(f"Evidence: {result['text']}")

print("\n" + "=" * 100)

MULTI-ATTRIBUTE EVIDENCE CONSISTENCY RESULTS

Rank 1 | Evidence ID: 14
Content similarity : 0.7019
Speaker consistency: PASS
Time consistency   : FAIL
Event consistency  : PASS
Overall consistency: FAIL
Evidence: Now, we're developing a remote control, which you probably already know.

Rank 2 | Evidence ID: 105
Content similarity : 0.6807
Speaker consistency: FAIL
Time consistency   : FAIL
Event consistency  : PASS
Overall consistency: FAIL
Evidence: remote controls. You want to integrate everything into one.

Rank 3 | Evidence ID: 102
Content similarity : 0.6047
Speaker consistency: PASS
Time consistency   : FAIL
Event consistency  : PASS
Overall consistency: FAIL
Evidence: we had three videos, a TV and a sort of amp thing all set up. So we got one of the universal remote controls that you program each of your things into. But that kept losing the signals, so we'd have to reprogram it every now and again. I think it was quite cheapy as well. So that might have had something to do with

In [18]:
# =============================================================================
# Verification Test Without Predefined Timestamp
# =============================================================================
# Purpose:
# Test whether the system can retrieve evidence for a MoM claim when the
# generated claim does NOT already contain an exact timestamp.
#
# The timestamp will be obtained from the retrieved evidence.
# =============================================================================

verification_claim_no_time = {
    "claim_id": 2,
    "text": "The remote control should be user-friendly and usable by everyone.",
    "speaker": "SPEAKER_03",
    "event_type": "INFORMATION"
}

print("=" * 80)
print("CLAIM FOR VERIFICATION")
print("=" * 80)

print(f"Claim ID   : {verification_claim_no_time['claim_id']}")
print(f"Claim      : {verification_claim_no_time['text']}")
print(f"Speaker    : {verification_claim_no_time['speaker']}")
print(f"Event Type : {verification_claim_no_time['event_type']}")
print("Timestamp  : Not provided")

CLAIM FOR VERIFICATION
Claim ID   : 2
Claim      : The remote control should be user-friendly and usable by everyone.
Speaker    : SPEAKER_03
Event Type : INFORMATION
Timestamp  : Not provided


In [19]:
# =============================================================================
# Retrieve Evidence for Claim 2
# =============================================================================
# Purpose:
# Retrieve the most semantically relevant transcript segments for the claim.
#
# Important:
# The claim does NOT contain a timestamp. The timestamp will be obtained
# from the retrieved evidence.
# =============================================================================

claim_text = verification_claim_no_time["text"]

# Create embedding for the claim
claim_embedding = bge_model.encode(
    [claim_text],
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Retrieve top 10 candidate evidence segments
VERIFICATION_TOP_K = 10

verification_scores, verification_indices = faiss_index.search(
    claim_embedding,
    VERIFICATION_TOP_K
)

retrieved_evidence_claim2 = []

for rank, (score, index) in enumerate(
    zip(verification_scores[0], verification_indices[0]),
    start=1
):

    evidence = evidence_documents[index]

    result = {
        "rank": rank,
        "similarity": float(score),
        "evidence_id": evidence["evidence_id"],
        "speaker": evidence["speaker"],
        "start": evidence["start"],
        "end": evidence["end"],
        "text": evidence["text"]
    }

    retrieved_evidence_claim2.append(result)


# Display retrieved evidence
print("=" * 80)
print("RETRIEVED EVIDENCE FOR CLAIM 2")
print("=" * 80)

for result in retrieved_evidence_claim2:

    print(
        f"\nRank {result['rank']} | "
        f"Similarity: {result['similarity']:.4f}"
    )

    print(f"Evidence ID: {result['evidence_id']}")
    print(f"Speaker: {result['speaker']}")

    print(
        f"Timestamp: "
        f"{format_timestamp(result['start'])} - "
        f"{format_timestamp(result['end'])}"
    )

    print(f"Text: {result['text']}")

    print("-" * 80)

RETRIEVED EVIDENCE FOR CLAIM 2

Rank 1 | Similarity: 0.7932
Evidence ID: 105
Speaker: SPEAKER_01
Timestamp: 00:11:47 - 00:11:50
Text: remote controls. You want to integrate everything into one.
--------------------------------------------------------------------------------

Rank 2 | Similarity: 0.7661
Evidence ID: 14
Speaker: SPEAKER_02
Timestamp: 00:01:57 - 00:02:00
Text: Now, we're developing a remote control, which you probably already know.
--------------------------------------------------------------------------------

Rank 3 | Similarity: 0.6829
Evidence ID: 95
Speaker: SPEAKER_03
Timestamp: 00:10:46 - 00:10:50
Text: We're not focusing on business market, any particular thing. It's everyone, user-friendly to everyone.
--------------------------------------------------------------------------------

Rank 4 | Similarity: 0.6682
Evidence ID: 107
Speaker: SPEAKER_03
Timestamp: 00:11:53 - 00:12:09
Text: experience has only been given the remote control with the object I buy, not doi

In [20]:
# =============================================================================
# Multi-Attribute Verification for Claim 2
# =============================================================================
# Purpose:
# Verify the generated MoM claim against retrieved evidence using:
#
#   1. Content consistency
#   2. Speaker consistency
#   3. Event-type consistency
#
# Since the original claim has no timestamp, the timestamp is obtained
# directly from the supporting evidence.
# =============================================================================

def verify_claim_against_evidence(claim, evidence):
    """
    Verify one MoM claim against one retrieved evidence segment.
    """

    # -------------------------------------------------------------------------
    # 1. Content consistency
    # -------------------------------------------------------------------------
    content_pass, content_similarity = check_content_consistency(
        claim["text"],
        evidence["text"]
    )

    # -------------------------------------------------------------------------
    # 2. Speaker consistency
    # -------------------------------------------------------------------------
    speaker_pass = check_speaker_consistency(
        claim,
        evidence
    )

    # -------------------------------------------------------------------------
    # 3. Event-type consistency
    # -------------------------------------------------------------------------
    event_pass = check_event_type_consistency(
        claim,
        evidence
    )

    # -------------------------------------------------------------------------
    # Overall verification
    # -------------------------------------------------------------------------
    overall_pass = (
        content_pass
        and speaker_pass
        and event_pass
    )

    return {
        "evidence_id": evidence["evidence_id"],
        "content_similarity": content_similarity,
        "content_consistent": content_pass,
        "speaker_consistent": speaker_pass,
        "event_type_consistent": event_pass,
        "overall_verified": overall_pass,
        "speaker": evidence["speaker"],
        "start": evidence["start"],
        "end": evidence["end"],
        "text": evidence["text"]
    }


# =============================================================================
# Verify all retrieved candidates
# =============================================================================

claim2_verification_results = []

for evidence in retrieved_evidence_claim2:

    result = verify_claim_against_evidence(
        verification_claim_no_time,
        evidence
    )

    # Preserve retrieval information
    result["rank"] = evidence["rank"]
    result["retrieval_similarity"] = evidence["similarity"]

    claim2_verification_results.append(result)


# =============================================================================
# Display results
# =============================================================================

print("=" * 100)
print("CLAIM 2 — MULTI-ATTRIBUTE VERIFICATION")
print("=" * 100)

print(f"\nClaim:")
print(verification_claim_no_time["text"])

print(f"\nClaimed Speaker:")
print(verification_claim_no_time["speaker"])

print("\n" + "-" * 100)

for result in claim2_verification_results:

    print(
        f"\nRank {result['rank']} | "
        f"Evidence ID: {result['evidence_id']}"
    )

    print(
        f"Retrieval similarity : "
        f"{result['retrieval_similarity']:.4f}"
    )

    print(
        f"Content similarity   : "
        f"{result['content_similarity']:.4f}"
    )

    print(
        f"Content consistency  : "
        f"{'PASS' if result['content_consistent'] else 'FAIL'}"
    )

    print(
        f"Speaker consistency  : "
        f"{'PASS' if result['speaker_consistent'] else 'FAIL'}"
    )

    print(
        f"Event consistency    : "
        f"{'PASS' if result['event_type_consistent'] else 'FAIL'}"
    )

    print(
        f"Overall verification : "
        f"{'VERIFIED' if result['overall_verified'] else 'FLAGGED'}"
    )

    print(
        f"Evidence timestamp   : "
        f"{format_timestamp(result['start'])} - "
        f"{format_timestamp(result['end'])}"
    )

    print(f"Evidence text        : {result['text']}")

    print("-" * 100)

CLAIM 2 — MULTI-ATTRIBUTE VERIFICATION

Claim:
The remote control should be user-friendly and usable by everyone.

Claimed Speaker:
SPEAKER_03

----------------------------------------------------------------------------------------------------

Rank 1 | Evidence ID: 105
Retrieval similarity : 0.7932
Content similarity   : 0.7932
Content consistency  : PASS
Speaker consistency  : FAIL
Event consistency    : PASS
Overall verification : FLAGGED
Evidence timestamp   : 00:11:47 - 00:11:50
Evidence text        : remote controls. You want to integrate everything into one.
----------------------------------------------------------------------------------------------------

Rank 2 | Evidence ID: 14
Retrieval similarity : 0.7661
Content similarity   : 0.7662
Content consistency  : PASS
Speaker consistency  : FAIL
Event consistency    : PASS
Overall verification : FLAGGED
Evidence timestamp   : 00:01:57 - 00:02:00
Evidence text        : Now, we're developing a remote control, which you probably 

In [21]:
# =============================================================================
# Check NLI / Entailment Model Availability
# =============================================================================
# Purpose:
# Check whether a Natural Language Inference (NLI) model is already
# available in the current environment before installing anything.
# =============================================================================

import importlib.util

print("=" * 80)
print("NLI ENVIRONMENT CHECK")
print("=" * 80)

print(f"Transformers available : {importlib.util.find_spec('transformers') is not None}")
print(f"PyTorch available      : {importlib.util.find_spec('torch') is not None}")
print(f"CUDA available         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU                     : {torch.cuda.get_device_name(0)}")

print("\nNo new model will be installed in this cell.")
print("Next step will depend on this environment check.")

NLI ENVIRONMENT CHECK
Transformers available : True
PyTorch available      : True
CUDA available         : True
GPU                     : Tesla T4

No new model will be installed in this cell.
Next step will depend on this environment check.


In [22]:
# =============================================================================
# Load NLI / Entailment Model
# =============================================================================
# Purpose:
# Load a Natural Language Inference model to improve the content-consistency
# component of the Multi-Attribute Evidence Consistency module.
#
# Input:
#     MoM claim + retrieved evidence
#
# Output:
#     Entailment / Contradiction / Neutral
#
# This model complements BGE semantic retrieval.
# BGE finds candidate evidence; NLI checks whether the evidence actually
# supports the claim.
# =============================================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

print("=" * 80)
print("LOADING NLI MODEL")
print("=" * 80)

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
)

NLI_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

nli_model = nli_model.to(NLI_DEVICE)
nli_model.eval()

print(f"\nModel  : {NLI_MODEL_NAME}")
print(f"Device : {NLI_DEVICE}")

print("\nNLI model loaded successfully.")

LOADING NLI MODEL


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]


Model  : cross-encoder/nli-deberta-v3-small
Device : cuda

NLI model loaded successfully.


In [23]:
# =============================================================================
# NLI Model Sanity Test
# =============================================================================
# Purpose:
# Test whether the NLI model can distinguish:
#
#   1. Entailment     -> evidence supports the claim
#   2. Neutral        -> evidence is related but does not support the claim
#   3. Contradiction  -> evidence conflicts with the claim
#
# This test is performed before integrating NLI into the full
# Multi-Attribute Evidence Consistency pipeline.
# =============================================================================

def run_nli(claim, evidence):
    """
    Run Natural Language Inference between a claim and evidence.
    """

    inputs = nli_tokenizer(
        claim,
        evidence,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(NLI_DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    predicted_index = int(torch.argmax(probabilities))

    # Inspect model's label mapping
    label = nli_model.config.id2label[predicted_index]

    return label, float(probabilities[predicted_index])


# =============================================================================
# Test cases
# =============================================================================

nli_test_cases = [

    {
        "name": "ENTAILMENT TEST",
        "claim": "The remote control should be user-friendly and usable by everyone.",
        "evidence": "It's everyone, user-friendly to everyone."
    },

    {
        "name": "NEUTRAL TEST",
        "claim": "The remote control should be user-friendly and usable by everyone.",
        "evidence": "The remote control can have a flip top with a bigger screen."
    },

    {
        "name": "CONTRADICTION TEST",
        "claim": "The remote control should be user-friendly and usable by everyone.",
        "evidence": "The remote control is designed only for business users."
    }
]


# =============================================================================
# Run tests
# =============================================================================

print("=" * 90)
print("NLI MODEL SANITY TEST")
print("=" * 90)

for test in nli_test_cases:

    label, confidence = run_nli(
        test["claim"],
        test["evidence"]
    )

    print(f"\n{test['name']}")
    print("-" * 90)

    print(f"Claim    : {test['claim']}")
    print(f"Evidence : {test['evidence']}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.4f}")

print("\n" + "=" * 90)

NLI MODEL SANITY TEST

ENTAILMENT TEST
------------------------------------------------------------------------------------------
Claim    : The remote control should be user-friendly and usable by everyone.
Evidence : It's everyone, user-friendly to everyone.
Prediction: entailment
Confidence: 0.5396

NEUTRAL TEST
------------------------------------------------------------------------------------------
Claim    : The remote control should be user-friendly and usable by everyone.
Evidence : The remote control can have a flip top with a bigger screen.
Prediction: neutral
Confidence: 0.9995

CONTRADICTION TEST
------------------------------------------------------------------------------------------
Claim    : The remote control should be user-friendly and usable by everyone.
Evidence : The remote control is designed only for business users.
Prediction: contradiction
Confidence: 0.9993



In [25]:
# =============================================================================
# Corrected NLI Function
# =============================================================================
# Purpose:
# Perform NLI in the correct direction:
#
#     Premise    = Retrieved transcript evidence
#     Hypothesis = Generated MoM claim
#
# This asks:
# "Does the retrieved evidence support the generated claim?"
# =============================================================================

def run_nli_correct(evidence, claim):
    """
    Run NLI with:
        premise    = evidence
        hypothesis = claim
    """

    inputs = nli_tokenizer(
        evidence,
        claim,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(NLI_DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    predicted_index = int(torch.argmax(probabilities))

    label = nli_model.config.id2label[predicted_index]

    return label, float(probabilities[predicted_index])


print("=" * 80)
print("CORRECTED NLI FUNCTION")
print("=" * 80)
print("Premise    = Retrieved evidence")
print("Hypothesis = MoM claim")
print("Status     = Ready")

CORRECTED NLI FUNCTION
Premise    = Retrieved evidence
Hypothesis = MoM claim
Status     = Ready


In [26]:
# =============================================================================
# Corrected NLI Verification — Claim 2
# =============================================================================
# Purpose:
# Re-evaluate all retrieved evidence using the corrected NLI direction:
#
#     Premise    = Transcript evidence
#     Hypothesis = MoM claim
#
# This determines whether each evidence segment actually supports
# the generated claim.
# =============================================================================

nli_claim2_corrected_results = []

claim_text = verification_claim_no_time["text"]

for evidence in retrieved_evidence_claim2:

    label, confidence = run_nli_correct(
        evidence["text"],
        claim_text
    )

    result = {
        "rank": evidence["rank"],
        "evidence_id": evidence["evidence_id"],
        "retrieval_similarity": evidence["similarity"],
        "nli_label": label,
        "nli_confidence": confidence,
        "speaker": evidence["speaker"],
        "start": evidence["start"],
        "end": evidence["end"],
        "text": evidence["text"]
    }

    nli_claim2_corrected_results.append(result)


# =============================================================================
# Display corrected NLI results
# =============================================================================

print("=" * 100)
print("CORRECTED NLI VERIFICATION — CLAIM 2")
print("=" * 100)

print(f"\nClaim:")
print(claim_text)

print(f"\nClaimed Speaker:")
print(verification_claim_no_time["speaker"])

print("\n" + "-" * 100)

for result in nli_claim2_corrected_results:

    print(
        f"\nRank {result['rank']} | "
        f"Evidence ID: {result['evidence_id']}"
    )

    print(
        f"Retrieval similarity : "
        f"{result['retrieval_similarity']:.4f}"
    )

    print(
        f"NLI prediction       : "
        f"{result['nli_label']}"
    )

    print(
        f"NLI confidence       : "
        f"{result['nli_confidence']:.4f}"
    )

    print(
        f"Speaker              : "
        f"{result['speaker']}"
    )

    print(
        f"Timestamp            : "
        f"{format_timestamp(result['start'])} - "
        f"{format_timestamp(result['end'])}"
    )

    print(
        f"Evidence             : "
        f"{result['text']}"
    )

    print("-" * 100)

CORRECTED NLI VERIFICATION — CLAIM 2

Claim:
The remote control should be user-friendly and usable by everyone.

Claimed Speaker:
SPEAKER_03

----------------------------------------------------------------------------------------------------

Rank 1 | Evidence ID: 105
Retrieval similarity : 0.7932
NLI prediction       : neutral
NLI confidence       : 0.9785
Speaker              : SPEAKER_01
Timestamp            : 00:11:47 - 00:11:50
Evidence             : remote controls. You want to integrate everything into one.
----------------------------------------------------------------------------------------------------

Rank 2 | Evidence ID: 14
Retrieval similarity : 0.7661
NLI prediction       : neutral
NLI confidence       : 0.9969
Speaker              : SPEAKER_02
Timestamp            : 00:01:57 - 00:02:00
Evidence             : Now, we're developing a remote control, which you probably already know.
----------------------------------------------------------------------------------------

In [27]:
# =============================================================================
# Inspect Context Around Supporting Evidence
# =============================================================================
# Purpose:
# Evidence 95 is semantically relevant but NLI classified it as neutral.
#
# Because meeting speech often contains incomplete conversational fragments,
# we inspect the surrounding speaker-attributed transcript context.
#
# This will allow us to build a contextual evidence window for verification.
# =============================================================================

TARGET_EVIDENCE_ID = 95

# Find the position of Evidence 95
target_index = next(
    i for i, item in enumerate(evidence_documents)
    if item["evidence_id"] == TARGET_EVIDENCE_ID
)

# Number of transcript segments to inspect on either side
CONTEXT_WINDOW = 3

start_index = max(
    0,
    target_index - CONTEXT_WINDOW
)

end_index = min(
    len(evidence_documents),
    target_index + CONTEXT_WINDOW + 1
)

context_evidence = evidence_documents[
    start_index:end_index
]

print("=" * 100)
print("CONTEXT AROUND EVIDENCE 95")
print("=" * 100)

for item in context_evidence:

    marker = (
        " <-- TARGET EVIDENCE"
        if item["evidence_id"] == TARGET_EVIDENCE_ID
        else ""
    )

    print(
        f"\nEvidence ID: {item['evidence_id']}{marker}"
    )

    print(
        f"Speaker: {item['speaker']}"
    )

    print(
        f"Timestamp: "
        f"{format_timestamp(item['start'])} - "
        f"{format_timestamp(item['end'])}"
    )

    print(
        f"Text: {item['text']}"
    )

    print("-" * 100)

CONTEXT AROUND EVIDENCE 95

Evidence ID: 92
Speaker: SPEAKER_02
Timestamp: 00:10:31 - 00:10:33
Text: Yes. Yeah,
----------------------------------------------------------------------------------------------------

Evidence ID: 93
Speaker: SPEAKER_02
Timestamp: 00:10:34 - 00:10:35
Text: I presume so.
----------------------------------------------------------------------------------------------------

Evidence ID: 94
Speaker: SPEAKER_03
Timestamp: 00:10:38 - 00:10:44
Text: You've got Market Range International, and you did say earlier it's got to be accessible and usable by all age groups.
----------------------------------------------------------------------------------------------------

Evidence ID: 95 <-- TARGET EVIDENCE
Speaker: SPEAKER_03
Timestamp: 00:10:46 - 00:10:50
Text: We're not focusing on business market, any particular thing. It's everyone, user-friendly to everyone.
----------------------------------------------------------------------------------------------------

Evide

In [28]:
# =============================================================================
# Contextual Evidence Construction and NLI Test
# =============================================================================
# Purpose:
# Combine neighboring transcript segments around the retrieved evidence
# into a single contextual evidence block.
#
# This addresses a common meeting-transcript problem:
# individual ASR segments may be incomplete or depend on surrounding speech.
#
# The contextual block will then be evaluated by NLI.
# =============================================================================

# -------------------------------------------------------------------------
# Select the relevant contextual evidence
# -------------------------------------------------------------------------

context_ids = [94, 95, 96]

selected_context = [
    item
    for item in evidence_documents
    if item["evidence_id"] in context_ids
]

# Sort chronologically
selected_context = sorted(
    selected_context,
    key=lambda x: x["start"]
)


# -------------------------------------------------------------------------
# Combine the evidence into one contextual passage
# -------------------------------------------------------------------------

context_text = " ".join(
    item["text"]
    for item in selected_context
)


# -------------------------------------------------------------------------
# Determine contextual timestamp range
# -------------------------------------------------------------------------

context_start = min(
    item["start"]
    for item in selected_context
)

context_end = max(
    item["end"]
    for item in selected_context
)


# -------------------------------------------------------------------------
# Display contextual evidence
# -------------------------------------------------------------------------

print("=" * 100)
print("CONTEXTUAL EVIDENCE")
print("=" * 100)

print(
    f"\nTimestamp: "
    f"{format_timestamp(context_start)} - "
    f"{format_timestamp(context_end)}"
)

print(
    f"\nSpeakers: "
    f"{', '.join(sorted(set(item['speaker'] for item in selected_context)))}"
)

print("\nContext:")
print(context_text)


# -------------------------------------------------------------------------
# Run corrected NLI
# -------------------------------------------------------------------------

nli_label, nli_confidence = run_nli_correct(
    context_text,
    verification_claim_no_time["text"]
)


# -------------------------------------------------------------------------
# Display NLI result
# -------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CONTEXTUAL NLI RESULT")
print("=" * 100)

print(f"\nClaim:")
print(verification_claim_no_time["text"])

print(f"\nNLI prediction : {nli_label}")
print(f"NLI confidence : {nli_confidence:.4f}")

print(
    f"\nEvidence timestamp: "
    f"{format_timestamp(context_start)} - "
    f"{format_timestamp(context_end)}"
)

print("\n" + "=" * 100)

CONTEXTUAL EVIDENCE

Timestamp: 00:10:38 - 00:10:54

Speakers: SPEAKER_03

Context:
You've got Market Range International, and you did say earlier it's got to be accessible and usable by all age groups. We're not focusing on business market, any particular thing. It's everyone, user-friendly to everyone. Big target group.

CONTEXTUAL NLI RESULT

Claim:
The remote control should be user-friendly and usable by everyone.

NLI prediction : neutral
NLI confidence : 0.9964

Evidence timestamp: 00:10:38 - 00:10:54



In [29]:
# =============================================================================
# Positive NLI Verification Test
# =============================================================================
# Purpose:
# Test the NLI verifier using a claim that is directly supported by the
# retrieved contextual evidence.
#
# This establishes a positive VERIFIED case for the evaluation.
# =============================================================================

positive_claim = {
    "claim_id": 3,
    "text": "The product should be accessible and usable by all age groups.",
    "speaker": "SPEAKER_03",
    "event_type": "INFORMATION"
}

positive_evidence = context_text

print("=" * 100)
print("POSITIVE NLI VERIFICATION TEST")
print("=" * 100)

print("\nClaim:")
print(positive_claim["text"])

print("\nEvidence:")
print(positive_evidence)

print("\nClaimed Speaker:")
print(positive_claim["speaker"])


# -------------------------------------------------------------------------
# Run NLI
# -------------------------------------------------------------------------

positive_label, positive_confidence = run_nli_correct(
    positive_evidence,
    positive_claim["text"]
)


# -------------------------------------------------------------------------
# Display result
# -------------------------------------------------------------------------

print("\n" + "-" * 100)

print(f"NLI prediction : {positive_label}")
print(f"NLI confidence : {positive_confidence:.4f}")

print("\n" + "=" * 100)

POSITIVE NLI VERIFICATION TEST

Claim:
The product should be accessible and usable by all age groups.

Evidence:
You've got Market Range International, and you did say earlier it's got to be accessible and usable by all age groups. We're not focusing on business market, any particular thing. It's everyone, user-friendly to everyone. Big target group.

Claimed Speaker:
SPEAKER_03

----------------------------------------------------------------------------------------------------
NLI prediction : neutral
NLI confidence : 0.9956



In [30]:
# =============================================================================
# Exact Proposition NLI Test
# =============================================================================
# Purpose:
# Test NLI using a claim that closely matches the wording explicitly present
# in the meeting transcript.
#
# This isolates whether the problem is:
#   1. NLI itself, or
#   2. additional information introduced by the generated MoM claim.
# =============================================================================

exact_claim = {
    "claim_id": 4,
    "text": "It should be accessible and usable by all age groups.",
    "speaker": "SPEAKER_03",
    "event_type": "INFORMATION"
}

exact_evidence = evidence_documents[
    next(
        i for i, item in enumerate(evidence_documents)
        if item["evidence_id"] == 94
    )
]["text"]

print("=" * 100)
print("EXACT PROPOSITION NLI TEST")
print("=" * 100)

print("\nClaim:")
print(exact_claim["text"])

print("\nEvidence:")
print(exact_evidence)

print("\nSpeaker:")
print(exact_claim["speaker"])


# -------------------------------------------------------------------------
# Run NLI
# -------------------------------------------------------------------------

exact_label, exact_confidence = run_nli_correct(
    exact_evidence,
    exact_claim["text"]
)


# -------------------------------------------------------------------------
# Display result
# -------------------------------------------------------------------------

print("\n" + "-" * 100)

print(f"NLI prediction : {exact_label}")
print(f"NLI confidence : {exact_confidence:.4f}")

print("\n" + "=" * 100)

EXACT PROPOSITION NLI TEST

Claim:
It should be accessible and usable by all age groups.

Evidence:
You've got Market Range International, and you did say earlier it's got to be accessible and usable by all age groups.

Speaker:
SPEAKER_03

----------------------------------------------------------------------------------------------------
NLI prediction : entailment
NLI confidence : 0.9905



In [31]:
# =============================================================================
# Multi-Attribute Evidence Verification — Version 1
# =============================================================================
# Purpose:
# Combine retrieval, NLI, speaker consistency, and event-type consistency
# into a single verification decision.
#
# Verification principle:
#
#     VERIFIED =
#         NLI entailment
#         AND speaker consistency
#         AND event-type consistency
#
# Otherwise:
#
#     FLAGGED
#
# The timestamp is taken directly from the verified evidence.
# =============================================================================


def verify_evidence(claim, evidence):
    """
    Perform multi-attribute verification for one claim/evidence pair.
    """

    # -------------------------------------------------------------------------
    # 1. NLI-based content verification
    # -------------------------------------------------------------------------

    nli_label, nli_confidence = run_nli_correct(
        evidence["text"],
        claim["text"]
    )

    content_pass = (
        nli_label.lower() == "entailment"
    )

    # -------------------------------------------------------------------------
    # 2. Speaker consistency
    # -------------------------------------------------------------------------

    speaker_pass = check_speaker_consistency(
        claim,
        evidence
    )

    # -------------------------------------------------------------------------
    # 3. Event-type consistency
    # -------------------------------------------------------------------------

    event_pass = check_event_type_consistency(
        claim,
        evidence
    )

    # -------------------------------------------------------------------------
    # 4. Overall verification decision
    # -------------------------------------------------------------------------

    verified = (
        content_pass
        and speaker_pass
        and event_pass
    )

    return {
        "evidence_id": evidence["evidence_id"],
        "retrieval_similarity": evidence.get(
            "similarity",
            evidence.get("retrieval_similarity", 0.0)
        ),
        "nli_label": nli_label,
        "nli_confidence": nli_confidence,
        "content_consistent": content_pass,
        "speaker_consistent": speaker_pass,
        "event_type_consistent": event_pass,
        "verified": verified,
        "speaker": evidence["speaker"],
        "start": evidence["start"],
        "end": evidence["end"],
        "text": evidence["text"]
    }


print("=" * 100)
print("MULTI-ATTRIBUTE EVIDENCE VERIFIER")
print("=" * 100)

print("\nVerification rules:")
print("  Content       → NLI entailment")
print("  Speaker       → Exact speaker match")
print("  Event type    → Consistency check")
print("  Overall       → All checks must PASS")

print("\nVerifier is ready.")

MULTI-ATTRIBUTE EVIDENCE VERIFIER

Verification rules:
  Content       → NLI entailment
  Speaker       → Exact speaker match
  Event type    → Consistency check
  Overall       → All checks must PASS

Verifier is ready.


In [32]:
# =============================================================================
# Complete Multi-Attribute Verification Test
# =============================================================================
# Purpose:
# Test the complete verifier using:
#
#   Case 1 → Directly supported claim
#   Case 2 → Claim containing unsupported additional information
#
# This demonstrates both VERIFIED and FLAGGED outcomes.
# =============================================================================


# =============================================================================
# CASE 1 — Directly Supported Claim
# =============================================================================

claim_verified = {
    "claim_id": 3,
    "text": "It should be accessible and usable by all age groups.",
    "speaker": "SPEAKER_03",
    "event_type": "INFORMATION"
}

evidence_verified = evidence_documents[
    next(
        i for i, item in enumerate(evidence_documents)
        if item["evidence_id"] == 94
    )
]


result_verified = verify_evidence(
    claim_verified,
    evidence_verified
)


# =============================================================================
# CASE 2 — Claim with Unsupported Association
# =============================================================================

claim_flagged = {
    "claim_id": 2,
    "text": "The remote control should be user-friendly and usable by everyone.",
    "speaker": "SPEAKER_03",
    "event_type": "INFORMATION"
}

evidence_flagged = evidence_documents[
    next(
        i for i, item in enumerate(evidence_documents)
        if item["evidence_id"] == 95
    )
]


result_flagged = verify_evidence(
    claim_flagged,
    evidence_flagged
)


# =============================================================================
# Display Results
# =============================================================================

print("=" * 100)
print("COMPLETE MULTI-ATTRIBUTE VERIFICATION TEST")
print("=" * 100)


print("\nCASE 1 — DIRECTLY SUPPORTED CLAIM")
print("-" * 100)

print(f"Claim       : {claim_verified['text']}")
print(f"Evidence ID : {result_verified['evidence_id']}")
print(f"Evidence    : {result_verified['text']}")
print(
    f"Timestamp   : "
    f"{format_timestamp(result_verified['start'])} - "
    f"{format_timestamp(result_verified['end'])}"
)

print(
    f"\nNLI         : "
    f"{result_verified['nli_label']} "
    f"({result_verified['nli_confidence']:.4f})"
)

print(
    f"Speaker     : "
    f"{'PASS' if result_verified['speaker_consistent'] else 'FAIL'}"
)

print(
    f"Event Type  : "
    f"{'PASS' if result_verified['event_type_consistent'] else 'FAIL'}"
)

print(
    f"FINAL       : "
    f"{'VERIFIED' if result_verified['verified'] else 'FLAGGED'}"
)


print("\n\nCASE 2 — CLAIM WITH UNSUPPORTED ASSOCIATION")
print("-" * 100)

print(f"Claim       : {claim_flagged['text']}")
print(f"Evidence ID : {result_flagged['evidence_id']}")
print(f"Evidence    : {result_flagged['text']}")
print(
    f"Timestamp   : "
    f"{format_timestamp(result_flagged['start'])} - "
    f"{format_timestamp(result_flagged['end'])}"
)

print(
    f"\nNLI         : "
    f"{result_flagged['nli_label']} "
    f"({result_flagged['nli_confidence']:.4f})"
)

print(
    f"Speaker     : "
    f"{'PASS' if result_flagged['speaker_consistent'] else 'FAIL'}"
)

print(
    f"Event Type  : "
    f"{'PASS' if result_flagged['event_type_consistent'] else 'FAIL'}"
)

print(
    f"FINAL       : "
    f"{'VERIFIED' if result_flagged['verified'] else 'FLAGGED'}"
)


print("\n" + "=" * 100)

COMPLETE MULTI-ATTRIBUTE VERIFICATION TEST

CASE 1 — DIRECTLY SUPPORTED CLAIM
----------------------------------------------------------------------------------------------------
Claim       : It should be accessible and usable by all age groups.
Evidence ID : 94
Evidence    : You've got Market Range International, and you did say earlier it's got to be accessible and usable by all age groups.
Timestamp   : 00:10:38 - 00:10:44

NLI         : entailment (0.9905)
Speaker     : PASS
Event Type  : PASS
FINAL       : VERIFIED


CASE 2 — CLAIM WITH UNSUPPORTED ASSOCIATION
----------------------------------------------------------------------------------------------------
Claim       : The remote control should be user-friendly and usable by everyone.
Evidence ID : 95
Evidence    : We're not focusing on business market, any particular thing. It's everyone, user-friendly to everyone.
Timestamp   : 00:10:46 - 00:10:50

NLI         : neutral (0.9995)
Speaker     : PASS
Event Type  : PASS
FINAL  

In [33]:
# =============================================================================
# Inspect BART Baseline Output
# =============================================================================
# Purpose:
# Display the BART-generated baseline MoM that will later be passed through
# the evidence retrieval and verification pipeline.
#
# We keep the original baseline unchanged so that the verification stage
# can objectively identify supported and unsupported claims.
# =============================================================================

import json

BART_BASELINE_PATH = (
    DATA_DIR
    / "transcripts"
    / "ES2004a_bart_baseline.json"
)

with open(BART_BASELINE_PATH, "r", encoding="utf-8") as f:
    bart_baseline = json.load(f)

print("=" * 100)
print("BART BASELINE MoM")
print("=" * 100)

print(json.dumps(
    bart_baseline,
    indent=2,
    ensure_ascii=False
))

print("\n" + "=" * 100)

BART BASELINE MoM
{
  "meeting_id": "ES2004a",
  "model": "facebook/bart-large-cnn",
  "num_chunks": 7,
  "summaries": [
    {
      "chunk_id": 1,
      "summary": "Project manager introduces herself and the team. They will work on a remote control that can be used by dogs and grannies. The team will also do tool training and discuss the project plan."
    },
    {
      "chunk_id": 2,
      "summary": "A selection of some of the most popular questions asked by the audience. The questions ranged from a cat to a crocodile to a T-Rex to a dog."
    },
    {
      "chunk_id": 3,
      "summary": "CNN.com's John Defterios takes on the challenge to create an animated GIF of a fictional bird of prey. He tries to think on the spot of what type of bird he wants to depict. The result is a picture of an eagle, a seagull, a big cat, a vampire bat and an eagle."
    },
    {
      "chunk_id": 4,
      "summary": "Market Range International is aiming to be accessible and usable by all age groups. 

In [34]:
# =============================================================================
# Extract Test Claims from BART Baseline
# =============================================================================
# Purpose:
# Create a small set of individual claims from the BART baseline output.
#
# These claims will be passed through the evidence retrieval and verification
# pipeline.
#
# We intentionally include:
#   1. A likely supported claim
#   2. A likely unsupported/hallucinated claim
#
# This allows us to test whether the verification system can distinguish
# grounded and ungrounded BART output.
# =============================================================================

baseline_test_claims = [

    {
        "claim_id": "BART_01",
        "text": "The project manager introduces herself and the team.",
        "source_chunk": 1
    },

    {
        "claim_id": "BART_02",
        "text": "The team will work on a remote control that can be used by dogs and grannies.",
        "source_chunk": 1
    },

    {
        "claim_id": "BART_03",
        "text": "CNN.com's John Defterios creates an animated GIF of a fictional bird of prey.",
        "source_chunk": 3
    },

    {
        "claim_id": "BART_04",
        "text": "The product is intended to be accessible and usable by all age groups.",
        "source_chunk": 4
    },

    {
        "claim_id": "BART_05",
        "text": "This week's show focuses on remote controls and how they can be made easier to use.",
        "source_chunk": 5
    }
]


print("=" * 100)
print("BART BASELINE TEST CLAIMS")
print("=" * 100)

for claim in baseline_test_claims:

    print(
        f"\nClaim ID     : {claim['claim_id']}"
    )

    print(
        f"Source chunk : {claim['source_chunk']}"
    )

    print(
        f"Claim        : {claim['text']}"
    )

    print("-" * 100)

print(
    f"\nTotal test claims: {len(baseline_test_claims)}"
)

print("=" * 100)

BART BASELINE TEST CLAIMS

Claim ID     : BART_01
Source chunk : 1
Claim        : The project manager introduces herself and the team.
----------------------------------------------------------------------------------------------------

Claim ID     : BART_02
Source chunk : 1
Claim        : The team will work on a remote control that can be used by dogs and grannies.
----------------------------------------------------------------------------------------------------

Claim ID     : BART_03
Source chunk : 3
Claim        : CNN.com's John Defterios creates an animated GIF of a fictional bird of prey.
----------------------------------------------------------------------------------------------------

Claim ID     : BART_04
Source chunk : 4
Claim        : The product is intended to be accessible and usable by all age groups.
----------------------------------------------------------------------------------------------------

Claim ID     : BART_05
Source chunk : 5
Claim        : This week'

In [35]:
# =============================================================================
# BART Claim → Evidence Retrieval
# =============================================================================
# Purpose:
# Retrieve the top-K transcript evidence segments for each BART-generated
# claim using BGE + FAISS.
#
# This is the retrieval stage of the proposed pipeline.
#
# IMPORTANT:
# Retrieval does NOT mean verification.
# The retrieved evidence will be passed to the multi-attribute verifier
# in the next stage.
# =============================================================================

BART_TOP_K = 5

bart_claim_retrievals = []

for claim in baseline_test_claims:

    # -------------------------------------------------------------------------
    # Create BGE embedding for the BART claim
    # -------------------------------------------------------------------------

    claim_embedding = bge_model.encode(
        [claim["text"]],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # -------------------------------------------------------------------------
    # Retrieve top-K evidence
    # -------------------------------------------------------------------------

    scores, indices = faiss_index.search(
        claim_embedding,
        BART_TOP_K
    )

    claim_results = []

    for rank, (score, index) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        evidence = evidence_documents[index]

        claim_results.append({
            "rank": rank,
            "similarity": float(score),
            "evidence_id": evidence["evidence_id"],
            "speaker": evidence["speaker"],
            "start": evidence["start"],
            "end": evidence["end"],
            "text": evidence["text"]
        })

    bart_claim_retrievals.append({
        "claim_id": claim["claim_id"],
        "claim": claim["text"],
        "source_chunk": claim["source_chunk"],
        "evidence": claim_results
    })


# =============================================================================
# Display retrieval results
# =============================================================================

print("=" * 110)
print("BART CLAIM → TOP-K EVIDENCE RETRIEVAL")
print("=" * 110)

for claim_result in bart_claim_retrievals:

    print(
        f"\n\nCLAIM {claim_result['claim_id']}"
    )

    print(
        f"Source chunk: {claim_result['source_chunk']}"
    )

    print(
        f"Claim: {claim_result['claim']}"
    )

    print("\nRetrieved evidence:")

    for evidence in claim_result["evidence"]:

        print(
            f"\n  Rank {evidence['rank']} | "
            f"Similarity: {evidence['similarity']:.4f}"
        )

        print(
            f"  Evidence ID: {evidence['evidence_id']}"
        )

        print(
            f"  Speaker: {evidence['speaker']}"
        )

        print(
            f"  Timestamp: "
            f"{format_timestamp(evidence['start'])} - "
            f"{format_timestamp(evidence['end'])}"
        )

        print(
            f"  Text: {evidence['text']}"
        )

        print("  " + "-" * 90)

print("\n" + "=" * 110)

BART CLAIM → TOP-K EVIDENCE RETRIEVAL


CLAIM BART_01
Source chunk: 1
Claim: The project manager introduces herself and the team.

Retrieved evidence:

  Rank 1 | Similarity: 0.7390
  Evidence ID: 9
  Speaker: SPEAKER_02
  Timestamp: 00:01:24 - 00:01:33
  Text: I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.
  ------------------------------------------------------------------------------------------

  Rank 2 | Similarity: 0.6682
  Evidence ID: 12
  Speaker: SPEAKER_02
  Timestamp: 00:01:46 - 00:01:49
  Text: talk about the project plan, discuss our own ideas and everything.
  ------------------------------------------------------------------------------------------

  Rank 3 | Similarity: 0.6175
  Evidence ID: 47
  Speaker: SPEAKER_02
  Timestamp: 00:06:15 - 00:06:16
  Text: the work today. I
  ------------------------------------------------------------------------------------------

  Rank 4 | Similarity: 0.5883
  Evidenc

In [4]:
# =============================================================================
# BART Claims → NLI Evidence Verification
# =============================================================================
# Purpose:
# Apply the validated NLI model to the Top-K retrieved evidence for every
# BART-generated claim.
#
# BGE + FAISS:
#     Finds candidate evidence.
#
# NLI:
#     Checks whether each candidate evidence segment actually supports
#     the BART claim.
#
# This cell does NOT make the final VERIFIED / FLAGGED decision yet.
# It only produces the content-support evidence needed for the next stage.
# =============================================================================

bart_nli_results = []

for claim_result in bart_claim_retrievals:

    claim_text = claim_result["claim"]

    claim_results = []

    for evidence in claim_result["evidence"]:

        # ---------------------------------------------------------------------
        # Run NLI
        #
        # Premise    = transcript evidence
        # Hypothesis = BART-generated claim
        # ---------------------------------------------------------------------

        nli_label, nli_confidence = run_nli_correct(
            evidence["text"],
            claim_text
        )

        claim_results.append({
            "rank": evidence["rank"],
            "evidence_id": evidence["evidence_id"],
            "retrieval_similarity": evidence["similarity"],
            "nli_label": nli_label,
            "nli_confidence": nli_confidence,
            "speaker": evidence["speaker"],
            "start": evidence["start"],
            "end": evidence["end"],
            "text": evidence["text"]
        })

    bart_nli_results.append({
        "claim_id": claim_result["claim_id"],
        "claim": claim_text,
        "source_chunk": claim_result["source_chunk"],
        "evidence": claim_results
    })


# =============================================================================
# Display results
# =============================================================================

print("=" * 110)
print("BART CLAIMS → NLI EVIDENCE VERIFICATION")
print("=" * 110)

for claim_result in bart_nli_results:

    print("\n")
    print("=" * 110)

    print(
        f"CLAIM {claim_result['claim_id']}"
    )

    print(
        f"Source chunk: {claim_result['source_chunk']}"
    )

    print(
        f"Claim: {claim_result['claim']}"
    )

    print("-" * 110)

    for evidence in claim_result["evidence"]:

        print(
            f"\nRank {evidence['rank']} | "
            f"Evidence ID: {evidence['evidence_id']}"
        )

        print(
            f"Retrieval similarity : "
            f"{evidence['retrieval_similarity']:.4f}"
        )

        print(
            f"NLI prediction       : "
            f"{evidence['nli_label']}"
        )

        print(
            f"NLI confidence       : "
            f"{evidence['nli_confidence']:.4f}"
        )

        print(
            f"Speaker              : "
            f"{evidence['speaker']}"
        )

        print(
            f"Timestamp            : "
            f"{format_timestamp(evidence['start'])} - "
            f"{format_timestamp(evidence['end'])}"
        )

        print(
            f"Evidence             : "
            f"{evidence['text']}"
        )

        print("-" * 100)

print("\n" + "=" * 110)

NameError: name 'bart_claim_retrievals' is not defined

In [5]:
# =============================================================================
# Rebuild BART Claim → Evidence Retrieval Results
# =============================================================================
# Purpose:
# Recreate the variable `bart_claim_retrievals` after a runtime reset or
# out-of-order cell execution.
#
# This uses the already-loaded BGE model and FAISS index.
# No models or packages are installed in this cell.
# =============================================================================

BART_TOP_K = 5

bart_claim_retrievals = []

for claim in baseline_test_claims:

    # -------------------------------------------------------------------------
    # Create BGE embedding for the claim
    # -------------------------------------------------------------------------

    claim_embedding = bge_model.encode(
        [claim["text"]],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # -------------------------------------------------------------------------
    # Retrieve top-K evidence
    # -------------------------------------------------------------------------

    scores, indices = faiss_index.search(
        claim_embedding,
        BART_TOP_K
    )

    claim_results = []

    for rank, (score, index) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        evidence = evidence_documents[index]

        claim_results.append({
            "rank": rank,
            "similarity": float(score),
            "evidence_id": evidence["evidence_id"],
            "speaker": evidence["speaker"],
            "start": evidence["start"],
            "end": evidence["end"],
            "text": evidence["text"]
        })

    bart_claim_retrievals.append({
        "claim_id": claim["claim_id"],
        "claim": claim["text"],
        "source_chunk": claim["source_chunk"],
        "evidence": claim_results
    })


# =============================================================================
# Confirm variable was successfully recreated
# =============================================================================

print("=" * 100)
print("BART RETRIEVAL RESULTS REBUILT")
print("=" * 100)

print(
    f"\nNumber of BART claims : "
    f"{len(bart_claim_retrievals)}"
)

print(
    f"Evidence per claim    : "
    f"{BART_TOP_K}"
)

print("\nReady for NLI verification.")

print("=" * 100)

NameError: name 'baseline_test_claims' is not defined

In [6]:
# =============================================================================
# Recreate BART Baseline Test Claims
# =============================================================================
# Purpose:
# Recreate the five test claims derived from the saved BART baseline output.
#
# This is needed because the current Colab runtime no longer contains the
# Python variable `baseline_test_claims`.
#
# No models or packages are installed in this cell.
# =============================================================================

baseline_test_claims = [

    {
        "claim_id": "BART_01",
        "text": "The project manager introduces herself and the team.",
        "source_chunk": 1
    },

    {
        "claim_id": "BART_02",
        "text": "The team will work on a remote control that can be used by dogs and grannies.",
        "source_chunk": 1
    },

    {
        "claim_id": "BART_03",
        "text": "CNN.com's John Defterios creates an animated GIF of a fictional bird of prey.",
        "source_chunk": 3
    },

    {
        "claim_id": "BART_04",
        "text": "The product is intended to be accessible and usable by all age groups.",
        "source_chunk": 4
    },

    {
        "claim_id": "BART_05",
        "text": "This week's show focuses on remote controls and how they can be made easier to use.",
        "source_chunk": 5
    }
]


print("=" * 100)
print("BART BASELINE TEST CLAIMS RECREATED")
print("=" * 100)

for claim in baseline_test_claims:
    print(
        f"{claim['claim_id']} | "
        f"Chunk {claim['source_chunk']} | "
        f"{claim['text']}"
    )

print(f"\nTotal claims: {len(baseline_test_claims)}")
print("=" * 100)

BART BASELINE TEST CLAIMS RECREATED
BART_01 | Chunk 1 | The project manager introduces herself and the team.
BART_02 | Chunk 1 | The team will work on a remote control that can be used by dogs and grannies.
BART_03 | Chunk 3 | CNN.com's John Defterios creates an animated GIF of a fictional bird of prey.
BART_04 | Chunk 4 | The product is intended to be accessible and usable by all age groups.
BART_05 | Chunk 5 | This week's show focuses on remote controls and how they can be made easier to use.

Total claims: 5


In [7]:
# =============================================================================
# Rebuild BART Claim → Evidence Retrieval Results
# =============================================================================
# Purpose:
# Recreate the Top-K evidence retrieved for each BART-generated claim.
#
# Uses:
#   BGE embeddings + existing FAISS index
#
# No packages or models are installed in this cell.
# =============================================================================

BART_TOP_K = 5

bart_claim_retrievals = []

for claim in baseline_test_claims:

    # Create BGE embedding for the claim
    claim_embedding = bge_model.encode(
        [claim["text"]],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # Retrieve Top-K evidence
    scores, indices = faiss_index.search(
        claim_embedding,
        BART_TOP_K
    )

    claim_results = []

    for rank, (score, index) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        evidence = evidence_documents[index]

        claim_results.append({
            "rank": rank,
            "similarity": float(score),
            "evidence_id": evidence["evidence_id"],
            "speaker": evidence["speaker"],
            "start": evidence["start"],
            "end": evidence["end"],
            "text": evidence["text"]
        })

    bart_claim_retrievals.append({
        "claim_id": claim["claim_id"],
        "claim": claim["text"],
        "source_chunk": claim["source_chunk"],
        "evidence": claim_results
    })


# Confirm successful reconstruction
print("=" * 100)
print("BART CLAIM RETRIEVAL REBUILT")
print("=" * 100)

print(f"Claims processed : {len(bart_claim_retrievals)}")
print(f"Top-K per claim  : {BART_TOP_K}")

print("\nReady for NLI verification.")

print("=" * 100)

NameError: name 'bge_model' is not defined

In [8]:
# =============================================================================
# Restore BGE Embedding Model
# =============================================================================
# Purpose:
# Reload the BGE model required for semantic evidence retrieval.
#
# The model was previously used successfully on the Tesla T4.
# No package installation is performed here.
# =============================================================================

from sentence_transformers import SentenceTransformer
import torch

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("=" * 80)
print("LOADING BGE EMBEDDING MODEL")
print("=" * 80)

BGE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=BGE_DEVICE
)

print(f"\nModel  : {BGE_MODEL_NAME}")
print(f"Device : {BGE_DEVICE}")

print("\nBGE model restored successfully.")

LOADING BGE EMBEDDING MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Model  : BAAI/bge-small-en-v1.5
Device : cuda

BGE model restored successfully.


In [9]:
# =============================================================================
# Restore Evidence Documents
# =============================================================================
# Purpose:
# Recreate the evidence-document list from the saved speaker-attributed
# transcript after the Colab runtime reset.
#
# Each speaker-attributed utterance becomes one evidence document containing:
#   - Evidence ID
#   - Speaker
#   - Start/end timestamp
#   - Duration
#   - Transcript text
#
# No model or package installation is performed here.
# =============================================================================

import json

SPEAKER_TRANSCRIPT_PATH = (
    DATA_DIR
    / "transcripts"
    / "ES2004a_speaker_transcript.json"
)

with open(
    SPEAKER_TRANSCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    speaker_transcript_data = json.load(f)


# -------------------------------------------------------------------------
# Inspect structure and extract speaker utterances
# -------------------------------------------------------------------------

print("=" * 80)
print("RESTORING EVIDENCE DOCUMENTS")
print("=" * 80)

print(
    f"\nLoaded file: {SPEAKER_TRANSCRIPT_PATH}"
)

print(
    f"Top-level type: {type(speaker_transcript_data).__name__}"
)

print(
    f"Top-level keys: "
    f"{list(speaker_transcript_data.keys())}"
)


# The saved file contains the speaker utterances used earlier.
speaker_utterances = speaker_transcript_data["speaker_utterances"]


# -------------------------------------------------------------------------
# Build evidence documents
# -------------------------------------------------------------------------

evidence_documents = []

for evidence_id, utterance in enumerate(
    speaker_utterances,
    start=1
):

    evidence_documents.append({
        "evidence_id": evidence_id,
        "speaker": utterance["speaker"],
        "start": float(utterance["start"]),
        "end": float(utterance["end"]),
        "duration": float(
            utterance["end"] - utterance["start"]
        ),
        "text": utterance["text"]
    })


# -------------------------------------------------------------------------
# Display confirmation
# -------------------------------------------------------------------------

print(
    f"\nEvidence documents restored: "
    f"{len(evidence_documents)}"
)

print("\nFirst evidence document:")
print(evidence_documents[0])

print("\nLast evidence document:")
print(evidence_documents[-1])

print("\nEvidence documents are ready.")

print("=" * 80)

NameError: name 'DATA_DIR' is not defined

In [10]:
# =============================================================================
# Restore Project Paths
# =============================================================================
# Purpose:
# Recreate the project directory variables after the Colab runtime reset.
#
# No packages or models are installed in this cell.
# =============================================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

DATA_DIR = PROJECT_DIR / "data"

TRANSCRIPTS_DIR = DATA_DIR / "transcripts"

print("=" * 80)
print("PROJECT PATHS RESTORED")
print("=" * 80)

print(f"\nProject directory : {PROJECT_DIR}")
print(f"Data directory    : {DATA_DIR}")
print(f"Transcripts       : {TRANSCRIPTS_DIR}")

print(
    f"\nProject exists    : "
    f"{PROJECT_DIR.exists()}"
)

print(
    f"Transcript folder : "
    f"{TRANSCRIPTS_DIR.exists()}"
)

print("\nPaths are ready.")
print("=" * 80)

PROJECT PATHS RESTORED

Project directory : /content/drive/MyDrive/MTechIndProj/MoM_Project
Data directory    : /content/drive/MyDrive/MTechIndProj/MoM_Project/data
Transcripts       : /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts

Project exists    : True
Transcript folder : True

Paths are ready.


In [3]:
# =============================================================================
# Restore Evidence Documents
# =============================================================================
# Purpose:
# Recreate the evidence-document list from the saved speaker-attributed
# transcript after the Colab runtime reset.
#
# No model or package installation is performed in this cell.
# =============================================================================

import json

SPEAKER_TRANSCRIPT_PATH = (
    TRANSCRIPTS_DIR
    / "ES2004a_speaker_transcript.json"
)

with open(
    SPEAKER_TRANSCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    speaker_transcript_data = json.load(f)


# -------------------------------------------------------------------------
# Extract speaker-attributed utterances
# -------------------------------------------------------------------------

speaker_utterances = speaker_transcript_data["speaker_utterances"]


# -------------------------------------------------------------------------
# Build evidence documents
# -------------------------------------------------------------------------

evidence_documents = []

for evidence_id, utterance in enumerate(
    speaker_utterances,
    start=1
):

    evidence_documents.append({
        "evidence_id": evidence_id,
        "speaker": utterance["speaker"],
        "start": float(utterance["start"]),
        "end": float(utterance["end"]),
        "duration": float(
            utterance["end"] - utterance["start"]
        ),
        "text": utterance["text"]
    })


# -------------------------------------------------------------------------
# Verify restoration
# -------------------------------------------------------------------------

print("=" * 80)
print("EVIDENCE DOCUMENTS RESTORED")
print("=" * 80)

print(
    f"\nTranscript file: "
    f"{SPEAKER_TRANSCRIPT_PATH}"
)

print(
    f"Number of evidence documents: "
    f"{len(evidence_documents)}"
)

print("\nFirst evidence:")
print(evidence_documents[0])

print("\nEvidence 94:")
print(evidence_documents[93])

print("\nEvidence 95:")
print(evidence_documents[94])

print("\nEvidence documents are ready.")

print("=" * 80)

NameError: name 'TRANSCRIPTS_DIR' is not defined

In [4]:
# ============================================================
# CHECK EXISTING SPEAKER UTTERANCES
# Purpose:
# Check whether the speaker_utterances variable created in
# Cell 35A is still available after the runtime reset.
# ============================================================

if "speaker_utterances" in globals():

    print("=" * 80)
    print("speaker_utterances IS AVAILABLE")
    print("=" * 80)

    print("\nNumber of speaker utterances:", len(speaker_utterances))

    print("\nFirst utterance:")
    print(speaker_utterances[0])

    print("\nLast utterance:")
    print(speaker_utterances[-1])

else:

    print("=" * 80)
    print("speaker_utterances IS NOT AVAILABLE")
    print("=" * 80)

    print(
        "\nThe variable was lost during the runtime reset."
    )

speaker_utterances IS NOT AVAILABLE

The variable was lost during the runtime reset.


In [2]:
# ============================================================
# RESTORE SPEAKER UTTERANCES
# Purpose:
# Check whether the speaker_utterances variable from Cell 35A
# is still available in the current Colab runtime.
# ============================================================

if "speaker_utterances" in globals():

    print("=" * 80)
    print("speaker_utterances IS AVAILABLE")
    print("=" * 80)

    print("\nNumber of speaker utterances:", len(speaker_utterances))

    print("\nFirst utterance:")
    print(speaker_utterances[0])

    print("\nLast utterance:")
    print(speaker_utterances[-1])

else:

    print("=" * 80)
    print("speaker_utterances IS NOT AVAILABLE")
    print("=" * 80)

    print("\nThe variable was lost during the runtime reset.")

speaker_utterances IS NOT AVAILABLE

The variable was lost during the runtime reset.


In [3]:
# ============================================================
# INSPECT SAVED SPEAKER TRANSCRIPT
# Purpose:
# Determine the actual JSON structure saved by Cell 35A.
# We will only READ the file — nothing will be modified.
# ============================================================

import json

SPEAKER_TRANSCRIPT_PATH = (
    TRANSCRIPTS_DIR
    / "ES2004a_speaker_transcript.json"
)

with open(
    SPEAKER_TRANSCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    transcript_data = json.load(f)

print("=" * 80)
print("SAVED TRANSCRIPT INSPECTION")
print("=" * 80)

print("\nFile:")
print(SPEAKER_TRANSCRIPT_PATH)

print("\nTop-level type:")
print(type(transcript_data).__name__)

if isinstance(transcript_data, dict):

    print("\nTop-level keys:")
    for key in transcript_data.keys():
        print("  -", key)

elif isinstance(transcript_data, list):

    print("\nNumber of items:", len(transcript_data))

print("\nFirst 2000 characters of saved data:")
print(
    json.dumps(
        transcript_data,
        indent=2,
        ensure_ascii=False
    )[:2000]
)

print("\n" + "=" * 80)

NameError: name 'TRANSCRIPTS_DIR' is not defined

In [4]:
# ============================================================
# CHECK REQUIRED TRANSCRIPT FILE
# Purpose:
# Verify that the speaker-attributed transcript created from
# Cell 35A is still available on Google Drive.
# ============================================================

from pathlib import Path

SPEAKER_TRANSCRIPT_PATH = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project/"
    "data/transcripts/ES2004a_speaker_transcript.json"
)

BART_BASELINE_PATH = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project/"
    "data/transcripts/ES2004a_bart_baseline.json"
)

print("=" * 80)
print("CHECKING TRANSCRIPT FILES")
print("=" * 80)

print("\nSpeaker-attributed transcript:")
print(SPEAKER_TRANSCRIPT_PATH)
print("Exists:", SPEAKER_TRANSCRIPT_PATH.exists())

print("\nBART baseline:")
print(BART_BASELINE_PATH)
print("Exists:", BART_BASELINE_PATH.exists())

print("=" * 80)

CHECKING TRANSCRIPT FILES

Speaker-attributed transcript:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_speaker_transcript.json
Exists: True

BART baseline:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_bart_baseline.json
Exists: True


In [5]:
# ============================================================
# RESTORE SPEAKER UTTERANCES FROM SAVED JSON
# Purpose:
# Recreate the speaker_utterances variable from the
# speaker-attributed transcript saved earlier.
# ============================================================

import json

with open(
    SPEAKER_TRANSCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    transcript_data = json.load(f)

print("=" * 80)
print("TRANSCRIPT FILE LOADED")
print("=" * 80)

print("\nTop-level type:", type(transcript_data).__name__)

if isinstance(transcript_data, dict):
    print("Top-level keys:", list(transcript_data.keys()))
else:
    print("Number of items:", len(transcript_data))

print("=" * 80)

TRANSCRIPT FILE LOADED

Top-level type: dict
Top-level keys: ['meeting_id', 'source', 'gap_threshold_seconds', 'num_speakers', 'speakers', 'num_utterances', 'utterances']


In [6]:
# ============================================================
# RESTORE SPEAKER UTTERANCES
# Purpose:
# Recreate speaker_utterances from the saved JSON file.
# ============================================================

speaker_utterances = transcript_data["utterances"]

print("=" * 80)
print("SPEAKER UTTERANCES RESTORED")
print("=" * 80)

print("\nMeeting ID:", transcript_data["meeting_id"])
print("Number of speakers:", transcript_data["num_speakers"])
print("Number of utterances:", len(speaker_utterances))

print("\nSpeakers:")
print(transcript_data["speakers"])

print("\nFirst 3 speaker utterances:")

for i, utterance in enumerate(speaker_utterances[:3], start=1):

    print(
        f"\n{i}. {utterance['speaker']}"
        f" | {utterance['start']:.2f}s"
        f" → {utterance['end']:.2f}s"
        f" | {utterance['duration']:.2f}s"
    )

    print("   ", utterance["text"])

print("\n" + "=" * 80)

SPEAKER UTTERANCES RESTORED

Meeting ID: ES2004a
Number of speakers: 4
Number of utterances: 148

Speakers:
['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02', 'SPEAKER_03']

First 3 speaker utterances:

1. SPEAKER_02 | 11.00s → 14.52s | 3.52s
    Are we, we're not like the dim lights, so we can see that a bit better.

2. SPEAKER_01 | 17.94s → 18.16s | 0.22s
    Yeah.

3. SPEAKER_02 | 18.94s → 20.95s | 2.00s
    Okay, that's fine.



In [7]:
# ============================================================
# BUILD EVIDENCE DOCUMENTS
# Purpose:
# Convert each speaker utterance into an evidence document
# for BGE embedding and FAISS retrieval.
#
# Evidence IDs are kept in the same order as the original
# 148 speaker utterances.
# ============================================================

evidence_documents = []

for evidence_id, utterance in enumerate(
    speaker_utterances,
    start=1
):

    evidence_documents.append({
        "evidence_id": evidence_id,
        "speaker": utterance["speaker"],
        "start": float(utterance["start"]),
        "end": float(utterance["end"]),
        "duration": float(utterance["duration"]),
        "text": utterance["text"]
    })


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("=" * 80)
print("EVIDENCE DOCUMENTS CREATED")
print("=" * 80)

print("\nNumber of evidence documents:",
      len(evidence_documents))

print("\nFirst evidence document:")
print(evidence_documents[0])

print("\nEvidence 94:")
print(evidence_documents[93])

print("\nEvidence 95:")
print(evidence_documents[94])

print("\nLast evidence document:")
print(evidence_documents[-1])

print("\n" + "=" * 80)

EVIDENCE DOCUMENTS CREATED

Number of evidence documents: 148

First evidence document:
{'evidence_id': 1, 'speaker': 'SPEAKER_02', 'start': 10.998, 'end': 14.521, 'duration': 3.5230000000000015, 'text': "Are we, we're not like the dim lights, so we can see that a bit better."}

Evidence 94:
{'evidence_id': 94, 'speaker': 'SPEAKER_03', 'start': 638.101, 'end': 644.951, 'duration': 6.850000000000023, 'text': "You've got Market Range International, and you did say earlier it's got to be accessible and usable by all age groups."}

Evidence 95:
{'evidence_id': 95, 'speaker': 'SPEAKER_03', 'start': 646.213, 'end': 650.76, 'duration': 4.5470000000000255, 'text': "We're not focusing on business market, any particular thing. It's everyone, user-friendly to everyone."}

Last evidence document:
{'evidence_id': 148, 'speaker': 'SPEAKER_03', 'start': 1046.428, 'end': 1048.43, 'duration': 2.0019999999999527, 'text': 'Do we go back to our room? Yeah.'}



In [8]:
# ============================================================
# GENERATE BGE EVIDENCE EMBEDDINGS
# Purpose:
# Convert each evidence document into a semantic embedding
# using BAAI/bge-small-en-v1.5.
#
# These embeddings will be indexed by FAISS for evidence
# retrieval during MoM verification.
# ============================================================

import numpy as np

# Extract evidence text
evidence_texts = [
    item["text"]
    for item in evidence_documents
]

print("=" * 80)
print("GENERATING BGE EVIDENCE EMBEDDINGS")
print("=" * 80)

print("\nNumber of evidence documents:",
      len(evidence_texts))

# Generate normalized embeddings
evidence_embeddings = bge_model.encode(
    evidence_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

# Ensure float32 for FAISS
evidence_embeddings = evidence_embeddings.astype(
    np.float32
)

print("\nEmbedding shape:",
      evidence_embeddings.shape)

print("Embedding dtype:",
      evidence_embeddings.dtype)

print(
    "\nExpected shape: "
    f"({len(evidence_documents)}, 384)"
)

print("\nBGE evidence embeddings generated successfully.")

print("=" * 80)

GENERATING BGE EVIDENCE EMBEDDINGS

Number of evidence documents: 148


NameError: name 'bge_model' is not defined

In [9]:
# ============================================================
# RELOAD BGE EMBEDDING MODEL
# Purpose:
# Reload BAAI/bge-small-en-v1.5 into the current Colab
# runtime after the runtime reset.
# ============================================================

from sentence_transformers import SentenceTransformer
import torch

print("=" * 80)
print("LOADING BGE MODEL")
print("=" * 80)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("\nDevice:", DEVICE)

bge_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device=DEVICE
)

print("\nModel loaded successfully:")
print("BAAI/bge-small-en-v1.5")

print("=" * 80)

LOADING BGE MODEL

Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Model loaded successfully:
BAAI/bge-small-en-v1.5


In [10]:
# ============================================================
# GENERATE BGE EVIDENCE EMBEDDINGS
# Purpose:
# Convert the 148 speaker-attributed evidence documents into
# 384-dimensional normalized embeddings for FAISS retrieval.
# ============================================================

import numpy as np

# Extract text from each evidence document
evidence_texts = [
    item["text"]
    for item in evidence_documents
]

print("=" * 80)
print("GENERATING BGE EVIDENCE EMBEDDINGS")
print("=" * 80)

print("\nNumber of evidence documents:",
      len(evidence_texts))

# Generate normalized embeddings
evidence_embeddings = bge_model.encode(
    evidence_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

# FAISS works efficiently with float32
evidence_embeddings = evidence_embeddings.astype(
    np.float32
)

print("\nEmbedding shape:",
      evidence_embeddings.shape)

print("Embedding dtype:",
      evidence_embeddings.dtype)

print(
    "\nExpected shape:",
    (len(evidence_documents), 384)
)

print("\nBGE evidence embeddings generated successfully.")

print("=" * 80)

GENERATING BGE EVIDENCE EMBEDDINGS

Number of evidence documents: 148


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embedding shape: (148, 384)
Embedding dtype: float32

Expected shape: (148, 384)

BGE evidence embeddings generated successfully.


In [11]:
# ============================================================
# BUILD FAISS EVIDENCE INDEX
# Purpose:
# Build a FAISS inner-product index over the normalized BGE
# evidence embeddings.
#
# Because the embeddings are normalized, inner product is
# equivalent to cosine similarity.
# ============================================================

import faiss

print("=" * 80)
print("BUILDING FAISS EVIDENCE INDEX")
print("=" * 80)

# Confirm embedding dimensions
embedding_dimension = evidence_embeddings.shape[1]

print("\nEmbedding dimension:", embedding_dimension)
print("Number of evidence vectors:", evidence_embeddings.shape[0])

# Create FAISS index
faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

# Add evidence embeddings
faiss_index.add(evidence_embeddings)

print("\nFAISS index created successfully.")
print("Index type:", type(faiss_index).__name__)
print("Vectors indexed:", faiss_index.ntotal)

print("\nExpected vectors:", len(evidence_documents))

print("\nFAISS evidence retrieval layer is ready.")

print("=" * 80)

ModuleNotFoundError: No module named 'faiss'

In [12]:
# ============================================================
# INSTALL FAISS
# Purpose:
# Install FAISS for semantic evidence retrieval.
# ============================================================

!pip install -q faiss-cpu

print("FAISS installation completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.1 MB/s eta 0:00:00
FAISS installation completed.


In [13]:
# ============================================================
# BUILD FAISS EVIDENCE INDEX
# Purpose:
# Build a FAISS inner-product index over the normalized BGE
# evidence embeddings.
#
# Since the BGE embeddings are normalized, inner product
# corresponds to cosine similarity.
# ============================================================

import faiss

print("=" * 80)
print("BUILDING FAISS EVIDENCE INDEX")
print("=" * 80)

# Get embedding dimension
embedding_dimension = evidence_embeddings.shape[1]

print("\nEmbedding dimension:", embedding_dimension)
print("Evidence vectors:", evidence_embeddings.shape[0])

# Create FAISS index
faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

# Add normalized BGE embeddings
faiss_index.add(evidence_embeddings)

print("\nFAISS index created successfully.")
print("Index type:", type(faiss_index).__name__)
print("Vectors indexed:", faiss_index.ntotal)

print("\nExpected vectors:", len(evidence_documents))

print("\nEvidence retrieval layer is ready.")

print("=" * 80)

BUILDING FAISS EVIDENCE INDEX

Embedding dimension: 384
Evidence vectors: 148

FAISS index created successfully.
Index type: IndexFlatIP
Vectors indexed: 148

Expected vectors: 148

Evidence retrieval layer is ready.


In [14]:
# ============================================================
# TEST FAISS EVIDENCE RETRIEVAL
# Purpose:
# Verify that BGE + FAISS can retrieve relevant meeting
# evidence for a generated MoM claim.
# ============================================================

import numpy as np

# Test claim
test_claim = (
    "It should be accessible and usable by all age groups."
)

print("=" * 80)
print("FAISS EVIDENCE RETRIEVAL TEST")
print("=" * 80)

print("\nClaim:")
print(test_claim)

# ------------------------------------------------------------
# Encode the claim using the same BGE model
# ------------------------------------------------------------

claim_embedding = bge_model.encode(
    [test_claim],
    normalize_embeddings=True,
    convert_to_numpy=True
).astype(np.float32)

# ------------------------------------------------------------
# Retrieve top 10 evidence candidates
# ------------------------------------------------------------

TOP_K = 10

similarities, indices = faiss_index.search(
    claim_embedding,
    TOP_K
)

# ------------------------------------------------------------
# Display retrieved evidence
# ------------------------------------------------------------

print(f"\nTop {TOP_K} retrieved evidence candidates:")
print("-" * 80)

for rank, (similarity, index) in enumerate(
    zip(similarities[0], indices[0]),
    start=1
):

    evidence = evidence_documents[index]

    print(
        f"\nRank {rank}"
        f" | Similarity: {similarity:.4f}"
        f" | Evidence ID: {evidence['evidence_id']}"
    )

    print(
        f"Speaker: {evidence['speaker']}"
        f" | {evidence['start']:.2f}s"
        f" → {evidence['end']:.2f}s"
    )

    print("Text:", evidence["text"])

print("\n" + "=" * 80)

FAISS EVIDENCE RETRIEVAL TEST

Claim:
It should be accessible and usable by all age groups.

Top 10 retrieved evidence candidates:
--------------------------------------------------------------------------------

Rank 1 | Similarity: 0.7465 | Evidence ID: 94
Speaker: SPEAKER_03 | 638.10s → 644.95s
Text: You've got Market Range International, and you did say earlier it's got to be accessible and usable by all age groups.

Rank 2 | Similarity: 0.6753 | Evidence ID: 17
Speaker: SPEAKER_02 | 139.38s → 141.02s
Text: maybe even pooches, should be able to use it.

Rank 3 | Similarity: 0.6518 | Evidence ID: 21
Speaker: SPEAKER_02 | 184.42s → 191.14s
Text: OK, space for everyone else.

Rank 4 | Similarity: 0.6231 | Evidence ID: 120
Speaker: SPEAKER_03 | 834.67s → 844.66s
Text: So that's a problem regardless of any design modifications you come up with. That's going to be a problem anyway with the older generation perhaps and that's another issue how we tackle that.

Rank 5 | Similarity: 0.6200 

In [15]:
# ============================================================
# RELOAD NLI MODEL
# Purpose:
# Reload the Natural Language Inference model used to verify
# whether retrieved meeting evidence supports a generated
# MoM claim.
#
# Model:
# cross-encoder/nli-deberta-v3-small
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
import torch

print("=" * 80)
print("LOADING NLI VERIFICATION MODEL")
print("=" * 80)

NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nModel:", NLI_MODEL_NAME)
print("Device:", DEVICE)

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
)

nli_model.to(DEVICE)
nli_model.eval()

print("\nNLI model loaded successfully.")

print("=" * 80)

LOADING NLI VERIFICATION MODEL

Model: cross-encoder/nli-deberta-v3-small
Device: cuda


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]


NLI model loaded successfully.


In [16]:
# ============================================================
# NLI SANITY TEST
# Purpose:
# Verify that the NLI model can distinguish:
#
#   1. Entailment  → evidence supports the claim
#   2. Neutral     → evidence is unrelated/insufficient
#   3. Contradiction → evidence conflicts with the claim
#
# Important:
# Evidence is the PREMISE.
# Claim is the HYPOTHESIS.
# ============================================================

import torch

def run_nli_correct(evidence, claim):
    """
    Run NLI with:
        premise    = meeting evidence
        hypothesis = generated MoM claim
    """

    inputs = nli_tokenizer(
        evidence,
        claim,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    predicted_index = int(
        torch.argmax(probabilities)
    )

    # DeBERTa NLI label mapping
    label = nli_model.config.id2label[
        predicted_index
    ]

    confidence = float(
        probabilities[predicted_index]
    )

    return label, confidence


# ------------------------------------------------------------
# Test 1: Entailment
# ------------------------------------------------------------

evidence_1 = (
    "You've got Market Range International, and you did "
    "say earlier it's got to be accessible and usable "
    "by all age groups."
)

claim_1 = (
    "It should be accessible and usable by all age groups."
)


# ------------------------------------------------------------
# Test 2: Neutral
# ------------------------------------------------------------

evidence_2 = (
    "The remote control should have a flip-top design."
)

claim_2 = (
    "It should be accessible and usable by all age groups."
)


# ------------------------------------------------------------
# Test 3: Contradiction
# ------------------------------------------------------------

evidence_3 = (
    "The product is intended only for business users."
)

claim_3 = (
    "The product is intended for everyone."
)


# ------------------------------------------------------------
# Run tests
# ------------------------------------------------------------

tests = [
    ("ENTAILMENT TEST", evidence_1, claim_1),
    ("NEUTRAL TEST", evidence_2, claim_2),
    ("CONTRADICTION TEST", evidence_3, claim_3)
]

print("=" * 80)
print("NLI SANITY TEST")
print("=" * 80)

for test_name, evidence, claim in tests:

    label, confidence = run_nli_correct(
        evidence,
        claim
    )

    print(f"\n{test_name}")
    print("-" * 80)

    print("Evidence:", evidence)
    print("Claim   :", claim)

    print(
        f"Prediction: {label}"
        f" | Confidence: {confidence:.4f}"
    )

print("\n" + "=" * 80)

NLI SANITY TEST

ENTAILMENT TEST
--------------------------------------------------------------------------------
Evidence: You've got Market Range International, and you did say earlier it's got to be accessible and usable by all age groups.
Claim   : It should be accessible and usable by all age groups.
Prediction: entailment | Confidence: 0.9905

NEUTRAL TEST
--------------------------------------------------------------------------------
Evidence: The remote control should have a flip-top design.
Claim   : It should be accessible and usable by all age groups.
Prediction: neutral | Confidence: 0.9992

CONTRADICTION TEST
--------------------------------------------------------------------------------
Evidence: The product is intended only for business users.
Claim   : The product is intended for everyone.
Prediction: contradiction | Confidence: 0.9991



In [17]:
# ============================================================
# CONTENT VERIFICATION
# Purpose:
# Verify whether a retrieved evidence segment actually
# supports a generated MoM claim.
#
# BGE/FAISS performs retrieval.
# DeBERTa NLI performs content verification.
#
# This keeps RETRIEVAL and VERIFICATION as separate stages.
# ============================================================

def verify_content(evidence_text, claim_text):
    """
    Verify whether evidence_text supports claim_text.

    Parameters
    ----------
    evidence_text : str
        Retrieved meeting evidence.

    claim_text : str
        Generated MoM claim.

    Returns
    -------
    dict
        NLI label and confidence.
    """

    label, confidence = run_nli_correct(
        evidence_text,
        claim_text
    )

    # Content passes only when NLI predicts entailment
    content_consistent = (
        label.lower() == "entailment"
    )

    return {
        "nli_label": label,
        "nli_confidence": confidence,
        "content_consistent": content_consistent
    }


# ------------------------------------------------------------
# Test using our known supporting evidence
# ------------------------------------------------------------

test_evidence = evidence_documents[93]["text"]

test_claim = (
    "It should be accessible and usable by all age groups."
)

content_result = verify_content(
    test_evidence,
    test_claim
)

print("=" * 80)
print("CONTENT VERIFICATION TEST")
print("=" * 80)

print("\nEvidence:")
print(test_evidence)

print("\nClaim:")
print(test_claim)

print("\nVerification result:")
print(content_result)

print("\n" + "=" * 80)

CONTENT VERIFICATION TEST

Evidence:
You've got Market Range International, and you did say earlier it's got to be accessible and usable by all age groups.

Claim:
It should be accessible and usable by all age groups.

Verification result:
{'nli_label': 'entailment', 'nli_confidence': 0.9904572367668152, 'content_consistent': True}



In [18]:
# ============================================================
# SPEAKER CONSISTENCY CHECK
# Purpose:
# Verify that the speaker attributed to a generated MoM claim
# matches the speaker in the selected evidence.
#
# Important:
# Speaker consistency is checked only when a speaker is
# explicitly specified in the generated claim metadata.
# ============================================================

def verify_speaker(claim_speaker, evidence_speaker):
    """
    Compare the claimed speaker with the evidence speaker.

    Parameters
    ----------
    claim_speaker : str or None
        Speaker attributed to the generated claim.

    evidence_speaker : str
        Speaker associated with the retrieved evidence.

    Returns
    -------
    dict
        Speaker consistency result.
    """

    # No speaker attribution in the claim
    if claim_speaker is None:
        return {
            "speaker_check_applicable": False,
            "speaker_consistent": True
        }

    speaker_consistent = (
        claim_speaker == evidence_speaker
    )

    return {
        "speaker_check_applicable": True,
        "speaker_consistent": speaker_consistent
    }


# ------------------------------------------------------------
# Test 1 — Correct speaker
# ------------------------------------------------------------

correct_speaker_result = verify_speaker(
    "SPEAKER_03",
    "SPEAKER_03"
)


# ------------------------------------------------------------
# Test 2 — Incorrect speaker
# ------------------------------------------------------------

incorrect_speaker_result = verify_speaker(
    "SPEAKER_02",
    "SPEAKER_03"
)


# ------------------------------------------------------------
# Test 3 — No speaker attribution
# ------------------------------------------------------------

no_speaker_result = verify_speaker(
    None,
    "SPEAKER_03"
)


print("=" * 80)
print("SPEAKER CONSISTENCY TEST")
print("=" * 80)

print("\nTest 1 — Correct speaker:")
print(correct_speaker_result)

print("\nTest 2 — Incorrect speaker:")
print(incorrect_speaker_result)

print("\nTest 3 — Speaker not specified:")
print(no_speaker_result)

print("\n" + "=" * 80)

SPEAKER CONSISTENCY TEST

Test 1 — Correct speaker:
{'speaker_check_applicable': True, 'speaker_consistent': True}

Test 2 — Incorrect speaker:
{'speaker_check_applicable': True, 'speaker_consistent': False}

Test 3 — Speaker not specified:
{'speaker_check_applicable': False, 'speaker_consistent': True}



In [19]:
# ============================================================
# TIME CONSISTENCY CHECK
# Purpose:
# Verify that the timestamp associated with a generated MoM
# claim is consistent with the timestamp of the retrieved
# evidence.
#
# Important:
# Time checking is applicable only when the claim contains
# timestamp information.
#
# A small tolerance is allowed because timestamps may be
# rounded or slightly shifted during processing.
# ============================================================

def verify_time(
    claim_start=None,
    claim_end=None,
    evidence_start=None,
    evidence_end=None,
    tolerance=2.0
):
    """
    Compare claim time range with evidence time range.

    Parameters
    ----------
    claim_start : float or None
        Start time associated with the claim.

    claim_end : float or None
        End time associated with the claim.

    evidence_start : float
        Start time of retrieved evidence.

    evidence_end : float
        End time of retrieved evidence.

    tolerance : float
        Allowed timestamp difference in seconds.

    Returns
    -------
    dict
        Time consistency result.
    """

    # --------------------------------------------------------
    # No claim timestamp → time check not applicable
    # --------------------------------------------------------

    if claim_start is None or claim_end is None:

        return {
            "time_check_applicable": False,
            "time_consistent": True
        }

    # --------------------------------------------------------
    # Expand evidence range slightly using tolerance
    # --------------------------------------------------------

    evidence_start_with_tolerance = (
        evidence_start - tolerance
    )

    evidence_end_with_tolerance = (
        evidence_end + tolerance
    )

    # --------------------------------------------------------
    # Check whether the two time ranges overlap
    # --------------------------------------------------------

    time_consistent = (
        claim_start <= evidence_end_with_tolerance
        and
        claim_end >= evidence_start_with_tolerance
    )

    return {
        "time_check_applicable": True,
        "time_consistent": time_consistent
    }


# ============================================================
# TEST CASES
# ============================================================

# Evidence 94:
# 638.101 → 644.951 seconds

evidence_start = evidence_documents[93]["start"]
evidence_end = evidence_documents[93]["end"]


# ------------------------------------------------------------
# Test 1 — Matching timestamp
# ------------------------------------------------------------

matching_time = verify_time(
    claim_start=638.0,
    claim_end=645.0,
    evidence_start=evidence_start,
    evidence_end=evidence_end
)


# ------------------------------------------------------------
# Test 2 — Incorrect timestamp
# ------------------------------------------------------------

incorrect_time = verify_time(
    claim_start=800.0,
    claim_end=810.0,
    evidence_start=evidence_start,
    evidence_end=evidence_end
)


# ------------------------------------------------------------
# Test 3 — No timestamp in claim
# ------------------------------------------------------------

no_time = verify_time(
    claim_start=None,
    claim_end=None,
    evidence_start=evidence_start,
    evidence_end=evidence_end
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 80)
print("TIME CONSISTENCY TEST")
print("=" * 80)

print(
    f"\nEvidence time: "
    f"{evidence_start:.3f}s → {evidence_end:.3f}s"
)

print("\nTest 1 — Matching timestamp:")
print(matching_time)

print("\nTest 2 — Incorrect timestamp:")
print(incorrect_time)

print("\nTest 3 — Timestamp not specified:")
print(no_time)

print("\n" + "=" * 80)

TIME CONSISTENCY TEST

Evidence time: 638.101s → 644.951s

Test 1 — Matching timestamp:
{'time_check_applicable': True, 'time_consistent': True}

Test 2 — Incorrect timestamp:
{'time_check_applicable': True, 'time_consistent': False}

Test 3 — Timestamp not specified:
{'time_check_applicable': False, 'time_consistent': True}



In [20]:
# ============================================================
# EVENT-TYPE CONSISTENCY CHECK
# Purpose:
# Compare the event type assigned to a generated MoM claim
# with the event type associated with the evidence.
#
# Current implementation:
#   - Uses explicit event-type metadata.
#   - Does NOT automatically classify raw transcript text.
#   - Therefore this is a supporting/optional attribute.
#
# Possible event types:
#   INFORMATION
#   DISCUSSION
#   DECISION
#   ACTION
#   QUESTION
#   BACKCHANNEL
#   OTHER
#
# This avoids relying on the previously tested zero-shot
# dialogue-act classifier, which was not sufficiently reliable.
# ============================================================

VALID_EVENT_TYPES = {
    "INFORMATION",
    "DISCUSSION",
    "DECISION",
    "ACTION",
    "QUESTION",
    "BACKCHANNEL",
    "OTHER"
}


def verify_event_type(
    claim_event_type,
    evidence_event_type
):
    """
    Compare claim and evidence event types.

    Event-type verification is optional until a reliable
    dialogue-act classifier is integrated.
    """

    # --------------------------------------------------------
    # No event type available
    # --------------------------------------------------------

    if (
        claim_event_type is None
        or evidence_event_type is None
    ):
        return {
            "event_check_applicable": False,
            "event_type_consistent": True
        }

    # --------------------------------------------------------
    # Validate event types
    # --------------------------------------------------------

    claim_event_type = claim_event_type.upper()
    evidence_event_type = evidence_event_type.upper()

    if (
        claim_event_type not in VALID_EVENT_TYPES
        or evidence_event_type not in VALID_EVENT_TYPES
    ):
        return {
            "event_check_applicable": False,
            "event_type_consistent": True
        }

    # --------------------------------------------------------
    # Compare event types
    # --------------------------------------------------------

    event_type_consistent = (
        claim_event_type == evidence_event_type
    )

    return {
        "event_check_applicable": True,
        "event_type_consistent": event_type_consistent
    }


# ============================================================
# TEST CASES
# ============================================================

# Test 1 — Matching event types
test_matching = verify_event_type(
    "INFORMATION",
    "INFORMATION"
)


# Test 2 — Different event types
test_different = verify_event_type(
    "DECISION",
    "INFORMATION"
)


# Test 3 — Event type unavailable
test_unavailable = verify_event_type(
    None,
    None
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 80)
print("EVENT-TYPE CONSISTENCY TEST")
print("=" * 80)

print("\nTest 1 — Matching event types:")
print(test_matching)

print("\nTest 2 — Different event types:")
print(test_different)

print("\nTest 3 — Event type unavailable:")
print(test_unavailable)

print("\n" + "=" * 80)

EVENT-TYPE CONSISTENCY TEST

Test 1 — Matching event types:
{'event_check_applicable': True, 'event_type_consistent': True}

Test 2 — Different event types:
{'event_check_applicable': True, 'event_type_consistent': False}

Test 3 — Event type unavailable:
{'event_check_applicable': False, 'event_type_consistent': True}



In [21]:
# ============================================================
# MULTI-ATTRIBUTE EVIDENCE CONSISTENCY
# Purpose:
# Combine content, speaker, time, and event-type consistency
# into one final verification decision.
#
# Final decision:
#   VERIFIED → all applicable checks pass
#   FLAGGED  → at least one applicable check fails
#
# Important:
# BGE similarity is NOT used as proof.
# BGE/FAISS retrieves evidence.
# NLI verifies content.
# Metadata checks verify the additional attributes.
# ============================================================

def verify_evidence(
    claim_text,
    evidence,
    claim_speaker=None,
    claim_start=None,
    claim_end=None,
    claim_event_type=None,
    evidence_event_type=None
):
    """
    Perform multi-attribute evidence consistency checking.

    Parameters
    ----------
    claim_text : str
        Generated MoM claim.

    evidence : dict
        Retrieved evidence document.

    claim_speaker : str or None
        Speaker attributed to the claim.

    claim_start, claim_end : float or None
        Timestamp associated with the claim.

    claim_event_type : str or None
        Event type assigned to the claim.

    evidence_event_type : str or None
        Event type assigned to the evidence.

    Returns
    -------
    dict
        Complete verification result.
    """

    # --------------------------------------------------------
    # 1. Content consistency
    # --------------------------------------------------------

    content_result = verify_content(
        evidence["text"],
        claim_text
    )


    # --------------------------------------------------------
    # 2. Speaker consistency
    # --------------------------------------------------------

    speaker_result = verify_speaker(
        claim_speaker,
        evidence["speaker"]
    )


    # --------------------------------------------------------
    # 3. Time consistency
    # --------------------------------------------------------

    time_result = verify_time(
        claim_start=claim_start,
        claim_end=claim_end,
        evidence_start=evidence["start"],
        evidence_end=evidence["end"]
    )


    # --------------------------------------------------------
    # 4. Event-type consistency
    # --------------------------------------------------------

    event_result = verify_event_type(
        claim_event_type,
        evidence_event_type
    )


    # --------------------------------------------------------
    # Collect applicable checks
    # --------------------------------------------------------

    checks = {
        "content_consistent":
            content_result["content_consistent"],

        "speaker_consistent":
            speaker_result["speaker_consistent"],

        "time_consistent":
            time_result["time_consistent"],

        "event_type_consistent":
            event_result["event_type_consistent"]
    }


    # --------------------------------------------------------
    # Final verification decision
    #
    # Every check is designed so that an unavailable attribute
    # returns True but is marked as not applicable.
    # --------------------------------------------------------

    final_verified = all(
        checks.values()
    )


    # --------------------------------------------------------
    # Return complete verification record
    # --------------------------------------------------------

    return {
        "verification_status":
            "VERIFIED" if final_verified else "FLAGGED",

        "claim":
            claim_text,

        "evidence_id":
            evidence["evidence_id"],

        "evidence_text":
            evidence["text"],

        "evidence_speaker":
            evidence["speaker"],

        "evidence_start":
            evidence["start"],

        "evidence_end":
            evidence["end"],

        "content": content_result,

        "speaker": speaker_result,

        "time": time_result,

        "event_type": event_result,

        "checks": checks
    }


print("=" * 80)
print("MULTI-ATTRIBUTE EVIDENCE CONSISTENCY FUNCTION CREATED")
print("=" * 80)

print("\nAttributes:")
print("  1. Content consistency  → DeBERTa NLI")
print("  2. Speaker consistency  → Speaker metadata")
print("  3. Time consistency     → Timestamp overlap")
print("  4. Event-type consistency → Event metadata")

print("\nFinal states:")
print("  VERIFIED → all applicable checks pass")
print("  FLAGGED  → one or more applicable checks fail")

print("\nVerification function is ready.")

print("=" * 80)

MULTI-ATTRIBUTE EVIDENCE CONSISTENCY FUNCTION CREATED

Attributes:
  1. Content consistency  → DeBERTa NLI
  2. Speaker consistency  → Speaker metadata
  3. Time consistency     → Timestamp overlap
  4. Event-type consistency → Event metadata

Final states:
  VERIFIED → all applicable checks pass
  FLAGGED  → one or more applicable checks fail

Verification function is ready.


In [22]:
# ============================================================
# MULTI-ATTRIBUTE VERIFICATION DEMONSTRATION
# Purpose:
# Demonstrate that the system can:
#
#   1. VERIFY a correctly attributed claim
#   2. FLAG a claim with an incorrect speaker
#
# Evidence used:
#   Evidence 94
# ============================================================

# ------------------------------------------------------------
# Retrieve Evidence 94
# ------------------------------------------------------------

evidence_94 = evidence_documents[93]


# ============================================================
# CASE 1 — CORRECT CLAIM
# ============================================================

claim_correct = (
    "It should be accessible and usable by all age groups."
)

result_verified = verify_evidence(
    claim_text=claim_correct,
    evidence=evidence_94,
    claim_speaker="SPEAKER_03",
    claim_start=638.0,
    claim_end=645.0,
    claim_event_type="INFORMATION",
    evidence_event_type="INFORMATION"
)


# ============================================================
# CASE 2 — INCORRECT SPEAKER
# ============================================================

claim_wrong_speaker = (
    "It should be accessible and usable by all age groups."
)

result_flagged = verify_evidence(
    claim_text=claim_wrong_speaker,
    evidence=evidence_94,
    claim_speaker="SPEAKER_02",   # Deliberately incorrect
    claim_start=638.0,
    claim_end=645.0,
    claim_event_type="INFORMATION",
    evidence_event_type="INFORMATION"
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 80)
print("MULTI-ATTRIBUTE VERIFICATION DEMONSTRATION")
print("=" * 80)


print("\nCASE 1 — CORRECT CLAIM")
print("-" * 80)

print("Claim:")
print(claim_correct)

print("\nExpected speaker: SPEAKER_03")

print("\nFinal status:")
print(result_verified["verification_status"])

print("\nIndividual checks:")

for check, value in result_verified["checks"].items():
    print(f"  {check}: {value}")


print("\n\nCASE 2 — INCORRECT SPEAKER")
print("-" * 80)

print("Claim:")
print(claim_wrong_speaker)

print("\nClaimed speaker: SPEAKER_02")
print("Actual evidence speaker: SPEAKER_03")

print("\nFinal status:")
print(result_flagged["verification_status"])

print("\nIndividual checks:")

for check, value in result_flagged["checks"].items():
    print(f"  {check}: {value}")


print("\n" + "=" * 80)

MULTI-ATTRIBUTE VERIFICATION DEMONSTRATION

CASE 1 — CORRECT CLAIM
--------------------------------------------------------------------------------
Claim:
It should be accessible and usable by all age groups.

Expected speaker: SPEAKER_03

Final status:
VERIFIED

Individual checks:
  content_consistent: True
  speaker_consistent: True
  time_consistent: True
  event_type_consistent: True


CASE 2 — INCORRECT SPEAKER
--------------------------------------------------------------------------------
Claim:
It should be accessible and usable by all age groups.

Claimed speaker: SPEAKER_02
Actual evidence speaker: SPEAKER_03

Final status:
FLAGGED

Individual checks:
  content_consistent: True
  speaker_consistent: False
  time_consistent: True
  event_type_consistent: True



In [23]:
# ============================================================
# AUTOMATIC CLAIM VERIFICATION
# Purpose:
# Automatically retrieve evidence for any MoM claim and then
# verify the retrieved candidates using:
#
#   1. BGE + FAISS       → evidence retrieval
#   2. DeBERTa NLI       → content verification
#   3. Speaker check     → speaker consistency
#   4. Time check        → timestamp consistency
#   5. Event-type check  → event consistency
#
# The system does NOT treat semantic similarity alone as proof.
# ============================================================


def retrieve_evidence(
    claim_text,
    top_k=10
):
    """
    Retrieve the top-K evidence candidates for a claim
    using BGE embeddings and FAISS.
    """

    # --------------------------------------------------------
    # Encode claim
    # --------------------------------------------------------

    claim_embedding = bge_model.encode(
        [claim_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")


    # --------------------------------------------------------
    # FAISS search
    # --------------------------------------------------------

    similarities, indices = faiss_index.search(
        claim_embedding,
        top_k
    )


    # --------------------------------------------------------
    # Build retrieval results
    # --------------------------------------------------------

    retrieved = []

    for similarity, index in zip(
        similarities[0],
        indices[0]
    ):

        evidence = evidence_documents[index].copy()

        evidence["retrieval_similarity"] = float(
            similarity
        )

        retrieved.append(evidence)


    return retrieved



def automatically_verify_claim(
    claim_text,
    claim_speaker=None,
    claim_start=None,
    claim_end=None,
    claim_event_type=None,
    top_k=10
):
    """
    Automatically retrieve and verify evidence for a claim.

    The first retrieved evidence that receives an NLI
    entailment prediction is considered supporting evidence.

    If no retrieved candidate entails the claim, the claim
    is FLAGGED.
    """

    # ========================================================
    # STEP 1 — RETRIEVE CANDIDATES
    # ========================================================

    retrieved_evidence = retrieve_evidence(
        claim_text,
        top_k=top_k
    )


    # ========================================================
    # STEP 2 — NLI VERIFICATION
    # ========================================================

    candidate_results = []

    for evidence in retrieved_evidence:

        content_result = verify_content(
            evidence["text"],
            claim_text
        )

        candidate_results.append({
            "evidence": evidence,
            "content": content_result
        })


    # ========================================================
    # STEP 3 — SELECT SUPPORTING EVIDENCE
    #
    # IMPORTANT:
    # We prioritize NLI entailment rather than BGE similarity.
    # Among entailed candidates, the highest retrieval score
    # is preferred.
    # ========================================================

    supporting_candidates = [
        item
        for item in candidate_results
        if item["content"]["content_consistent"]
    ]


    supporting_candidates.sort(
        key=lambda x:
        x["evidence"]["retrieval_similarity"],
        reverse=True
    )


    # ========================================================
    # STEP 4 — NO SUPPORTING EVIDENCE
    # ========================================================

    if not supporting_candidates:

        return {
            "verification_status": "FLAGGED",
            "claim": claim_text,
            "reason": "No retrieved evidence entails the claim.",
            "retrieved_candidates": candidate_results
        }


    # ========================================================
    # STEP 5 — VERIFY BEST SUPPORTING EVIDENCE
    # ========================================================

    best = supporting_candidates[0]

    evidence = best["evidence"]


    verification_result = verify_evidence(
        claim_text=claim_text,
        evidence=evidence,
        claim_speaker=claim_speaker,
        claim_start=claim_start,
        claim_end=claim_end,
        claim_event_type=claim_event_type,
        evidence_event_type=None
    )


    # Store retrieval candidates for traceability
    verification_result["retrieved_candidates"] = (
        candidate_results
    )


    return verification_result


print("=" * 80)
print("AUTOMATIC CLAIM VERIFICATION PIPELINE CREATED")
print("=" * 80)

print("\nPipeline:")
print("  Claim")
print("    ↓")
print("  BGE embedding")
print("    ↓")
print("  FAISS Top-K retrieval")
print("    ↓")
print("  DeBERTa NLI")
print("    ↓")
print("  Best supporting evidence")
print("    ↓")
print("  Speaker / Time / Event checks")
print("    ↓")
print("  VERIFIED / FLAGGED")

print("\nAutomatic verification function is ready.")

print("=" * 80)

AUTOMATIC CLAIM VERIFICATION PIPELINE CREATED

Pipeline:
  Claim
    ↓
  BGE embedding
    ↓
  FAISS Top-K retrieval
    ↓
  DeBERTa NLI
    ↓
  Best supporting evidence
    ↓
  Speaker / Time / Event checks
    ↓
  VERIFIED / FLAGGED

Automatic verification function is ready.


In [24]:
# ============================================================
# AUTOMATIC VERIFICATION TEST — SUPPORTED CLAIM
# Purpose:
# Test whether the system can automatically:
#
#   1. Retrieve relevant evidence using BGE + FAISS
#   2. Verify the evidence using DeBERTa NLI
#   3. Check speaker consistency
#   4. Check timestamp consistency
#   5. Produce the final VERIFIED / FLAGGED decision
# ============================================================

test_claim = (
    "It should be accessible and usable by all age groups."
)

result = automatically_verify_claim(
    claim_text=test_claim,
    claim_speaker="SPEAKER_03",
    claim_start=638.0,
    claim_end=645.0,
    claim_event_type="INFORMATION",
    top_k=10
)


print("=" * 80)
print("AUTOMATIC VERIFICATION — SUPPORTED CLAIM")
print("=" * 80)

print("\nClaim:")
print(result["claim"])

print("\nFinal verification status:")
print(result["verification_status"])


# ------------------------------------------------------------
# Display selected evidence
# ------------------------------------------------------------

if "evidence_id" in result:

    print("\nSelected evidence:")
    print(
        f"Evidence ID: {result['evidence_id']}"
    )

    print(
        f"Speaker: {result['evidence_speaker']}"
    )

    print(
        f"Time: "
        f"{result['evidence_start']:.3f}s"
        f" → "
        f"{result['evidence_end']:.3f}s"
    )

    print(
        f"Retrieval similarity: "
        f"{result['retrieved_candidates'][0]['evidence']['retrieval_similarity']:.4f}"
    )

    print("\nEvidence text:")
    print(result["evidence_text"])

    print("\nIndividual checks:")

    for check, value in result["checks"].items():
        print(f"  {check}: {value}")

else:

    print("\nReason:")
    print(result["reason"])


print("\n" + "=" * 80)

AUTOMATIC VERIFICATION — SUPPORTED CLAIM

Claim:
It should be accessible and usable by all age groups.

Final verification status:
VERIFIED

Selected evidence:
Evidence ID: 94
Speaker: SPEAKER_03
Time: 638.101s → 644.951s
Retrieval similarity: 0.7465

Evidence text:
You've got Market Range International, and you did say earlier it's got to be accessible and usable by all age groups.

Individual checks:
  content_consistent: True
  speaker_consistent: True
  time_consistent: True
  event_type_consistent: True



In [25]:
# ============================================================
# CHECK SAVED MoM CANDIDATES
# Purpose:
# Verify that the rule-based MoM candidates created earlier
# are still available on Google Drive.
# ============================================================

from pathlib import Path

MOM_CANDIDATES_PATH = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project/"
    "data/transcripts/ES2004a_mom_candidates_rule_based.json"
)

print("=" * 80)
print("CHECKING SAVED MoM CANDIDATES")
print("=" * 80)

print("\nFile:")
print(MOM_CANDIDATES_PATH)

print("\nExists:", MOM_CANDIDATES_PATH.exists())

print("=" * 80)

CHECKING SAVED MoM CANDIDATES

File:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_mom_candidates_rule_based.json

Exists: True


In [26]:
# ============================================================
# RESTORE RULE-BASED MoM CANDIDATES
# Purpose:
# Load the previously generated rule-based MoM candidates
# from Google Drive so we can inspect and reuse them.
# ============================================================

import json

with open(
    MOM_CANDIDATES_PATH,
    "r",
    encoding="utf-8"
) as f:
    mom_candidate_data = json.load(f)

print("=" * 80)
print("MoM CANDIDATES LOADED")
print("=" * 80)

print("\nTop-level type:")
print(type(mom_candidate_data).__name__)

if isinstance(mom_candidate_data, dict):

    print("\nTop-level keys:")
    print(list(mom_candidate_data.keys()))

elif isinstance(mom_candidate_data, list):

    print("\nNumber of candidates:")
    print(len(mom_candidate_data))


print("\nFirst 3,000 characters of the saved data:")
print(
    json.dumps(
        mom_candidate_data,
        indent=2,
        ensure_ascii=False
    )[:3000]
)

print("\n" + "=" * 80)

MoM CANDIDATES LOADED

Top-level type:
dict

Top-level keys:
['meeting_id', 'method', 'original_utterances', 'candidate_count', 'candidates']

First 3,000 characters of the saved data:
{
  "meeting_id": "ES2004a",
  "method": "rule_based_keyword_filter",
  "original_utterances": 148,
  "candidate_count": 50,
  "candidates": [
    {
      "candidate_id": 5,
      "source_utterance_id": 9,
      "speaker": "SPEAKER_02",
      "start": 84.658,
      "end": 93.127,
      "text": "I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda."
    },
    {
      "candidate_id": 6,
      "source_utterance_id": 10,
      "speaker": "SPEAKER_02",
      "start": 95.79,
      "end": 101.315,
      "text": "We will do some stuff, get to know each other a bit better, feel more comfortable with each other."
    },
    {
      "candidate_id": 7,
      "source_utterance_id": 11,
      "speaker": "SPEAKER_02",
      "start": 102.536,
      "end": 105.239,

In [27]:
# ============================================================
# RESTORE MoM CANDIDATES
# Purpose:
# Extract the 50 rule-based candidate utterances from the
# saved candidate file for the next MoM processing stage.
# ============================================================

mom_candidates = mom_candidate_data["candidates"]

print("=" * 80)
print("MoM CANDIDATES RESTORED")
print("=" * 80)

print("\nMeeting ID:",
      mom_candidate_data["meeting_id"])

print("Original utterances:",
      mom_candidate_data["original_utterances"])

print("Candidate count:",
      mom_candidate_data["candidate_count"])

print("Restored candidates:",
      len(mom_candidates))

print("\nCandidate structure:")
print(mom_candidates[0])

print("\nFirst 5 candidates:")

for candidate in mom_candidates[:5]:

    print(
        f"\nCandidate {candidate['candidate_id']}"
        f" | Source utterance: "
        f"{candidate['source_utterance_id']}"
    )

    print(
        f"Speaker: {candidate['speaker']}"
        f" | {candidate['start']:.3f}s"
        f" → {candidate['end']:.3f}s"
    )

    print("Text:", candidate["text"])

print("\n" + "=" * 80)

MoM CANDIDATES RESTORED

Meeting ID: ES2004a
Original utterances: 148
Candidate count: 50
Restored candidates: 50

Candidate structure:
{'candidate_id': 5, 'source_utterance_id': 9, 'speaker': 'SPEAKER_02', 'start': 84.658, 'end': 93.127, 'text': "I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda."}

First 5 candidates:

Candidate 5 | Source utterance: 9
Speaker: SPEAKER_02 | 84.658s → 93.127s
Text: I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.

Candidate 6 | Source utterance: 10
Speaker: SPEAKER_02 | 95.790s → 101.315s
Text: We will do some stuff, get to know each other a bit better, feel more comfortable with each other.

Candidate 7 | Source utterance: 11
Speaker: SPEAKER_02 | 102.536s → 105.239s
Text: Then we'll go do tool training,

Candidate 8 | Source utterance: 12
Speaker: SPEAKER_02 | 106.340s → 109.823s
Text: talk about the project plan, discuss our own i

In [28]:
# ============================================================
# STRUCTURED MoM CANDIDATE CLASSIFICATION
# Purpose:
# Convert rule-based candidate utterances into structured
# MoM items using transparent keyword/pattern rules.
#
# Categories:
#   ACTION
#   DECISION
#   DISCUSSION
#   INFORMATION
#   OTHER
#
# This is NOT the final verification stage.
# Verification will independently check each claim against
# the meeting evidence.
# ============================================================

import re


# ------------------------------------------------------------
# Classification patterns
# ------------------------------------------------------------

ACTION_PATTERNS = [
    r"\bwe('ll| will) ",
    r"\bwe need to ",
    r"\bneed to ",
    r"\bshould ",
    r"\bgoing to ",
    r"\bgo and ",
    r"\bdo ",
    r"\bfinish ",
    r"\bcomplete ",
    r"\bprepare ",
    r"\bcreate ",
    r"\bmake ",
    r"\bdevelop ",
    r"\bwork on ",
    r"\bbring ",
    r"\bput ",
    r"\bget ",
]

DECISION_PATTERNS = [
    r"\bwe decided ",
    r"\bdecided to ",
    r"\bagreed ",
    r"\bagree to ",
    r"\bwe'll use ",
    r"\bwe will use ",
    r"\bchosen ",
    r"\bchoose ",
    r"\bsettled on ",
]

QUESTION_PATTERNS = [
    r"\?",
    r"\bwhat ",
    r"\bhow ",
    r"\bwhy ",
    r"\bcan ",
    r"\bdo we ",
    r"\bis it ",
]

DISCUSSION_PATTERNS = [
    r"\bdiscuss ",
    r"\bthinking ",
    r"\bthink ",
    r"\bidea ",
    r"\bideas ",
    r"\bconsider ",
    r"\bconsidering ",
    r"\bperhaps ",
    r"\bmaybe ",
    r"\bproblem ",
    r"\bissue ",
    r"\bdesign ",
    r"\buser-friendly ",
    r"\baccessible ",
    r"\busable ",
]


def classify_mom_candidate(text):
    """
    Classify a candidate utterance into a coarse MoM category.

    Priority:
        DECISION
        ACTION
        DISCUSSION
        QUESTION
        INFORMATION
        OTHER
    """

    text_lower = text.lower().strip()

    # --------------------------------------------------------
    # Decision
    # --------------------------------------------------------

    for pattern in DECISION_PATTERNS:

        if re.search(pattern, text_lower):
            return "DECISION"


    # --------------------------------------------------------
    # Action
    # --------------------------------------------------------

    for pattern in ACTION_PATTERNS:

        if re.search(pattern, text_lower):
            return "ACTION"


    # --------------------------------------------------------
    # Discussion
    # --------------------------------------------------------

    for pattern in DISCUSSION_PATTERNS:

        if re.search(pattern, text_lower):
            return "DISCUSSION"


    # --------------------------------------------------------
    # Question
    # --------------------------------------------------------

    for pattern in QUESTION_PATTERNS:

        if re.search(pattern, text_lower):
            return "QUESTION"


    # --------------------------------------------------------
    # Information
    # --------------------------------------------------------

    return "INFORMATION"


# ============================================================
# APPLY CLASSIFICATION
# ============================================================

structured_mom_candidates = []

for candidate in mom_candidates:

    event_type = classify_mom_candidate(
        candidate["text"]
    )

    structured_item = candidate.copy()

    structured_item["event_type"] = event_type

    structured_mom_candidates.append(
        structured_item
    )


# ============================================================
# SUMMARY
# ============================================================

from collections import Counter

event_counts = Counter(
    item["event_type"]
    for item in structured_mom_candidates
)

print("=" * 80)
print("STRUCTURED MoM CANDIDATES CREATED")
print("=" * 80)

print("\nTotal candidates:",
      len(structured_mom_candidates))

print("\nEvent-type distribution:")

for event_type, count in event_counts.items():

    print(
        f"  {event_type:<12} : {count}"
    )


print("\nFirst 10 structured candidates:")

for item in structured_mom_candidates[:10]:

    print(
        f"\n[{item['event_type']}] "
        f"{item['speaker']} "
        f"| {item['start']:.3f}s → "
        f"{item['end']:.3f}s"
    )

    print(" ", item["text"])


print("\n" + "=" * 80)

STRUCTURED MoM CANDIDATES CREATED

Total candidates: 50

Event-type distribution:
  INFORMATION  : 9
  ACTION       : 25
  DISCUSSION   : 11
  DECISION     : 1
  QUESTION     : 4

First 10 structured candidates:

[INFORMATION] SPEAKER_02 | 84.658s → 93.127s
  I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.

[ACTION] SPEAKER_02 | 95.790s → 101.315s
  We will do some stuff, get to know each other a bit better, feel more comfortable with each other.

[ACTION] SPEAKER_02 | 102.536s → 105.239s
  Then we'll go do tool training,

[DISCUSSION] SPEAKER_02 | 106.340s → 109.823s
  talk about the project plan, discuss our own ideas and everything.

[INFORMATION] SPEAKER_02 | 117.699s → 120.921s
  Now, we're developing a remote control, which you probably already know.

[INFORMATION] SPEAKER_02 | 122.503s → 132.290s
  We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wi

In [29]:
# ============================================================
# INSPECT ALL STRUCTURED MoM CANDIDATES
# Purpose:
# Review all 50 candidates together with their assigned
# event types before generating the final MoM.
#
# This is an inspection step only.
# No data is modified.
# ============================================================

print("=" * 80)
print("ALL STRUCTURED MoM CANDIDATES")
print("=" * 80)

for item in structured_mom_candidates:

    print(
        f"\n[{item['candidate_id']:03d}] "
        f"{item['event_type']:<12} | "
        f"{item['speaker']} | "
        f"{item['start']:.3f}s → {item['end']:.3f}s"
    )

    print(" ", item["text"])

print("\n" + "=" * 80)
print(
    "Total candidates displayed:",
    len(structured_mom_candidates)
)
print("=" * 80)

ALL STRUCTURED MoM CANDIDATES

[005] INFORMATION  | SPEAKER_02 | 84.658s → 93.127s
  I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.

[006] ACTION       | SPEAKER_02 | 95.790s → 101.315s
  We will do some stuff, get to know each other a bit better, feel more comfortable with each other.

[007] ACTION       | SPEAKER_02 | 102.536s → 105.239s
  Then we'll go do tool training,

[008] DISCUSSION   | SPEAKER_02 | 106.340s → 109.823s
  talk about the project plan, discuss our own ideas and everything.

[010] INFORMATION  | SPEAKER_02 | 117.699s → 120.921s
  Now, we're developing a remote control, which you probably already know.

[011] INFORMATION  | SPEAKER_02 | 122.503s → 132.290s
  We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

[012] INFORMATION  | SPEAKER_02 | 133.291s → 138.174s
  you know, not a hunk of metal. And user-friendly, granni

In [30]:
# ============================================================
# MoM-WORTHY CANDIDATE SELECTION
# Purpose:
# Select substantive meeting content from the 50 rule-based
# candidates while preserving the original candidates.
#
# We exclude obvious non-MoM content such as the animal
# drawing/practice exercise.
#
# The original 50 candidates remain unchanged.
# ============================================================

# Candidate IDs identified from the inspection as primarily
# unrelated to the actual project discussion.
EXCLUDE_CANDIDATE_IDS = {
    16,   # animal drawing exercise
    19,   # tiger drawing exercise
    27,   # animal characteristics
    29,   # choosing animal
    32,   # changing drawing
    35,   # thinking about drawing
    41,   # big cat drawing
    48    # pen/drawing activity
}


mom_worthy_candidates = []

for candidate in structured_mom_candidates:

    if candidate["candidate_id"] in EXCLUDE_CANDIDATE_IDS:
        continue

    item = candidate.copy()

    # Mark this as a selected MoM-worthy candidate
    item["mom_worthy"] = True

    mom_worthy_candidates.append(item)


print("=" * 80)
print("MoM-WORTHY CANDIDATE SELECTION")
print("=" * 80)

print("\nOriginal candidates:",
      len(structured_mom_candidates))

print("Excluded candidates:",
      len(EXCLUDE_CANDIDATE_IDS))

print("MoM-worthy candidates:",
      len(mom_worthy_candidates))

print("\nExcluded candidate IDs:")
print(sorted(EXCLUDE_CANDIDATE_IDS))

print("\nFirst 10 selected candidates:")

for item in mom_worthy_candidates[:10]:

    print(
        f"\n[{item['candidate_id']:03d}] "
        f"{item['event_type']:<12} | "
        f"{item['speaker']} | "
        f"{item['start']:.3f}s → "
        f"{item['end']:.3f}s"
    )

    print(" ", item["text"])


print("\n" + "=" * 80)

MoM-WORTHY CANDIDATE SELECTION

Original candidates: 50
Excluded candidates: 8
MoM-worthy candidates: 42

Excluded candidate IDs:
[16, 19, 27, 29, 32, 35, 41, 48]

First 10 selected candidates:

[005] INFORMATION  | SPEAKER_02 | 84.658s → 93.127s
  I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.

[006] ACTION       | SPEAKER_02 | 95.790s → 101.315s
  We will do some stuff, get to know each other a bit better, feel more comfortable with each other.

[007] ACTION       | SPEAKER_02 | 102.536s → 105.239s
  Then we'll go do tool training,

[008] DISCUSSION   | SPEAKER_02 | 106.340s → 109.823s
  talk about the project plan, discuss our own ideas and everything.

[010] INFORMATION  | SPEAKER_02 | 117.699s → 120.921s
  Now, we're developing a remote control, which you probably already know.

[011] INFORMATION  | SPEAKER_02 | 122.503s → 132.290s
  We want it to be original, something that people haven't thought of. It's not out in t

In [31]:
# ============================================================
# SAVE MoM-WORTHY CANDIDATES
# Purpose:
# Save the selected meeting-relevant candidates separately.
#
# This file will be used as input for the next MoM generation
# and evidence verification stages.
#
# The original 50-candidate file is NOT modified.
# ============================================================

MOM_WORTHY_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_mom_worthy_candidates.json"
)

mom_worthy_output = {
    "meeting_id": transcript_data["meeting_id"],
    "source": "Rule-based MoM candidate selection",
    "original_candidate_count": len(structured_mom_candidates),
    "excluded_candidate_count": len(EXCLUDE_CANDIDATE_IDS),
    "mom_worthy_candidate_count": len(mom_worthy_candidates),
    "excluded_candidate_ids": sorted(EXCLUDE_CANDIDATE_IDS),
    "candidates": mom_worthy_candidates
}

with open(MOM_WORTHY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        mom_worthy_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 80)
print("MoM-WORTHY CANDIDATES SAVED")
print("=" * 80)

print("\nFile:")
print(MOM_WORTHY_PATH)

print("\nOriginal candidates:", len(structured_mom_candidates))
print("Excluded candidates:", len(EXCLUDE_CANDIDATE_IDS))
print("Saved MoM-worthy candidates:", len(mom_worthy_candidates))

print("\nFile exists:", MOM_WORTHY_PATH.exists())

print("\n" + "=" * 80)

NameError: name 'TRANSCRIPTS_DIR' is not defined

In [32]:
# ============================================================
# RESTORE PROJECT PATHS
# Purpose:
# Recreate the project directory variables after a runtime reset.
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

DATA_DIR = PROJECT_DIR / "data"

TRANSCRIPTS_DIR = DATA_DIR / "transcripts"

TRANSCRIPTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project directory:")
print(PROJECT_DIR)

print("\nTranscripts directory:")
print(TRANSCRIPTS_DIR)

print("\nDirectory exists:", TRANSCRIPTS_DIR.exists())

Project directory:
/content/drive/MyDrive/MTechIndProj/MoM_Project

Transcripts directory:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts

Directory exists: True


In [33]:
# ============================================================
# SAVE MoM-WORTHY CANDIDATES
# Purpose:
# Save the selected meeting-relevant candidates separately.
#
# This file will be used as input for the next MoM generation
# and evidence verification stages.
#
# The original 50-candidate file is NOT modified.
# ============================================================

MOM_WORTHY_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_mom_worthy_candidates.json"
)

mom_worthy_output = {
    "meeting_id": transcript_data["meeting_id"],
    "source": "Rule-based MoM candidate selection",
    "original_candidate_count": len(structured_mom_candidates),
    "excluded_candidate_count": len(EXCLUDE_CANDIDATE_IDS),
    "mom_worthy_candidate_count": len(mom_worthy_candidates),
    "excluded_candidate_ids": sorted(EXCLUDE_CANDIDATE_IDS),
    "candidates": mom_worthy_candidates
}

with open(MOM_WORTHY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        mom_worthy_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 80)
print("MoM-WORTHY CANDIDATES SAVED")
print("=" * 80)

print("\nFile:")
print(MOM_WORTHY_PATH)

print("\nOriginal candidates:", len(structured_mom_candidates))
print("Excluded candidates:", len(EXCLUDE_CANDIDATE_IDS))
print("Saved MoM-worthy candidates:", len(mom_worthy_candidates))

print("\nFile exists:", MOM_WORTHY_PATH.exists())

print("\n" + "=" * 80)

MoM-WORTHY CANDIDATES SAVED

File:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_mom_worthy_candidates.json

Original candidates: 50
Excluded candidates: 8
Saved MoM-worthy candidates: 42

File exists: True



In [34]:
# ============================================================
# INSPECT MoM-WORTHY CANDIDATES
# Purpose:
# Display all selected candidates before claim generation.
#
# This allows us to verify that the selected content is
# appropriate for conversion into MoM claims.
# ============================================================

print("=" * 100)
print("MoM-WORTHY CANDIDATES")
print("=" * 100)

for item in mom_worthy_candidates:

    print(
        f"\n[{item['candidate_id']:03d}] "
        f"{item['event_type']:<12} | "
        f"{item['speaker']} | "
        f"{item['start']:.3f}s → "
        f"{item['end']:.3f}s"
    )

    print(" ", item["text"])

print("\n" + "=" * 100)
print("Total candidates:", len(mom_worthy_candidates))
print("=" * 100)

MoM-WORTHY CANDIDATES

[005] INFORMATION  | SPEAKER_02 | 84.658s → 93.127s
  I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.

[006] ACTION       | SPEAKER_02 | 95.790s → 101.315s
  We will do some stuff, get to know each other a bit better, feel more comfortable with each other.

[007] ACTION       | SPEAKER_02 | 102.536s → 105.239s
  Then we'll go do tool training,

[008] DISCUSSION   | SPEAKER_02 | 106.340s → 109.823s
  talk about the project plan, discuss our own ideas and everything.

[010] INFORMATION  | SPEAKER_02 | 117.699s → 120.921s
  Now, we're developing a remote control, which you probably already know.

[011] INFORMATION  | SPEAKER_02 | 122.503s → 132.290s
  We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

[012] INFORMATION  | SPEAKER_02 | 133.291s → 138.174s
  you know, not a hunk of metal. And user-friendly, grannies to ki

In [35]:
# ============================================================
# MoM TOPIC GROUPING
# Purpose:
# Group related MoM-worthy candidates into coherent topics.
#
# Candidate IDs are preserved so that every generated claim
# remains traceable to its original transcript evidence.
# ============================================================

MOM_TOPIC_GROUPS = {

    "meeting_introduction_and_agenda": [
        5, 6, 7, 8
    ],

    "product_objective_and_requirements": [
        10, 11, 12, 13
    ],

    "design_process": [
        14, 15
    ],

    "pricing_and_production_cost": [
        49, 50, 54, 55, 56, 57
    ],

    "target_users_and_market": [
        59, 60, 61, 62
    ],

    "existing_remote_control_experience": [
        63, 64, 66, 67, 68, 69
    ],

    "lcd_and_menu_design": [
        72, 73
    ],

    "programming_and_older_users": [
        76, 79
    ],

    "screen_and_material_design": [
        85, 88, 89, 90, 91, 92
    ],

    "next_meeting_tasks": [
        93
    ],

    "company_and_design_reference": [
        96, 97, 98, 99
    ]
}


print("=" * 90)
print("MoM TOPIC GROUPING")
print("=" * 90)

total_grouped = 0

for topic, candidate_ids in MOM_TOPIC_GROUPS.items():

    print(f"\n{topic}")
    print("-" * 90)

    for candidate_id in candidate_ids:

        matching = [
            item for item in mom_worthy_candidates
            if item["candidate_id"] == candidate_id
        ]

        if matching:

            item = matching[0]

            print(
                f"[{candidate_id:03d}] "
                f"{item['speaker']} | "
                f"{item['start']:.3f}s → "
                f"{item['end']:.3f}s"
            )

            print(f"  {item['text']}")

            total_grouped += 1


print("\n" + "=" * 90)
print("SUMMARY")
print("=" * 90)

print("MoM-worthy candidates:", len(mom_worthy_candidates))
print("Grouped candidates:", total_grouped)
print("Topic groups:", len(MOM_TOPIC_GROUPS))

print("\nAll selected candidates grouped:",
      total_grouped == len(mom_worthy_candidates))

print("=" * 90)

MoM TOPIC GROUPING

meeting_introduction_and_agenda
------------------------------------------------------------------------------------------
[005] SPEAKER_02 | 84.658s → 93.127s
  I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.
[006] SPEAKER_02 | 95.790s → 101.315s
  We will do some stuff, get to know each other a bit better, feel more comfortable with each other.
[007] SPEAKER_02 | 102.536s → 105.239s
  Then we'll go do tool training,
[008] SPEAKER_02 | 106.340s → 109.823s
  talk about the project plan, discuss our own ideas and everything.

product_objective_and_requirements
------------------------------------------------------------------------------------------
[010] SPEAKER_02 | 117.699s → 120.921s
  Now, we're developing a remote control, which you probably already know.
[011] SPEAKER_02 | 122.503s → 132.290s
  We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, ap

In [36]:
# ============================================================
# CHECK FOR UNGROUPED CANDIDATES
# Purpose:
# Identify any MoM-worthy candidate that was not assigned
# to one of the topic groups.
# ============================================================

grouped_candidate_ids = set()

for candidate_ids in MOM_TOPIC_GROUPS.values():
    grouped_candidate_ids.update(candidate_ids)

selected_candidate_ids = {
    item["candidate_id"]
    for item in mom_worthy_candidates
}

ungrouped_candidate_ids = (
    selected_candidate_ids - grouped_candidate_ids
)

print("=" * 80)
print("UNGROUPED CANDIDATE CHECK")
print("=" * 80)

print("\nSelected candidates:", len(selected_candidate_ids))
print("Grouped candidates:", len(grouped_candidate_ids))
print("Ungrouped candidates:", len(ungrouped_candidate_ids))

print("\nUngrouped candidate IDs:")
print(sorted(ungrouped_candidate_ids))

for candidate_id in sorted(ungrouped_candidate_ids):

    matching = [
        item for item in mom_worthy_candidates
        if item["candidate_id"] == candidate_id
    ]

    if matching:

        item = matching[0]

        print("\nCandidate:", candidate_id)
        print("Event type:", item["event_type"])
        print("Speaker:", item["speaker"])
        print(
            f"Time: {item['start']:.3f}s → "
            f"{item['end']:.3f}s"
        )
        print("Text:", item["text"])

print("\n" + "=" * 80)

UNGROUPED CANDIDATE CHECK

Selected candidates: 42
Grouped candidates: 41
Ungrouped candidates: 1

Ungrouped candidate IDs:
[52]

Candidate: 52
Event type: ACTION
Speaker: SPEAKER_03
Time: 588.435s → 592.239s
Text: Should we be making notes of this? We can just refer to this later, can't we? I think so. I think so.



In [37]:
# ============================================================
# ADD MISSING CANDIDATE TO TOPIC GROUP
# Purpose:
# Add candidate 52 to the appropriate topic group so that
# every MoM-worthy candidate is accounted for.
# ============================================================

MOM_TOPIC_GROUPS["pricing_and_production_cost"].append(52)

print("=" * 80)
print("TOPIC GROUP UPDATED")
print("=" * 80)

print("\nAdded candidate: 52")
print("Topic: pricing_and_production_cost")

print("\nCandidates in pricing_and_production_cost:")
print(
    MOM_TOPIC_GROUPS["pricing_and_production_cost"]
)

print("\nTotal grouped candidates:")

grouped_candidate_ids = set()

for candidate_ids in MOM_TOPIC_GROUPS.values():
    grouped_candidate_ids.update(candidate_ids)

print(len(grouped_candidate_ids))

print(
    "\nAll selected candidates grouped:",
    len(grouped_candidate_ids) == len(mom_worthy_candidates)
)

print("=" * 80)

TOPIC GROUP UPDATED

Added candidate: 52
Topic: pricing_and_production_cost

Candidates in pricing_and_production_cost:
[49, 50, 54, 55, 56, 57, 52]

Total grouped candidates:
42

All selected candidates grouped: True


In [38]:
# ============================================================
# STRUCTURED MoM CLAIM GENERATION
# Purpose:
# Convert grouped transcript candidates into concise,
# evidence-preserving MoM claims.
#
# Important:
# - Claims are based only on transcript content.
# - Source candidate IDs are preserved for traceability.
# - Timestamps are inherited from the source candidates.
# - No unsupported facts are introduced.
# ============================================================

candidate_lookup = {
    item["candidate_id"]: item
    for item in mom_worthy_candidates
}


def get_candidates(candidate_ids):
    """Return candidate records for the supplied IDs."""
    return [
        candidate_lookup[candidate_id]
        for candidate_id in candidate_ids
        if candidate_id in candidate_lookup
    ]


def create_claim(
    claim_id,
    topic,
    claim_text,
    candidate_ids,
    event_type
):
    """
    Create one structured MoM claim from source candidates.
    """

    source_items = get_candidates(candidate_ids)

    if not source_items:
        return None

    # Use the earliest start and latest end among
    # the supporting source candidates.
    start = min(item["start"] for item in source_items)
    end = max(item["end"] for item in source_items)

    # If all supporting candidates have the same speaker,
    # retain that speaker. Otherwise mark as MULTIPLE.
    speakers = {
        item["speaker"]
        for item in source_items
    }

    speaker = (
        list(speakers)[0]
        if len(speakers) == 1
        else "MULTIPLE"
    )

    return {
        "claim_id": claim_id,
        "topic": topic,
        "claim_text": claim_text,
        "source_candidate_ids": candidate_ids,
        "speaker": speaker,
        "start": start,
        "end": end,
        "event_type": event_type
    }


mom_claims = []


# ------------------------------------------------------------
# 1. Meeting introduction and agenda
# ------------------------------------------------------------

claim = create_claim(
    1,
    "meeting_introduction_and_agenda",
    "The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.",
    [5, 6, 7, 8],
    "INFORMATION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 2. Product objective and requirements
# ------------------------------------------------------------

claim = create_claim(
    2,
    "product_objective_and_requirements",
    "The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.",
    [10, 11, 12, 13],
    "INFORMATION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 3. Design process
# ------------------------------------------------------------

claim = create_claim(
    3,
    "design_process",
    "The design process includes functional design, conceptual design, and detailed design, with individual work addressing product requirements and implementation.",
    [14, 15],
    "INFORMATION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 4. Pricing and production cost
# ------------------------------------------------------------

claim = create_claim(
    4,
    "pricing_and_production_cost",
    "The discussed selling price was 25 euros, while production costs were estimated at 12.50, with discussion about market pricing and whether to undercut existing products.",
    [49, 50, 54, 55, 56, 57],
    "DISCUSSION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 5. Meeting notes
# ------------------------------------------------------------

claim = create_claim(
    5,
    "meeting_notes",
    "The participants discussed whether notes should be made for the discussion or whether they could refer back to it later.",
    [52],
    "DISCUSSION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 6. Target users and market
# ------------------------------------------------------------

claim = create_claim(
    6,
    "target_users_and_market",
    "The product is intended to be accessible and user-friendly for a broad range of users and is planned for an international market.",
    [59, 60, 62],
    "DISCUSSION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 7. Existing remote-control experience
# ------------------------------------------------------------

claim = create_claim(
    7,
    "existing_remote_control_experience",
    "The discussion highlighted difficulties with existing universal remote controls, including signal loss, reprogramming, limited functionality, and excessive small buttons.",
    [63, 64, 66, 67, 68, 69],
    "DISCUSSION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 8. LCD and menu design
# ------------------------------------------------------------

claim = create_claim(
    8,
    "lcd_and_menu_design",
    "The team discussed using an LCD display with menus similar to a mobile phone to reduce button complexity and make browsing and navigation easier.",
    [72, 73],
    "DISCUSSION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 9. Programming and older users
# ------------------------------------------------------------

claim = create_claim(
    9,
    "programming_and_older_users",
    "The team identified programming difficulties for older users as a design issue that would need to be addressed.",
    [76, 79],
    "DISCUSSION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 10. Screen and material design
# ------------------------------------------------------------

claim = create_claim(
    10,
    "screen_and_material_design",
    "The team discussed using a simple LCD screen, obvious pictures and symbols, a possible flip-top design with a larger screen, and lightweight molded plastic while considering cost implications.",
    [85, 88, 89, 90, 91, 92],
    "DISCUSSION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 11. Work before next meeting
# ------------------------------------------------------------

claim = create_claim(
    11,
    "next_meeting_tasks",
    "Participants planned to work on their individual tasks before the next meeting.",
    [93],
    "ACTION"
)

if claim:
    mom_claims.append(claim)


# ------------------------------------------------------------
# 12. Company and design reference
# ------------------------------------------------------------

claim = create_claim(
    12,
    "company_and_design_reference",
    "The discussion considered a fashionable electronics design while noting that molded plastic could provide a practical and potentially lower-cost option.",
    [96, 97, 98, 99],
    "DISCUSSION"
)

if claim:
    mom_claims.append(claim)


print("=" * 100)
print("STRUCTURED MoM CLAIMS")
print("=" * 100)

print("\nNumber of claims generated:", len(mom_claims))

for claim in mom_claims:

    print(
        f"\nClaim {claim['claim_id']:02d} | "
        f"{claim['event_type']}"
    )

    print("Topic:", claim["topic"])
    print("Claim:", claim["claim_text"])

    print(
        "Sources:",
        claim["source_candidate_ids"]
    )

    print(
        f"Time: {claim['start']:.3f}s → "
        f"{claim['end']:.3f}s"
    )

    print("Speaker:", claim["speaker"])

print("\n" + "=" * 100)

STRUCTURED MoM CLAIMS

Number of claims generated: 12

Claim 01 | INFORMATION
Topic: meeting_introduction_and_agenda
Claim: The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.
Sources: [5, 6, 7, 8]
Time: 84.658s → 109.823s
Speaker: SPEAKER_02

Claim 02 | INFORMATION
Topic: product_objective_and_requirements
Claim: The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.
Sources: [10, 11, 12, 13]
Time: 117.699s → 141.016s
Speaker: SPEAKER_02

Claim 03 | INFORMATION
Topic: design_process
Claim: The design process includes functional design, conceptual design, and detailed design, with individual work addressing product requirements and implementation.
Sources: [14, 15]
Time: 142.933s → 167.740s
Speaker: SPEAKER_02

Claim 04 | DISCUSSION
Topic: pricing_and_production_cost
Claim: The discussed selling pric

In [39]:
# ============================================================
# SAVE STRUCTURED MoM CLAIMS
# Purpose:
# Save the generated MoM claims as a separate JSON artifact.
#
# Each claim retains:
# - source candidate IDs
# - speaker information
# - timestamps
# - event type
# - topic
#
# This preserves traceability for the evidence verification stage.
# ============================================================

MOM_CLAIMS_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_structured_mom_claims.json"
)

mom_claims_output = {
    "meeting_id": transcript_data["meeting_id"],
    "source": "Structured MoM claim generation",
    "candidate_count": len(mom_worthy_candidates),
    "topic_group_count": len(MOM_TOPIC_GROUPS),
    "claim_count": len(mom_claims),
    "claims": mom_claims
}

with open(MOM_CLAIMS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        mom_claims_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 90)
print("STRUCTURED MoM CLAIMS SAVED")
print("=" * 90)

print("\nFile:")
print(MOM_CLAIMS_PATH)

print("\nMoM-worthy candidates:", len(mom_worthy_candidates))
print("Topic groups:", len(MOM_TOPIC_GROUPS))
print("Structured claims:", len(mom_claims))

print("\nFile exists:", MOM_CLAIMS_PATH.exists())

print("\n" + "=" * 90)

STRUCTURED MoM CLAIMS SAVED

File:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_structured_mom_claims.json

MoM-worthy candidates: 42
Topic groups: 11
Structured claims: 12

File exists: True

